# Tipburn Leaf-Level Machine Learning Pipeline and Validation Report

This notebook implements the validated spectral-analysis and machine-learning workflow, from leaf-level preprocessing and plant-grouped model development to final holdout evaluation. Retained results and supporting validation evidence are compiled into one standalone HTML report.

## 1. Imports, Constants, and Dataset Registry

Core configuration, feature lists, labels, plotting styles, and workbook registry.

In [1]:
from __future__ import annotations

import base64
import html
import json
import io
import joblib
import os
import platform
import re
import sys
import warnings
from datetime import datetime, timezone
from dataclasses import dataclass
from pathlib import Path
from IPython.display import display
from typing import Any

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, f1_score
from scipy.signal import savgol_filter

_MPLCONFIGDIR = Path.cwd().resolve() / ".cache" / "matplotlib"
_MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(_MPLCONFIGDIR))

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

pd.set_option("display.max_colwidth", None)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Resolve the repository root so shared modules import from notebooks or scripts.
for _candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
    if (_candidate / "core" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from core.preprocessing import (
    EPS,
    REIP_MIN,
    REIP_MAX,
    DEFAULT_WINDOW,
    POLYORDER,
    CONTEXT_COLS,
    ID_COLS,
    LEAF_ID_COLS,
    EDA_FEATURE_COLS,
    FINAL_FEATURE_COLS,
    group_to_leaf_level,
    interpolate_each_row,
    preprocess_leaf_spectra,
    rowwise_minmax,
)
from core.model import (
    RANDOM_STATE,
    FINAL_MODEL_NAME,
    FINAL_MODEL_C,
    FINAL_DECISION_THRESHOLD,
    build_locked_final_pipeline,
    continuous_model_score,
)
from core.metrics import metric_summary, binary_classification_metrics
from core.validation import detect_wavelength_columns, assert_quality_checks
from core.convergence import (
    catch_convergence_warning,
    build_convergence_record,
    summarize_convergence_audit,
    check_convergence_audit,
    CONVERGENCE_AUDIT_COLUMNS,
)

FINAL_TEST_SIZE = 0.20
REPEAT_SEEDS = [42, 52, 62, 72, 82]

SMOOTHING_WINDOWS = [
    3,
    5,
    7,
    9,
    11,
    13,
    15,
    21,
]

FEATURE_SETS = {
    "3_features": [
        "REIP_nm",
        "NDRE",
        "NIR_Mean_SmoothedRaw",
    ],
    "4_features": [
        "REIP_nm",
        "NDRE",
        "Red_Edge_Slope",
        "NIR_Mean_SmoothedRaw",
    ],
    "5_features": list(FINAL_FEATURE_COLS),
    "6_features": list(EDA_FEATURE_COLS),
}

GROUP_ORDER = [
    "Control_W3_DAT21",
    "Tipburn_W3_DAT21",
    "Control_W4_DAT28",
    "Tipburn_W4_DAT28",
]

GROUP_STYLE = {
    "Control_W3_DAT21": {"label": "Healthy W3 (DAT21)", "color": "#2f6fbb"},
    "Tipburn_W3_DAT21": {"label": "Tipburn W3 (DAT21)", "color": "#c84c3a"},
    "Control_W4_DAT28": {"label": "Healthy W4 (DAT28)", "color": "#74a9cf"},
    "Tipburn_W4_DAT28": {"label": "Tipburn W4 (DAT28)", "color": "#2d8a58"},
}


@dataclass(frozen=True)
class DatasetSpec:
    filename: str
    label: str
    week: str
    dat: int


DATASET_REGISTRY = {
    "Control_W3_DAT21": DatasetSpec(
        filename="lettuce_control_3rdweek(DAT21).xlsx",
        label="Healthy",
        week="Week 3",
        dat=21,
    ),
    "Tipburn_W3_DAT21": DatasetSpec(
        filename="lettuce_tipburn_3rdweek(DAT21).xlsx",
        label="Tipburn",
        week="Week 3",
        dat=21,
    ),
    "Control_W4_DAT28": DatasetSpec(
        filename="lettuce_control_4thweek(DAT28).xlsx",
        label="Healthy",
        week="Week 4",
        dat=28,
    ),
    "Tipburn_W4_DAT28": DatasetSpec(
        filename="lettuce_tipburn_4thweek(DAT28).xlsx",
        label="Tipburn",
        week="Week 4",
        dat=28,
    ),
}

## 2. Dataset Loading and Leaf-Level Aggregation

Dataset resolution, wavelength detection, workbook loading, and scan-to-leaf grouping helpers.

In [2]:
def require_sklearn() -> dict[str, Any]:
    """Load the required scikit-learn components."""
    try:
        from sklearn.cross_decomposition import PLSRegression
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.linear_model import LogisticRegression
        from sklearn.metrics import (
            confusion_matrix,
            f1_score,
            precision_score,
            recall_score,
            roc_auc_score,
            roc_curve,
        )
        from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import StandardScaler
        from sklearn.svm import SVC
    except ImportError as exc:
        raise RuntimeError(
            "scikit-learn is required to run the modelling analysis."
        ) from exc

    return {
        "PLSRegression": PLSRegression,
        "RandomForestClassifier": RandomForestClassifier,
        "LogisticRegression": LogisticRegression,
        "confusion_matrix": confusion_matrix,
        "f1_score": f1_score,
        "precision_score": precision_score,
        "recall_score": recall_score,
        "roc_auc_score": roc_auc_score,
        "roc_curve": roc_curve,
        "GridSearchCV": GridSearchCV,
        "StratifiedGroupKFold": StratifiedGroupKFold,
        "Pipeline": Pipeline,
        "StandardScaler": StandardScaler,
        "SVC": SVC,
    }


def resolve_dataset_path(
    dataset_dir: Path,
    filename: str,
) -> Path:
    """Return the full path of a dataset file."""

    file_path = dataset_dir / filename

    if not file_path.exists():
        raise FileNotFoundError(
            f"Dataset file was not found:\n{file_path}"
        )

    return file_path


def ensure_output_dir(output_dir: Path) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir


def format_sample_group_id(dataset_id: str, lettuce_sample_id: Any) -> str:
    """Create stable group ids such as Tipburn_W3_DAT21_S1."""
    if pd.isna(lettuce_sample_id):
        sample_code = "SNA"
    else:
        text = str(lettuce_sample_id).strip()
        try:
            number = float(text)
            sample_code = f"S{int(number)}" if number.is_integer() else f"S{text}"
        except ValueError:
            cleaned = "".join(ch if ch.isalnum() else "_" for ch in text)
            sample_code = f"S{cleaned}"
    return f"{dataset_id}_{sample_code}"


def load_one_dataset(dataset_dir: Path, dataset_id: str, spec: DatasetSpec) -> pd.DataFrame:
    path = resolve_dataset_path(dataset_dir, spec.filename)
    df = pd.read_excel(path)
    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")].copy()

    missing_id_cols = [col for col in ID_COLS if col not in df.columns]
    if missing_id_cols:
        raise ValueError(f"{dataset_id} is missing ID columns: {missing_id_cols}")

    df["Dataset_ID"] = dataset_id
    df["Label"] = spec.label
    df["Week"] = spec.week
    df["DAT"] = spec.dat
    df["Source_File"] = str(path)
    df["Sample_Group_ID"] = [
        format_sample_group_id(dataset_id, sample_id)
        for sample_id in df["Lettuce_Sample_ID"]
    ]
    return df


def load_all_datasets(dataset_dir: Path) -> tuple[pd.DataFrame, list[str], np.ndarray]:
    frames = [
        load_one_dataset(dataset_dir, dataset_id, spec)
        for dataset_id, spec in DATASET_REGISTRY.items()
    ]
    raw_df = pd.concat(frames, ignore_index=True)

    wavelength_cols = detect_wavelength_columns(raw_df)
    if len(wavelength_cols) == 0:
        raise ValueError("No wavelength columns detected in the 500-850 nm range.")

    wavelengths = np.array([float(str(col).strip()) for col in wavelength_cols])
    raw_df[wavelength_cols] = raw_df[wavelength_cols].apply(pd.to_numeric, errors="coerce")
    return raw_df, wavelength_cols, wavelengths



## 3. Spectral Preprocessing and Audit Helpers

Smoothing, normalization, derivative spectra, feature extraction, and audit-table helpers.

In [3]:
def make_audit_tables(raw_df: pd.DataFrame,
                      leaf_df: pd.DataFrame,
                      wavelength_cols: list[str]) -> dict[str, pd.DataFrame]:
    raw_audit = (
        raw_df
        .groupby(CONTEXT_COLS)
        .agg(
            Raw_Rows=("Lettuce_Sample_ID", "size"),
            Lettuce_Samples=("Lettuce_Sample_ID", "nunique"),
            Leaves=("Leaf_ID", "nunique"),
            Areas=("Area_ID", "nunique"),
            Sample_Groups=("Sample_Group_ID", "nunique"),
        )
        .reset_index()
    )

    leaf_audit = (
        leaf_df
        .groupby(CONTEXT_COLS)
        .agg(
            Leaf_Rows=("Leaf_ID", "size"),
            Sample_Groups=("Sample_Group_ID", "nunique"),
            Unique_Lettuce_Samples=("Lettuce_Sample_ID", "nunique"),
            Area_Count_Min=("Area_Count", "min"),
            Area_Count_Median=("Area_Count", "median"),
            Area_Count_Max=("Area_Count", "max"),
            Scan_Count_Total_Min=("Scan_Count_Total", "min"),
            Scan_Count_Total_Median=("Scan_Count_Total", "median"),
            Scan_Count_Total_Max=("Scan_Count_Total", "max"),
        )
        .reset_index()
    )

    missing_by_dataset = []
    for dataset_id, subset in raw_df.groupby("Dataset_ID"):
        values = subset[wavelength_cols]
        missing_by_dataset.append(
            {
                "Dataset_ID": dataset_id,
                "Spectral_Cells": int(values.size),
                "Missing_Spectral_Cells": int(values.isna().sum().sum()),
                "Missing_Spectral_Percent": values.isna().sum().sum() / values.size * 100,
            }
        )

    return {
        "raw_dataset_audit": raw_audit,
        "leaf_level_audit": leaf_audit,
        "missing_spectral_audit": pd.DataFrame(missing_by_dataset),
    }


def validate_savgol_settings(
    window_length: int,
    polyorder: int,
    band_count: int,
) -> None:
    """Validate a Savitzky-Golay window before sensitivity analysis."""

    if window_length % 2 == 0:
        raise ValueError(
            f"Window length must be odd, got {window_length}."
        )

    if window_length <= polyorder:
        raise ValueError(
            f"Window length {window_length} must be greater "
            f"than polynomial order {polyorder}."
        )

    if window_length > band_count:
        raise ValueError(
            f"Window length {window_length} exceeds the "
            f"{band_count} retained wavelength bands."
        )


def extract_reip_for_smoothing_window(
    leaf_subset: pd.DataFrame,
    wavelength_cols: list[str],
    wavelengths: np.ndarray,
    window_length: int,
    polyorder: int = POLYORDER,
) -> pd.DataFrame:
    """Recalculate leaf-level REIP using one smoothing window."""

    validate_savgol_settings(
        window_length,
        polyorder,
        len(wavelength_cols),
    )

    wavelength_steps = np.diff(wavelengths)

    if not np.allclose(
        wavelength_steps,
        wavelength_steps[0],
    ):
        raise AssertionError(
            "The wavelength grid must be evenly spaced."
        )

    raw_spectra = leaf_subset[
        wavelength_cols
    ].to_numpy(dtype=float)

    if not np.isfinite(raw_spectra).all():
        raise AssertionError(
            "Non-finite values were found in the "
            "development leaf spectra."
        )

    smoothed_spectra = savgol_filter(
        raw_spectra,
        window_length=window_length,
        polyorder=polyorder,
        axis=1,
        mode="interp",
    )

    normalized_spectra = rowwise_minmax(
        smoothed_spectra
    )

    derivative_spectra = np.gradient(
        normalized_spectra,
        wavelengths,
        axis=1,
    )

    red_edge_mask = (
        (wavelengths >= REIP_MIN)
        & (wavelengths <= REIP_MAX)
    )

    red_edge_wavelengths = wavelengths[
        red_edge_mask
    ]

    red_edge_derivative = derivative_spectra[
        :,
        red_edge_mask,
    ]

    reip_positions = np.argmax(
        red_edge_derivative,
        axis=1,
    )

    result = leaf_subset[
        [
            *CONTEXT_COLS,
            *LEAF_ID_COLS,
        ]
    ].copy()

    result["Window_Length"] = window_length

    wavelength_interval_nm = float(
        wavelength_steps[0]
    )

    result["Window_Width_nm"] = int(
        round(
            window_length
            * wavelength_interval_nm
        )
    )

    result["REIP_nm"] = red_edge_wavelengths[
        reip_positions
    ]

    return result


def cohens_d(
    healthy_values: np.ndarray,
    tipburn_values: np.ndarray,
) -> float:
    """Cohen's d calculated as Tipburn minus Healthy."""

    healthy_values = np.asarray(
        healthy_values,
        dtype=float,
    )

    tipburn_values = np.asarray(
        tipburn_values,
        dtype=float,
    )

    n_healthy = len(healthy_values)
    n_tipburn = len(tipburn_values)

    if n_healthy < 2 or n_tipburn < 2:
        return np.nan

    pooled_variance = (
        (
            (n_healthy - 1)
            * healthy_values.var(ddof=1)
        )
        + (
            (n_tipburn - 1)
            * tipburn_values.var(ddof=1)
        )
    ) / (
        n_healthy
        + n_tipburn
        - 2
    )

    pooled_sd = np.sqrt(
        pooled_variance
    )

    if pooled_sd < EPS:
        return np.nan

    return float(
        (
            tipburn_values.mean()
            - healthy_values.mean()
        )
        / pooled_sd
    )


def build_reip_smoothing_sensitivity(
    development_leaf_df: pd.DataFrame,
    wavelength_cols: list[str],
    wavelengths: np.ndarray,
    windows: list[int],
) -> dict[str, pd.DataFrame]:
    """Build development-only plant-level REIP sensitivity results."""

    leaf_results = pd.concat(
        [
            extract_reip_for_smoothing_window(
                development_leaf_df,
                wavelength_cols,
                wavelengths,
                window_length,
            )
            for window_length in windows
        ],
        ignore_index=True,
    )

    plant_results = (
        leaf_results
        .groupby(
            [
                "Window_Length",
                "Window_Width_nm",
                "Dataset_ID",
                "Week",
                "Label",
                "Sample_Group_ID",
            ],
            as_index=False,
        )
        .agg(
            Leaf_Count=(
                "Leaf_ID",
                "nunique",
            ),
            Plant_Mean_REIP_nm=(
                "REIP_nm",
                "mean",
            ),
        )
    )

    if not plant_results[
        "Leaf_Count"
    ].eq(3).all():
        raise AssertionError(
            "Each plant must contribute exactly three leaves "
            "to the smoothing sensitivity analysis."
        )

    effect_rows = []

    for window_length in windows:
        window_data = plant_results.loc[
            plant_results[
                "Window_Length"
            ].eq(window_length)
        ]

        for week in [
            "Week 3",
            "Week 4",
        ]:
            week_data = window_data.loc[
                window_data["Week"].eq(week)
            ]

            healthy_values = week_data.loc[
                week_data["Label"].eq("Healthy"),
                "Plant_Mean_REIP_nm",
            ].to_numpy(dtype=float)

            tipburn_values = week_data.loc[
                week_data["Label"].eq("Tipburn"),
                "Plant_Mean_REIP_nm",
            ].to_numpy(dtype=float)

            if len(healthy_values) != 24:
                raise AssertionError(
                    f"{week}, window {window_length}: "
                    f"expected 24 Healthy plants, "
                    f"got {len(healthy_values)}."
                )

            if len(tipburn_values) != 24:
                raise AssertionError(
                    f"{week}, window {window_length}: "
                    f"expected 24 Tipburn plants, "
                    f"got {len(tipburn_values)}."
                )

            effect_rows.append(
                {
                    "Window_Length": window_length,
                    "Window_Width_nm": int(
                        window_data[
                            "Window_Width_nm"
                        ].iloc[0]
                    ),
                    "Week": week,
                    "Healthy_Plants": len(
                        healthy_values
                    ),
                    "Tipburn_Plants": len(
                        tipburn_values
                    ),
                    "Healthy_REIP_Mean": float(
                        healthy_values.mean()
                    ),
                    "Tipburn_REIP_Mean": float(
                        tipburn_values.mean()
                    ),
                    "Cohen_d_Tipburn_Minus_Healthy": (
                        cohens_d(
                            healthy_values,
                            tipburn_values,
                        )
                    ),
                }
            )

    effect_long = pd.DataFrame(
        effect_rows
    )

    report_table = (
        effect_long
        .pivot(
            index=[
                "Window_Length",
                "Window_Width_nm",
            ],
            columns="Week",
            values=(
                "Cohen_d_Tipburn_Minus_Healthy"
            ),
        )
        .reset_index()
        .rename(
            columns={
                "Window_Length": "Window (bands)",
                "Window_Width_nm": "Width (nm)",
            }
        )
        .rename_axis(
            columns=None
        )
        .sort_values(
            "Window (bands)"
        )
        .reset_index(drop=True)
    )

    report_table = report_table[
        [
            "Window (bands)",
            "Width (nm)",
            "Week 3",
            "Week 4",
        ]
    ]

    return {
        "leaf_results": leaf_results,
        "plant_results": plant_results,
        "effect_long": effect_long,
        "report_table": report_table,
    }

## 4. Split Helper
Fixed plant-level split.

In [4]:
def split_final_test_by_sample(features: pd.DataFrame,
                               test_size: float = FINAL_TEST_SIZE,
                               random_state: int = RANDOM_STATE) -> tuple[pd.Series, pd.DataFrame]:
    """
    Hold out 20% of lettuce sample groups from each Dataset_ID.

    This keeps the final test balanced across Week 3/Week 4 and
    healthy/tipburn dataset groups.
    """
    rng = np.random.default_rng(random_state)
    group_table = (
        features[["Dataset_ID", "Label", "Week", "DAT", "Sample_Group_ID"]]
        .drop_duplicates()
        .sort_values(["Dataset_ID", "Sample_Group_ID"])
        .reset_index(drop=True)
    )

    test_groups: list[str] = []
    for dataset_id, subset in group_table.groupby("Dataset_ID", sort=False):
        groups = subset["Sample_Group_ID"].to_numpy()
        groups = groups.copy()
        rng.shuffle(groups)
        n_test = max(1, int(round(len(groups) * test_size)))
        test_groups.extend(groups[:n_test].tolist())

    is_test = features["Sample_Group_ID"].isin(test_groups)
    split_labels = pd.Series(np.where(is_test, "final_test", "train_val"),
                             index=features.index,
                             name="Split")

    split_group_table = group_table.copy()
    split_group_table["Split"] = np.where(
        split_group_table["Sample_Group_ID"].isin(test_groups),
        "final_test",
        "train_val",
    )
    return split_labels, split_group_table



## 5. Model Evaluation Helpers

Grouped cross-validation, model grids, metrics, final-holdout evaluation, confusion matrix, and ROC helpers.

In [5]:
def group_label_counts(y: np.ndarray, groups: np.ndarray) -> pd.Series:
    group_labels = (
        pd.DataFrame({"y": y, "group": groups})
        .drop_duplicates("group")
        .groupby("y")["group"]
        .nunique()
    )
    return group_labels


def make_group_cv(y: np.ndarray,
                  groups: np.ndarray,
                  preferred_splits: int,
                  random_state: int):
    sk = require_sklearn()
    StratifiedGroupKFold = sk["StratifiedGroupKFold"]

    counts = group_label_counts(y, groups)
    min_groups_per_class = int(counts.min())
    n_splits = min(preferred_splits, min_groups_per_class)
    if n_splits < 2:
        raise ValueError(
            "Need at least two independent groups per class for grouped CV."
        )
    return StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )


def get_models_and_grids(random_state: int = RANDOM_STATE) -> tuple[dict[str, Any], dict[str, dict[str, list[Any]]]]:
    sk = require_sklearn()
    Pipeline = sk["Pipeline"]
    StandardScaler = sk["StandardScaler"]
    SVC = sk["SVC"]
    RandomForestClassifier = sk["RandomForestClassifier"]
    LogisticRegression = sk["LogisticRegression"]

    models = {
        "SVM_RBF": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel="rbf", random_state=random_state)),
        ]),
        "RandomForest": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", RandomForestClassifier(random_state=random_state)),
        ]),
        "LogisticRegression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=50, random_state=random_state)),
        ]),
    }
    grids = {
        "SVM_RBF": {
            "clf__C": [0.1, 1, 10, 100],
            "clf__gamma": ["scale", "auto", 0.001, 0.01],
        },
        "RandomForest": {
            "clf__n_estimators": [50, 100, 200],
            "clf__max_depth": [3, 5, 8, None],
            "clf__min_samples_leaf": [1, 2, 3],
        },
        "LogisticRegression": {
            "clf__C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 300],
        },
    }
    return models, grids



## 6. Project Paths

Robust project-root and output-directory resolution for VS Code and Jupyter runs.

In [6]:
def find_project_dir(start: Path | None = None) -> Path:
    """Find the project folder without relying on its name."""
    start = (start or Path.cwd()).resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "core").is_dir()
            and (candidate / "dashboard").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project folder. "
        "Open the notebook from inside the extracted project folder."
    )


PROJECT_DIR = find_project_dir()
DATASET_DIR = PROJECT_DIR / "dataset"
OUTPUT_DIR = ensure_output_dir(PROJECT_DIR / "outputs")

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Dataset folder not found: {DATASET_DIR}"
    )

print(f"Project directory: {PROJECT_DIR}")
print(f"Dataset directory: {DATASET_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

Project directory: /Users/Dell/Documents/FYP/CP2 Submission/smart_leaf_analyzer
Dataset directory: /Users/Dell/Documents/FYP/CP2 Submission/smart_leaf_analyzer/dataset
Output directory: /Users/Dell/Documents/FYP/CP2 Submission/smart_leaf_analyzer/outputs


## 7. HTML Report Helpers

HTML table sections, base64 figure embedding, and single-file report writer.

In [7]:
def slugify(value: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "-", value.lower()).strip("-")
    return slug or "section"


def tint_table_rows(table_html: str, row_values: list[str], color_map: dict[str, str]) -> str:
    """Tint <tbody> rows with an inline background color keyed by an aligned row value."""
    def _replace_tbody(match: re.Match) -> str:
        body = match.group(1)
        row_htmls = re.findall(r"<tr>.*?</tr>", body, flags=re.S)
        if len(row_htmls) != len(row_values):
            return match.group(0)
        rebuilt = []
        for row_html, value in zip(row_htmls, row_values):
            color = color_map.get(value)
            if color:
                row_html = row_html.replace("<tr>", f'<tr style="background-color:{color};">', 1)
            rebuilt.append(row_html)
        return f"<tbody>{''.join(rebuilt)}</tbody>"
    return re.sub(r"<tbody>(.*?)</tbody>", _replace_tbody, table_html, count=1, flags=re.S)


def dataframe_to_report_section(
    title: str,
    dataframe: pd.DataFrame,
    description: str | None = None,
    collapsible: bool = False,
    section_id: str | None = None,
    row_tint_col: str | None = None,
    row_tint_colors: dict[str, str] | None = None,
) -> str:
    if row_tint_col != "Display_Name":
        dataframe = dataframe.drop(columns=["Display_Name"], errors="ignore")
    sid = section_id or slugify(title)
    escaped_sid = html.escape(sid)
    escaped_title = html.escape(title)
    desc_html = ""
    if description:
        desc_html = f'<p>{html.escape(description)}</p>'
    table_html = dataframe.to_html(
        escape=True,
        index=True,
        border=0,
        float_format=lambda value: f"{value:.6f}",
        classes="report-table",
        table_id=f"{sid}-table",
    )
    if row_tint_col and row_tint_col in dataframe.columns:
        table_html = tint_table_rows(
            table_html,
            dataframe[row_tint_col].astype(str).tolist(),
            row_tint_colors or {},
        )
    export_controls = (
        '<div class="table-export-controls">'
        '<label class="table-export-check">'
        f'<input type="checkbox" class="table-export-select" data-section-id="{escaped_sid}" checked> '
        'Include in selected export'
        '</label>'
        f'<button type="button" class="table-export-button" data-section-id="{escaped_sid}">Download CSV</button>'
        '</div>'
    )
    wrapped_table = f'{export_controls}<div class="table-wrap">{table_html}</div>'
    if collapsible:
        wrapped_table = (
            '<details class="table-details">'
            f'<summary>Open full table ({len(dataframe):,} rows)</summary>'
            f'{wrapped_table}'
            '</details>'
        )
    return (
        f'<section id="{escaped_sid}" class="report-section">'
        f'<h2>{escaped_title}</h2>'
        f'{desc_html}'
        f'{wrapped_table}'
        '</section>'
    )


def figure_to_base64_html(
    figure,
    title: str,
    description: str | None = None,
    section_id: str | None = None,
) -> str:
    sid = section_id or slugify(title)
    buffer = io.BytesIO()
    figure.savefig(
        buffer,
        format="png",
        dpi=180,
        bbox_inches="tight",
    )
    buffer.seek(0)
    encoded = base64.b64encode(buffer.read()).decode("ascii")
    plt.close(figure)
    desc_html = ""
    if description:
        desc_html = f'<p>{html.escape(description)}</p>'
    return (
        f'<section id="{html.escape(sid)}">'
        f'<h2>{html.escape(title)}</h2>'
        f'{desc_html}'
        f'<figure><img src="data:image/png;base64,{encoded}" alt="{html.escape(title)}"></figure>'
        '</section>'
    )


# Define the HTML report order independently of notebook execution order.
REPORT_PARTS: dict[int, str] = {
    1: "Part 1 - Original Dataset to Final Model",
    2: "Part 2 - Synthetic Data Experiment",
    3: "Part 3 - Wavelength-Pair Index Search",
}

REPORT_SECTION_LAYOUT: dict[int, dict[str, Any]] = {
    1: {
        "main": [
            "report-metadata",
            "dataset-and-preprocessing-summary",
            "fixed-plant-level-split-summary",
            "split-class-counts",
            "development-smoothing-window-sensitivity",
            "six-feature-candidate-eda",
            "six-feature-distributions-by-label",
            "five-feature-logistic-regression-coefficient-interpretation",
            "basic-spectral-overview",
            "spectral-and-six-feature-pls-da-overview",
            "full-spectrum-pls-da-overview",
            "feature-set-and-model-comparison",
            "logistic-regression-best-parameter-selection-frequency",
            "two-by-two-finalist-seed-level-results",
            "two-by-two-finalist-comparison",
            "locked-final-candidate",
            "dummy-baseline",
            "dummy-baseline-predictions",
            "plant-label-permutation-diagnostic",
            "development-oof-leaf-level-scores",
            "development-oof-leaf-level-evaluation",
            "development-oof-confusion-matrix",
            "development-oof-week-label-score-summary",
            "week-effect-delta-table",
            "development-oof-leaf-level-misclassification-summary",
            "threshold-sensitivity-development-oof",
            "consistent-false-negatives-original-model",
            "original-model-misclassified-vs-correct-spectra",
            "final-holdout-results",
            "final-holdout-confusion-matrix",
        ],
        "supp": [
            "missing-spectral-value-audit",
            "measurement-completeness-exceptions",
            "logistic-regression-best-parameter-by-fold",
            "original-model-development-leaf-classifications",
            "original-model-final-holdout-misclassifications",
            "logistic-regression-convergence-audit-summary",
            "logistic-regression-convergence-audit-detail",
        ],
    },
    2: {
        "subsections": [
            ("Results", [
                "development-augmentation-size-experiments",
                "development-augmentation-size-experiments-confusion-matrices",
                "development-macro-f1-vs-a-comparison",
                "frozen-final-synthetic-holdout-comparison",
                "frozen-final-holdout-confusion-matrix-comparison",
                "final-holdout-leaf-level-cross-experiment-comparison",
            ]),
            ("A. Generation and Counts", [
                "synthetic-experiment-policy",
                "synthetic-validation-generation-summary",
            ]),
            ("B. Acceptance Checks", [
                "synthetic-validation-per-fold-gate-b",
                "synthetic-validation-per-fold-gate-c",
                "synthetic-validation-pooled-rejection-reasons",
            ]),
            ("C. Quality Checks", [
                "synthetic-validation-pooled-gate-summary",
                "synthetic-validation-suspiciously-close-synthetic-rows",
            ]),
            ("D. Real vs Synthetic Comparison", [
                "synthetic-validation-real-vs-synthetic-overlays",
                "synthetic-validation-mean-sd-spectra",
                "synthetic-validation-final-feature-distributions",
                "synthetic-validation-feature-range-checks",
                "synthetic-validation-descriptive-feature-statistics",
                "synthetic-validation-pca-comparison",
            ]),
        ],
        "supp": [
            "draw-sensitivity-diagnostic",
            "draw-sensitivity-fold-level-results",
            "synthetic-alpha-noise-sensitivity",
        ],
    },
    3: {
        "supp": [
            "wavelength-search-metric-comparison",
            "wavelength-search-paired-diff-summary",
            "wavelength-search-pair-stability",
            "wavelength-search-fold-choices",
        ],
    },
}


def write_validation_html_report(
    output_path: Path,
    report_sections: list[str],
    report_metadata: dict[str, Any],
    report_appendix_sections: list[str] | None = None,
) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    report_appendix_sections = report_appendix_sections or []

    all_sections = list(report_sections) + list(report_appendix_sections)
    section_pool: dict[str, str] = {}
    for _section in all_sections:
        _m = re.search(r'<section id="([^"]+)"', _section)
        if not _m:
            raise ValueError(f"Report section without an id: {_section[:80]!r}")
        _sid = _m.group(1)
        if _sid in section_pool:
            raise ValueError(f"Duplicate report section id: {_sid}")
        section_pool[_sid] = _section

    def _part_groups(_part: int) -> list:
        _cfg = REPORT_SECTION_LAYOUT.get(_part, {})
        _groups: list = []
        if _cfg.get("main"):
            _groups.append((None, list(_cfg["main"]), None))
        for _idx, (_sub_label, _sub_ids) in enumerate(_cfg.get("subsections", [])):
            _groups.append((_sub_label, list(_sub_ids), f"sub-{_idx}"))
        if _cfg.get("supp"):
            _groups.append(("Supplementary", list(_cfg["supp"]), "supplementary"))
        return _groups

    _layout_ids: set[str] = set()
    for _part in REPORT_PARTS:
        for _lbl, _ids, _suf in _part_groups(_part):
            _layout_ids.update(_ids)
    _expected_pool = _layout_ids - {"report-metadata"}
    _extra = set(section_pool) - _expected_pool
    if _extra:
        raise ValueError(
            "Report sections produced but not placed in REPORT_SECTION_LAYOUT "
            f"(would be silently dropped): {sorted(_extra)}. Assign each to a part/subsection/tier."
        )

    def _present(ids: list[str]) -> list[str]:
        return [sid for sid in ids if sid == "report-metadata" or sid in section_pool]

    def _section_title(section_html: str) -> str:
        _t = re.search(r"<h2[^>]*>(.*?)</h2>", section_html, flags=re.S)
        return _t.group(1).strip() if _t else ""

    def _title_for(sid: str) -> str:
        if sid == "report-metadata":
            return "Report Metadata"
        return _section_title(section_pool[sid])

    def _present_groups(_part: int) -> list:
        _out = []
        for _lbl, _ids, _suf in _part_groups(_part):
            _pids = _present(_ids)
            if _pids:
                _out.append((_lbl, _pids, _suf))
        return _out

    _toc_blocks: list[str] = [
        "<h2>Contents</h2>",
        '<ol class="toc-global"><li><a href="#table-export-panel">Export Tables</a></li></ol>',
    ]
    for _part in REPORT_PARTS:
        _groups = _present_groups(_part)
        if not _groups:
            continue
        _items = ""
        for _lbl, _ids, _suf in _groups:
            _sub = "".join(f'<li><a href="#{_sid}">{_title_for(_sid)}</a></li>' for _sid in _ids)
            if _lbl is None:
                _items += _sub
            else:
                _items += (
                    f'<li class="toc-supp-head"><a href="#part-{_part}-{_suf}">{html.escape(_lbl)}</a>'
                    f"<ol>{_sub}</ol></li>"
                )
        _toc_blocks.append(
            f'<div class="toc-part"><a href="#part-{_part}">{html.escape(REPORT_PARTS[_part])}</a>'
            f"<ol>{_items}</ol></div>"
        )
    sidebar_inner = "".join(_toc_blocks)
    metadata_items = "".join(
        f'<tr><th>{html.escape(str(key))}</th><td>{html.escape(str(value))}</td></tr>'
        for key, value in report_metadata.items()
    )
    generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    metadata_section_html = (
        '<section id="report-metadata"><h2>Report Metadata</h2>'
        f'<div class="table-wrap"><table class="meta-table"><tbody>{metadata_items}</tbody></table></div>'
        "</section>"
    )
    _body_blocks: list[str] = []
    for _part in REPORT_PARTS:
        _groups = _present_groups(_part)
        if not _groups:
            continue
        _body_blocks.append(
            f'<h1 class="part-divider" id="part-{_part}">{html.escape(REPORT_PARTS[_part])}</h1>'
        )
        for _lbl, _ids, _suf in _groups:
            if _lbl is not None:
                _body_blocks.append(
                    f'<h2 class="tier-divider" id="part-{_part}-{_suf}">{html.escape(_lbl)}</h2>'
                )
            for _sid in _ids:
                _body_blocks.append(
                    metadata_section_html if _sid == "report-metadata" else section_pool[_sid]
                )
    report_body = "".join(_body_blocks)
    css = """
    :root { color-scheme: light; --ink:#1f2933; --muted:#52606d; --line:#d9e2ec; --soft:#f7f9fb; --green:#0f766e; --red:#b42318; --amber:#b7791f; }
    * { box-sizing: border-box; }
    body { margin: 0; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; color: var(--ink); background: #eef2f6; line-height: 1.55; }
    main { flex: 1 1 0%; min-width: 0; max-width: 1180px; margin: 0 auto; padding: 32px 22px 72px; background: #fff; min-height: 100vh; }
    h1 { margin: 0 0 8px; font-size: 2rem; }
    h2 { margin-top: 42px; border-bottom: 2px solid var(--line); padding-bottom: 8px; font-size: 1.35rem; }
    .report-section h2 { text-align: center; }
    h3 { margin-top: 24px; font-size: 1.05rem; }
    p { color: var(--muted); max-width: 920px; }
    a { color: #1d4ed8; text-decoration: none; }
    a:hover { text-decoration: underline; }
    .meta-table, .report-table { border-collapse: collapse; width: 100%; font-size: 0.92rem; }
    .meta-table th, .meta-table td, .report-table th, .report-table td { border: 1px solid var(--line); padding: 7px 9px; vertical-align: top; }
    .report-table thead th { position: sticky; top: 0; background: #e9eef5; z-index: 1; }
    .report-table tbody tr:nth-child(even) { background: var(--soft); }
    .table-wrap { overflow-x: auto; margin: 14px 0 24px; border: 1px solid var(--line); }
    .table-wrap table { border: 0; }
    .page { display: flex; align-items: flex-start; min-height: 100vh; }
    .sidebar-toc { position: sticky; top: 0; align-self: flex-start; flex: 0 0 260px; width: 260px; max-height: 100vh; overflow-y: auto; background: #f8fafc; border-right: 1px solid var(--line); padding: 20px 18px; }
    .sidebar-toc h2 { margin: 0 0 12px; font-size: 1.05rem; border-bottom: none; padding-bottom: 0; }
    .sidebar-toc ol { margin: 0; padding-left: 20px; }
    .sidebar-toc li { margin-bottom: 8px; font-size: 0.86rem; line-height: 1.35; }
    @media (max-width: 900px) { .page { flex-direction: column; } .sidebar-toc { position: static; width: 100%; max-height: none; border-right: none; border-bottom: 1px solid var(--line); } }
    h1.part-divider { margin: 60px 0 6px; font-size: 1.75rem; color: var(--green); border-bottom: 3px solid var(--green); padding-bottom: 8px; }
    h2.tier-divider { margin-top: 34px; font-size: 1.05rem; color: var(--muted); border-bottom: 1px dashed var(--line); text-transform: uppercase; letter-spacing: 0.05em; }
    .sidebar-toc .toc-global { margin: 0 0 6px; }
    .sidebar-toc .toc-part { margin: 12px 0 4px; }
    .sidebar-toc .toc-part > a { display: block; font-weight: 700; color: var(--green); font-size: 0.9rem; margin-bottom: 4px; }
    .sidebar-toc .toc-supp-head > a { font-style: italic; color: var(--muted); }
    .sidebar-toc .toc-supp-head > ol { margin-top: 6px; }
    details.table-details { margin: 12px 0 24px; }
    details.table-details summary { cursor: pointer; font-weight: 700; color: #334155; }
    .export-panel { background: #f8fafc; border: 1px solid var(--line); padding: 16px 20px; margin: 24px 0; }
    .export-panel h2 { margin-top: 0; }
    .export-actions, .table-export-controls { display: flex; flex-wrap: wrap; align-items: center; gap: 10px; margin: 12px 0; }
    .export-actions button, .table-export-button { border: 1px solid #b7c4d4; background: #fff; color: var(--ink); border-radius: 6px; padding: 7px 11px; cursor: pointer; font: inherit; }
    .export-actions button:hover, .table-export-button:hover { background: #e9eef5; }
    .table-export-check { color: var(--muted); font-size: 0.92rem; }
    .export-status { min-height: 1.4em; font-size: 0.92rem; color: var(--muted); }
    figure { margin: 16px 0 30px; }
    img { max-width: 100%; height: auto; border: 1px solid var(--line); background: #fff; }
    @media print { body { background: #fff; } .sidebar-toc { display: none; } main { max-width: none; padding: 0; margin: 0; } details.table-details { display: block; } details.table-details > * { display: block; } }
    """
    html_doc = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Tipburn Model Validation Report</title>
<style>{css}</style>
</head>
<body>
<div class="page">
<nav class="sidebar-toc">
{sidebar_inner}
</nav>
<main>
<h1>Tipburn Model Validation Report</h1>
<p>Generated at {html.escape(generated_at)}. All figures are included in this file.</p>
<section class="export-panel" id="table-export-panel">
<h2>Export Tables</h2>
<p>Select the required tables and download them as separate CSV files.</p>
<div class="export-actions">
<button type="button" id="download-selected-tables">Download Selected Tables</button>
<button type="button" id="select-all-tables">Select All</button>
<button type="button" id="clear-table-selection">Clear Selection</button>
</div>
<p class="export-status" id="table-export-status"></p>
</section>
{report_body}
<script>
(function () {{
  function sanitizeFilename(value) {{
    return (value || "table")
      .toLowerCase()
      .replace(/[^a-z0-9]+/g, "-")
      .replace(/^-+|-+$/g, "") || "table";
  }}

  function csvEscape(value) {{
    const text = String(value || "").replace(/\\s+/g, " ").trim();
    return /[",\\n]/.test(text) ? '"' + text.replace(/"/g, '""') + '"' : text;
  }}

  function tableToCsv(table) {{
    return Array.from(table.querySelectorAll("tr"))
      .map(row => Array.from(row.querySelectorAll("th, td"))
        .map(cell => csvEscape(cell.innerText))
        .join(","))
      .join("\\n");
  }}

  function downloadCsv(filename, csvText) {{
    const blob = new Blob([csvText], {{ type: "text/csv;charset=utf-8" }});
    const url = URL.createObjectURL(blob);
    const link = document.createElement("a");
    link.href = url;
    link.download = filename;
    document.body.appendChild(link);
    link.click();
    document.body.removeChild(link);
    URL.revokeObjectURL(url);
  }}

  function sectionTitle(section) {{
    const heading = section.querySelector("h2");
    return heading ? heading.innerText : section.id || "table";
  }}

  function downloadSectionTable(section) {{
    const table = section.querySelector("table.report-table");
    if (!table) return false;
    const title = sectionTitle(section);
    downloadCsv(sanitizeFilename(title) + ".csv", tableToCsv(table));
    return true;
  }}

  function updateStatus(message) {{
    const status = document.getElementById("table-export-status");
    if (status) status.innerText = message;
  }}

  document.querySelectorAll(".table-export-button").forEach(button => {{
    button.addEventListener("click", () => {{
      const section = document.getElementById(button.dataset.sectionId);
      if (section && downloadSectionTable(section)) {{
        updateStatus("Downloaded: " + sectionTitle(section));
      }}
    }});
  }});

  const downloadSelected = document.getElementById("download-selected-tables");
  if (downloadSelected) {{
    downloadSelected.addEventListener("click", () => {{
      const selected = Array.from(document.querySelectorAll(".table-export-select:checked"));
      let downloadCount = 0;
      selected.forEach(input => {{
        const section = document.getElementById(input.dataset.sectionId);
        if (section && downloadSectionTable(section)) downloadCount += 1;
      }});
      updateStatus(downloadCount ? "Downloaded " + downloadCount + " selected table(s)." : "No tables selected.");
    }});
  }}

  const selectAll = document.getElementById("select-all-tables");
  if (selectAll) {{
    selectAll.addEventListener("click", () => {{
      document.querySelectorAll(".table-export-select").forEach(input => {{ input.checked = true; }});
      updateStatus("All report tables selected.");
    }});
  }}

  const clearSelection = document.getElementById("clear-table-selection");
  if (clearSelection) {{
    clearSelection.addEventListener("click", () => {{
      document.querySelectorAll(".table-export-select").forEach(input => {{ input.checked = false; }});
      updateStatus("Table selection cleared.");
    }});
  }}
}})();
</script>
</main>
</div>
</body>
</html>
"""
    output_path.write_text(html_doc, encoding="utf-8")

## 8. Locked Final-Model Utilities

Locked five-feature Logistic Regression data checks and repeated grouped-CV helper.

In [8]:
def prepare_locked_final_data(
    features_df: pd.DataFrame,
    train_mask: np.ndarray,
    test_mask: np.ndarray,
    feature_columns: list[str],
) -> dict[str, Any]:
    missing = [feature for feature in feature_columns if feature not in features_df.columns]
    if missing:
        raise AssertionError(f"Missing locked final features: {missing}")
    X_train = features_df.loc[train_mask, feature_columns].reset_index(drop=True)
    X_test = features_df.loc[test_mask, feature_columns].reset_index(drop=True)
    y_train = features_df.loc[train_mask, "y"].to_numpy(int)
    y_test = features_df.loc[test_mask, "y"].to_numpy(int)
    groups_train = features_df.loc[train_mask, "Sample_Group_ID"].to_numpy(str)
    groups_test = features_df.loc[test_mask, "Sample_Group_ID"].to_numpy(str)
    meta_train = features_df.loc[train_mask].reset_index(drop=True)
    meta_test = features_df.loc[test_mask].reset_index(drop=True)
    if X_train.columns.tolist() != feature_columns or X_test.columns.tolist() != feature_columns:
        raise AssertionError("Locked feature order was not preserved.")
    if feature_columns == FINAL_FEATURE_COLS and X_train.columns.tolist() != FINAL_FEATURE_COLS:
        raise AssertionError("Locked feature order does not match FINAL_FEATURE_COLS.")
    if not np.isfinite(X_train.to_numpy(float)).all() or not np.isfinite(X_test.to_numpy(float)).all():
        raise AssertionError("Locked final feature matrices contain non-finite values.")
    if not (len(X_train) == len(y_train) == len(groups_train) == len(meta_train)):
        raise AssertionError("Development row, label, group, and metadata lengths differ.")
    if not (len(X_test) == len(y_test) == len(groups_test) == len(meta_test)):
        raise AssertionError("Holdout row, label, group, and metadata lengths differ.")
    if set(np.unique(y_train)) != {0, 1} or set(np.unique(y_test)) != {0, 1}:
        raise AssertionError("Both classes must exist in development and holdout partitions.")
    if int(features_df.groupby("Sample_Group_ID")["y"].nunique().max()) != 1:
        raise AssertionError("At least one plant has more than one label.")
    if set(groups_train).intersection(set(groups_test)):
        raise AssertionError("Development and holdout plant sets overlap.")
    return {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "groups_train": groups_train,
        "groups_test": groups_test,
        "meta_train": meta_train,
        "meta_test": meta_test,
    }


def run_grouped_cv(
    X: pd.DataFrame,
    y: np.ndarray,
    groups: np.ndarray,
    meta: pd.DataFrame,
    cv_splits: int = 5,
    inner_splits: int = 3,
    random_state: int = RANDOM_STATE,
    model_names: list[str] | None = None,
    scenario_label: str | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    sk = require_sklearn()
    GridSearchCV = sk["GridSearchCV"]
    models, grids = get_models_and_grids(random_state)
    if model_names is not None:
        models = {name: models[name] for name in model_names}
        grids = {name: grids[name] for name in model_names}
    X_arr = X.to_numpy(float)
    outer_cv = make_group_cv(y, groups, cv_splits, random_state)
    metric_rows = []
    prediction_rows = []
    for model_name, pipeline in models.items():
        y_true_all: list[int] = []
        y_pred_all: list[int] = []
        y_prob_all: list[float] = []
        for fold_idx, (train_idx, val_idx) in enumerate(outer_cv.split(X_arr, y, groups), start=1):
            X_train_fold = X_arr[train_idx]
            X_val_fold = X_arr[val_idx]
            y_train_fold = y[train_idx]
            y_val_fold = y[val_idx]
            train_groups_fold = groups[train_idx]
            inner_cv = make_group_cv(y_train_fold, train_groups_fold, inner_splits, random_state + fold_idx)
            search = GridSearchCV(
                estimator=pipeline,
                param_grid=grids[model_name],
                cv=inner_cv,
                scoring="f1_macro",
                refit=True,
                n_jobs=1,
            )
            with catch_convergence_warning() as fold_convergence_state:
                search.fit(X_train_fold, y_train_fold, groups=train_groups_fold)
            if model_name == FINAL_MODEL_NAME:
                # Record convergence from the fitted estimator without changing model fitting.
                lr_convergence_audit_rows.append(build_convergence_record(
                    stage="Model comparison: outer-fold GridSearchCV refit",
                    seed=random_state,
                    outer_fold=fold_idx,
                    scenario=scenario_label or model_name,
                    estimator=search.best_estimator_,
                    convergence_triggered=fold_convergence_state["triggered"],
                ))
            y_pred = search.best_estimator_.predict(X_val_fold)
            y_prob = continuous_model_score(search.best_estimator_, X_val_fold)
            y_true_all.extend(y_val_fold.tolist())
            y_pred_all.extend(y_pred.tolist())
            y_prob_all.extend(y_prob.tolist())
            fold_meta = meta.iloc[val_idx][[*CONTEXT_COLS, *LEAF_ID_COLS, "Split"]].copy()
            fold_meta["CV_Seed"] = random_state
            fold_meta["Fold"] = fold_idx
            fold_meta["Model"] = model_name
            fold_meta["y_true"] = y_val_fold
            fold_meta["y_pred"] = y_pred
            fold_meta["Tipburn_Score"] = y_prob
            fold_meta["Best_Params"] = str(search.best_params_)
            prediction_rows.append(fold_meta)
        metrics = metric_summary(
            np.array(y_true_all),
            np.array(y_pred_all),
            np.array(y_prob_all),
            model_name,
            "train_val_grouped_cv",
        )
        metric_rows.append(metrics)
    cv_metrics = pd.DataFrame(metric_rows).sort_values(["F1_macro", "ROC_AUC"], ascending=False).reset_index(drop=True)
    cv_predictions = pd.concat(prediction_rows, ignore_index=True)
    return cv_metrics, cv_predictions


## 9. Development-Only Synthetic Spectra Utilities

Utilities for the separate synthetic-training experiment. Synthetic spectra are generated only inside each development CV training fold and are passed through the same smoothing, normalization, interpolation, and feature extraction functions as genuine spectra.

In [9]:
SYNTHETIC_ALPHA_LOW = 0.3
SYNTHETIC_ALPHA_HIGH = 0.7
SYNTHETIC_RESIDUAL_SCALE = 0.2
SYNTHETIC_COMBINED_RATIO = 0.5
SYNTHETIC_MAX_ATTEMPTS_MULTIPLIER = 20
FINAL_SYNTHETIC_ONLY_SEED = 4201
FINAL_COMBINED_SEED = 4202

label_name_map = {0: "Healthy", 1: "Tipburn"}

# Match development scenario C to each outer-fold size; the final holdout uses a separate fixed design.

SYNTHETIC_SCENARIOS = [
    {
        "Scenario": "A_original_real_training_only",
        "Display_Name": "A. Original real training data only",
        "Synthetic_Ratio": 0.0,
        "Use_Real_Training": True,
    },
    {
        "Scenario": "B_synthetic_only_equal_real_count",
        "Display_Name": "B. Synthetic training data only, equal to real training count",
        "Synthetic_Ratio": 1.0,
        "Use_Real_Training": False,
    },
    {
        "Scenario": "C_Mixed_FoldMatched_50_50",
        "Display_Name": "C. Fold-size-matched ~50/50 mixed: real and synthetic rows each sized to approximately half of the current outer training fold (N = len(fit_idx)); real + synthetic = N, differing by at most one row when N is odd ",
        "Synthetic_Ratio": 0.5,
        "Use_Real_Training": True,
        "Fold_Matched_50_50": True,
    },
]


def make_repeated_scan_residuals(raw_df: pd.DataFrame,
                                 wavelength_cols: list[str]) -> pd.DataFrame:
    """Return individual-scan residuals relative to their measurement-unit mean."""
    measurement_keys = [*CONTEXT_COLS, "Sample_Group_ID", *ID_COLS]
    residual_source = raw_df[measurement_keys + wavelength_cols].copy()
    residual_source = residual_source.dropna(subset=wavelength_cols).reset_index(drop=True)
    measurement_means = (
        residual_source
        .groupby(measurement_keys, dropna=False)[wavelength_cols]
        .transform("mean")
    )
    residual_values = residual_source[wavelength_cols].to_numpy(float) - measurement_means.to_numpy(float)
    residual_df = residual_source[measurement_keys].copy()
    residual_df[wavelength_cols] = residual_values
    if residual_df.empty:
        raise AssertionError("No repeated-scan residuals were available for synthetic generation.")
    if not np.isfinite(residual_df[wavelength_cols].to_numpy(float)).all():
        raise AssertionError("Repeated-scan residuals contain non-finite values.")
    return residual_df


def _allocate_integer_counts(total_count: int,
                             strata: pd.DataFrame,
                             weight_col: str,
                             tie_cols: list[str]) -> pd.Series:
    if total_count < 0:
        raise ValueError("Allocation count cannot be negative.")
    if strata.empty:
        raise ValueError("Cannot allocate into an empty stratum table.")
    if total_count == 0:
        return pd.Series(0, index=strata.index, dtype=int)
    weights = strata[weight_col].astype(float)
    if weights.sum() <= 0:
        raise ValueError("Allocation weights must sum to a positive value.")
    raw_alloc = weights / weights.sum() * total_count
    base_alloc = np.floor(raw_alloc).astype(int)
    remainder = int(total_count - base_alloc.sum())
    order_df = strata.loc[:, tie_cols].copy()
    order_df["_fractional"] = raw_alloc - base_alloc
    order_df["_weight"] = weights
    order_df["_index"] = strata.index
    order_df = order_df.sort_values(
        ["_fractional", "_weight", *tie_cols, "_index"],
        ascending=[False, False, *([True] * len(tie_cols)), True],
    )
    for idx in order_df["_index"].head(remainder):
        base_alloc.loc[idx] += 1
    if int(base_alloc.sum()) != total_count:
        raise AssertionError("Integer allocation did not preserve the requested total.")
    return base_alloc.astype(int)


def _balanced_two_label_targets(total_count: int, available_by_label: dict[int, int]) -> dict[int, int]:
    """Split ``total_count`` into per-label targets as close to 50/50 as availability allows.

    Used by the Development fold-size-matched Mixed scenario, where the real-subset
    size depends on the current outer fold size rather than a fixed constant, so the Healthy/
    Tipburn split has to be computed per fold instead of read from a fixed {0: 72, 1: 72} dict.
    """
    if total_count < 0:
        raise ValueError("Balanced two-label split count cannot be negative.")
    labels = sorted(available_by_label)
    if labels != [0, 1]:
        raise AssertionError("Balanced two-label split expects exactly labels {0, 1}.")
    base = total_count // 2
    remainder = total_count - 2 * base
    targets = {label: base for label in labels}
    if remainder:
        richer_label = max(labels, key=lambda label: (available_by_label[label], -label))
        targets[richer_label] += remainder
    for label in labels:
        if targets[label] > available_by_label[label]:
            raise AssertionError(
                f"Not enough real fold-training rows for label {label} to reach the fold-matched target "
                f"({targets[label]} requested, {available_by_label[label]} available)."
            )
    return targets


def _select_stratum_positions_preserving_groups(stratum_meta: pd.DataFrame,
                                                requested_count: int,
                                                group_col: str,
                                                rng: np.random.Generator) -> tuple[list[int], bool]:
    """Select exact row count while keeping whole plant groups whenever possible.

    Moved here (from the Final-holdout section) so the Development Mixed scenario
    (C_Mixed_FoldMatched_50_50) and the Final holdout Mixed scenario
    (C_Mixed_Fixed_50_50, ADR 0004) call the exact same balanced-subset selection logic,
    even though their target counts are computed differently (fold-size-matched vs. a
    fixed 72/72 constant).
    """
    if requested_count < 0:
        raise ValueError("Requested subset count cannot be negative.")
    if requested_count == 0:
        return [], False
    if requested_count > len(stratum_meta):
        raise AssertionError("Requested more real rows than are available in a stratum.")
    if requested_count == len(stratum_meta):
        return stratum_meta["_Final_Row_Position"].astype(int).tolist(), False

    group_table = (
        stratum_meta
        .groupby("_Final_Group_Key", dropna=False)
        .agg(Group_Leaf_Count=("_Final_Row_Position", "size"))
        .reset_index()
    )
    group_table["_Order"] = rng.random(len(group_table))
    group_table = group_table.sort_values(["_Order", "_Final_Group_Key"]).reset_index(drop=True)

    subset_sums: dict[int, list[int]] = {0: []}
    for group_idx, group_row in group_table.iterrows():
        group_count = int(group_row["Group_Leaf_Count"])
        for current_total, current_indices in list(subset_sums.items()):
            new_total = current_total + group_count
            if new_total <= requested_count and new_total not in subset_sums:
                subset_sums[new_total] = [*current_indices, int(group_idx)]
        if requested_count in subset_sums:
            break

    if requested_count in subset_sums:
        selected_group_keys = set(group_table.loc[subset_sums[requested_count], "_Final_Group_Key"].astype(str))
        selected_positions = stratum_meta.loc[
            stratum_meta["_Final_Group_Key"].astype(str).isin(selected_group_keys),
            "_Final_Row_Position",
        ].astype(int).tolist()
        return selected_positions, False

    selected_group_keys: list[str] = []
    selected_count = 0
    greedy_groups = group_table.sort_values(["Group_Leaf_Count", "_Order"], ascending=[False, True])
    for _, group_row in greedy_groups.iterrows():
        group_count = int(group_row["Group_Leaf_Count"])
        if selected_count + group_count <= requested_count:
            selected_group_keys.append(str(group_row["_Final_Group_Key"]))
            selected_count += group_count
        if selected_count == requested_count:
            break

    selected_positions = stratum_meta.loc[
        stratum_meta["_Final_Group_Key"].astype(str).isin(selected_group_keys),
        "_Final_Row_Position",
    ].astype(int).tolist()
    remaining_count = requested_count - len(selected_positions)
    if remaining_count > 0:
        remaining_meta = stratum_meta.loc[
            ~stratum_meta["_Final_Group_Key"].astype(str).isin(selected_group_keys)
        ].copy()
        remaining_meta["_Order"] = rng.random(len(remaining_meta))
        selected_positions.extend(
            remaining_meta.sort_values(["_Order", group_col, "Leaf_ID"])
            .head(remaining_count)["_Final_Row_Position"]
            .astype(int)
            .tolist()
        )
    if len(selected_positions) != requested_count:
        raise AssertionError("Balanced real subset selection did not preserve the requested row count.")
    return selected_positions, True


def select_balanced_real_subset(feature_frame: pd.DataFrame,
                                target_values: np.ndarray,
                                meta_frame: pd.DataFrame,
                                target_per_label: dict[int, int],
                                random_seed: int,
                                group_col: str = "Sample_Group_ID",
                                stratify_cols: tuple[str, ...] = ("Week", "DAT"),
                                forbidden_groups: set[str] | None = None) -> dict[str, Any]:
    """Select a reproducible, label-balanced real subset from development rows only.

    Shared by the Development Mixed scenario (C_Mixed_FoldMatched_50_50) and the
    Final holdout Mixed scenario (C_Mixed_Fixed_50_50, ADR 0004), so both stages sample real
    rows the same way even though their target counts differ (fold-size-matched vs. fixed
    72/72).
    """
    if len(feature_frame) != len(target_values) or len(meta_frame) != len(target_values):
        raise AssertionError("Feature, target, and metadata rows must be aligned before balanced real subset selection.")
    if feature_frame.columns.tolist() != FINAL_FEATURE_COLS:
        raise AssertionError("Locked final feature order changed before balanced real subset selection.")
    forbidden_groups = set() if forbidden_groups is None else {str(group) for group in forbidden_groups}
    rng = np.random.default_rng(random_seed)
    y_array = np.asarray(target_values, dtype=int)
    work_meta = meta_frame.reset_index(drop=True).copy()
    work_meta["_Final_Row_Position"] = np.arange(len(work_meta), dtype=int)
    work_meta["_Final_y"] = y_array
    work_meta["_Final_Group_Key"] = work_meta[group_col].astype(str)
    if set(work_meta["_Final_Group_Key"]).intersection(forbidden_groups):
        raise AssertionError("Holdout plants were offered to balanced real subset selection.")

    selected_positions: list[int] = []
    audit_rows: list[dict[str, Any]] = []
    active_stratify_cols = [col for col in stratify_cols if col in work_meta.columns]
    for label_value, requested_label_count in target_per_label.items():
        label_meta = work_meta.loc[work_meta["_Final_y"].eq(int(label_value))].copy()
        if len(label_meta) < requested_label_count:
            raise AssertionError(f"Not enough real development rows for {label_name_map[int(label_value)]}.")
        if active_stratify_cols:
            strata = (
                label_meta
                .groupby(active_stratify_cols, dropna=False)
                .agg(Available_Leaves=("Leaf_ID", "size"))
                .reset_index()
                .sort_values(active_stratify_cols)
                .reset_index(drop=True)
            )
            strata["Requested_Leaves"] = _allocate_integer_counts(
                requested_label_count,
                strata,
                "Available_Leaves",
                active_stratify_cols,
            ).to_numpy(int)
        else:
            strata = pd.DataFrame({"Available_Leaves": [len(label_meta)], "Requested_Leaves": [requested_label_count]})

        for _, stratum in strata.iterrows():
            requested_stratum_count = int(stratum["Requested_Leaves"])
            if requested_stratum_count == 0:
                continue
            stratum_mask = label_meta.index == label_meta.index
            for col in active_stratify_cols:
                value = stratum[col]
                if pd.isna(value):
                    stratum_mask &= label_meta[col].isna()
                else:
                    stratum_mask &= label_meta[col].eq(value)
            stratum_meta = label_meta.loc[stratum_mask].copy()
            stratum_positions, used_partial_group = _select_stratum_positions_preserving_groups(
                stratum_meta,
                requested_stratum_count,
                group_col,
                rng,
            )
            selected_positions.extend(stratum_positions)
            audit_row = {
                "Label": label_name_map[int(label_value)],
                "Available_Leaves": int(len(stratum_meta)),
                "Requested_Leaves": requested_stratum_count,
                "Selected_Leaves": int(len(stratum_positions)),
                "Available_Plants": int(stratum_meta["_Final_Group_Key"].nunique()),
                "Selected_Plants": int(work_meta.loc[stratum_positions, "_Final_Group_Key"].nunique()),
                "Used_Partial_Plant_Group": bool(used_partial_group),
            }
            for col in active_stratify_cols:
                audit_row[col] = stratum[col]
            audit_rows.append(audit_row)

    if len(selected_positions) != len(set(selected_positions)):
        raise AssertionError("Balanced real subset selected a development row more than once.")
    selected_positions = sorted(selected_positions)
    subset_X = feature_frame.iloc[selected_positions].copy().reset_index(drop=True)
    subset_y = y_array[selected_positions].copy()
    subset_meta = meta_frame.iloc[selected_positions].copy().reset_index(drop=True)
    if set(subset_meta[group_col].astype(str)).intersection(forbidden_groups):
        raise AssertionError("Balanced real subset contains holdout plants.")
    for label_value, expected_count in target_per_label.items():
        actual_count = int(np.sum(subset_y == int(label_value)))
        if actual_count != int(expected_count):
            raise AssertionError(
                f"Balanced real subset {label_name_map[int(label_value)]} count is {actual_count}, expected {expected_count}."
            )
    audit = pd.DataFrame(audit_rows).sort_values(["Label", *active_stratify_cols]).reset_index(drop=True)
    return {"X": subset_X, "y": subset_y, "meta": subset_meta, "audit": audit}


def allocate_synthetic_counts_by_label(train_meta: pd.DataFrame,
                                       leaf_pool: pd.DataFrame,
                                       total_synthetic: int) -> pd.DataFrame:
    """Allocate synthetic counts deterministically by Label x Week."""
    if total_synthetic < 0:
        raise ValueError("Synthetic count cannot be negative.")
    if train_meta.empty or leaf_pool.empty:
        raise ValueError("Cannot allocate synthetic rows from an empty training fold.")
    label_table = (
        train_meta
        .groupby("Label", dropna=False)
        .agg(Real_Fold_Training_Leaves=("Leaf_ID", "size"))
        .reset_index()
        .sort_values("Label")
        .reset_index(drop=True)
    )
    base_label_count = total_synthetic // len(label_table)
    label_table["Requested_Label_Count"] = base_label_count
    label_remainder = int(total_synthetic - label_table["Requested_Label_Count"].sum())
    if label_remainder > 0:
        label_order = (
            label_table
            .sort_values(["Real_Fold_Training_Leaves", "Label"], ascending=[False, True])
            .index
            .tolist()
        )
        for idx in label_order[:label_remainder]:
            label_table.loc[idx, "Requested_Label_Count"] += 1
    if label_table["Requested_Label_Count"].max() - label_table["Requested_Label_Count"].min() > 1:
        raise AssertionError("Healthy and Tipburn synthetic totals differ by more than one.")

    source_strata = (
        leaf_pool
        .groupby(["Label", "Week"], dropna=False)
        .agg(
            Real_Fold_Training_Leaves=("Leaf_ID", "size"),
            Source_Plant_Count=("Sample_Group_ID", "nunique"),
            DAT_Count=("DAT", "nunique"),
            DAT_First=("DAT", "first"),
        )
        .reset_index()
        .sort_values(["Label", "Week"])
        .reset_index(drop=True)
    )
    source_strata["DAT"] = np.where(source_strata["DAT_Count"].eq(1), source_strata["DAT_First"], np.nan)
    allocation_frames: list[pd.DataFrame] = []
    for _, label_row in label_table.iterrows():
        label = label_row["Label"]
        label_total = int(label_row["Requested_Label_Count"])
        label_strata = source_strata.loc[source_strata["Label"].eq(label)].copy().reset_index(drop=True)
        if label_strata.empty:
            raise AssertionError(f"No source strata are available for label {label!r}.")
        allocated = _allocate_integer_counts(
            label_total,
            label_strata,
            "Real_Fold_Training_Leaves",
            ["Week"],
        )
        label_strata["Requested_Count"] = allocated.to_numpy(int)
        allocation_frames.append(label_strata)
    allocation_table = pd.concat(allocation_frames, ignore_index=True)
    allocated_positive = allocation_table.loc[allocation_table["Requested_Count"].gt(0)]
    weak_strata = allocated_positive.loc[allocated_positive["Source_Plant_Count"].lt(2)]
    if not weak_strata.empty:
        raise AssertionError(
            "Every allocated synthetic stratum must have at least two source plants: "
            f"{weak_strata[['Label', 'Week', 'Source_Plant_Count']].to_dict('records')}"
        )
    if int(allocation_table["Requested_Count"].sum()) != int(total_synthetic):
        raise AssertionError("Label x Week allocation did not sum to the requested synthetic count.")
    return allocation_table.loc[:, [
        "Label", "Week", "DAT", "Requested_Count", "Real_Fold_Training_Leaves", "Source_Plant_Count"
    ]].sort_values(["Label", "Week", "DAT"], na_position="last").reset_index(drop=True)


def eligible_synthetic_pair_strata(leaf_pool: pd.DataFrame,
                                   label: str,
                                   week: Any) -> pd.DataFrame:
    """Find same-label, same-week strata with optional same-DAT plant support."""
    label_week_pool = leaf_pool.loc[leaf_pool["Label"].eq(label) & leaf_pool["Week"].eq(week)].copy()
    if label_week_pool.empty:
        raise ValueError(f"No real source leaves found for label={label!r}, week={week!r}.")
    dat_strata = (
        label_week_pool
        .groupby(["Label", "Week", "DAT"], dropna=False)
        .agg(Leaf_Count=("Leaf_ID", "size"), Plant_Count=("Sample_Group_ID", "nunique"))
        .reset_index()
    )
    dat_strata = dat_strata.loc[dat_strata["Plant_Count"].ge(2)].copy()
    if not dat_strata.empty:
        dat_strata["Pairing_Level"] = "Label+Week+DAT"
        return dat_strata.sort_values(["Label", "Week", "DAT"]).reset_index(drop=True)
    if label_week_pool["Sample_Group_ID"].nunique() >= 2:
        return pd.DataFrame([{
            "Label": label,
            "Week": week,
            "DAT": np.nan,
            "Leaf_Count": len(label_week_pool),
            "Plant_Count": label_week_pool["Sample_Group_ID"].nunique(),
            "Pairing_Level": "Label+Week",
        }])
    raise AssertionError(
        f"Cannot generate synthetic spectra for label={label!r}, week={week!r}: "
        "no same-label, same-week stratum has at least two different plants."
    )


def subset_for_synthetic_stratum(leaf_pool: pd.DataFrame,
                                 stratum: pd.Series) -> pd.DataFrame:
    mask = leaf_pool["Label"].eq(stratum["Label"]) & leaf_pool["Week"].eq(stratum["Week"])
    if pd.notna(stratum.get("DAT", np.nan)):
        mask &= leaf_pool["DAT"].eq(stratum["DAT"])
    subset = leaf_pool.loc[mask].copy()
    if subset["Sample_Group_ID"].nunique() < 2:
        raise AssertionError("Synthetic source stratum must contain at least two plants.")
    return subset.reset_index(drop=True)


def sample_matching_residual(residual_pool: pd.DataFrame,
                             label: str,
                             week: Any,
                             dat: Any,
                             wavelength_cols: list[str],
                             rng: np.random.Generator) -> tuple[np.ndarray, str, dict[str, Any]]:
    """Sample a real repeated-scan residual from current fold-training plants only."""
    match_specs: list[tuple[str, pd.Series]] = []
    if pd.notna(dat):
        match_specs.append((
            "Label+Week+DAT",
            residual_pool["Label"].eq(label) & residual_pool["Week"].eq(week) & residual_pool["DAT"].eq(dat),
        ))
    match_specs.extend([
        (
            "Label+Week",
            residual_pool["Label"].eq(label) & residual_pool["Week"].eq(week),
        ),
        (
            "Week",
            residual_pool["Week"].eq(week),
        ),
        (
            "AnyFoldTraining",
            pd.Series(True, index=residual_pool.index),
        ),
    ])

    for match_level, mask in match_specs:
        positions = np.flatnonzero(mask.to_numpy())
        if len(positions) == 0:
            continue
        chosen_pos = int(rng.choice(positions))
        chosen_row = residual_pool.iloc[chosen_pos]
        residual = chosen_row[wavelength_cols].to_numpy(dtype=float)
        if not np.isfinite(residual).all():
            raise AssertionError("Sampled repeated-scan residual contains non-finite values.")
        residual_meta = {
            "Residual_Match_Level": match_level,
            "Residual_Source_Plant": chosen_row["Sample_Group_ID"],
            "Residual_Source_Leaf_ID": chosen_row["Leaf_ID"],
            "Residual_Source_Area_ID": chosen_row["Area_ID"],
        }
        return residual, match_level, residual_meta

    raise AssertionError("No repeated-scan residuals are available from the current fold-training plants.")


def calculate_spectral_roughness(values: np.ndarray) -> np.ndarray | float:
    """Mean absolute second difference; lower values indicate smoother spectra."""
    arr = np.asarray(values, dtype=float)
    if arr.ndim == 1:
        if arr.size < 3:
            return 0.0
        return float(np.mean(np.abs(np.diff(arr, n=2))))
    if arr.ndim != 2:
        raise ValueError("Spectral roughness expects a 1D or 2D numeric array.")
    if arr.shape[1] < 3:
        return np.zeros(arr.shape[0], dtype=float)
    return np.mean(np.abs(np.diff(arr, n=2, axis=1)), axis=1)


def derive_synthetic_acceptance_limits(real_fold_training_spectra: pd.DataFrame,
                                       wavelength_cols: list[str]) -> dict[str, Any]:
    """Derive acceptance limits from genuine fold-training raw leaf spectra only."""
    real_values = real_fold_training_spectra[wavelength_cols].to_numpy(dtype=float)
    if real_values.ndim != 2 or real_values.shape[1] != len(wavelength_cols):
        raise AssertionError("Real fold-training spectra have an unexpected wavelength shape.")
    if not np.isfinite(real_values).all():
        raise AssertionError("Real fold-training spectra contain non-finite values.")
    band_q1 = np.percentile(real_values, 25, axis=0)
    band_q3 = np.percentile(real_values, 75, axis=0)
    band_iqr = band_q3 - band_q1
    band_min = np.min(real_values, axis=0)
    band_max = np.max(real_values, axis=0)
    band_epsilon = np.maximum(np.abs(np.median(real_values, axis=0)) * 1e-9, EPS)
    band_tolerance = np.maximum(1.5 * band_iqr, band_epsilon)
    roughness = np.asarray(calculate_spectral_roughness(real_values), dtype=float)
    rough_q1 = float(np.percentile(roughness, 25))
    rough_q3 = float(np.percentile(roughness, 75))
    rough_iqr = rough_q3 - rough_q1
    rough_epsilon = max(float(np.median(roughness)) * 1e-9, EPS)
    rough_tolerance = max(3.0 * rough_iqr, rough_epsilon)
    real_nn = calculate_nearest_real_distances(real_values, real_values, exclude_self=True)
    finite_real_nn = real_nn[np.isfinite(real_nn) & (real_nn > 0)]
    suspicious_threshold = float(np.percentile(finite_real_nn, 1)) if len(finite_real_nn) else 0.0
    return {
        "wavelength_count": len(wavelength_cols),
        "band_low": np.maximum(0.0, band_min - band_tolerance),
        "band_high": band_max + band_tolerance,
        "roughness_low": max(0.0, float(np.min(roughness) - rough_tolerance)),
        "roughness_high": float(np.max(roughness) + rough_tolerance),
        "roughness_tolerance": rough_tolerance,
        "nearest_real_suspicious_threshold": suspicious_threshold,
        "band_limit_method": "per-band min/max expanded by max(1.5*IQR, epsilon) from fold-training spectra",
        "roughness_limit_method": "fold-training roughness min/max expanded by max(3*IQR, epsilon)",
    }


def validate_synthetic_leaf_spectrum(synthetic_spectrum: np.ndarray,
                                     source_a: pd.Series,
                                     source_b: pd.Series,
                                     real_fold_training_spectra: pd.DataFrame,
                                     acceptance_limits: dict[str, Any],
                                     wavelength_cols: list[str]) -> tuple[bool, list[str], dict[str, Any]]:
    """Validate one synthetic raw leaf spectrum using fold-training-derived limits."""
    spectrum = np.asarray(synthetic_spectrum, dtype=float)
    source_a_values = source_a[wavelength_cols].to_numpy(dtype=float)
    source_b_values = source_b[wavelength_cols].to_numpy(dtype=float)
    real_values = real_fold_training_spectra[wavelength_cols].to_numpy(dtype=float)
    reasons: list[str] = []
    if spectrum.ndim != 1 or spectrum.shape[0] != int(acceptance_limits["wavelength_count"]):
        reasons.append("incorrect_wavelength_length")
    if not np.isfinite(spectrum).all():
        reasons.append("non_finite_values")
    if spectrum.shape == source_a_values.shape and np.array_equal(spectrum, source_a_values):
        reasons.append("exact_duplicate_source_a")
    if spectrum.shape == source_b_values.shape and np.array_equal(spectrum, source_b_values):
        reasons.append("exact_duplicate_source_b")
    if spectrum.ndim == 1 and spectrum.shape[0] == real_values.shape[1]:
        exact_real_duplicate = bool(np.any(np.all(real_values == spectrum, axis=1)))
        if exact_real_duplicate:
            reasons.append("exact_duplicate_real_training_spectrum")
        below_limit = spectrum < acceptance_limits["band_low"]
        above_limit = spectrum > acceptance_limits["band_high"]
        if bool(np.any(below_limit | above_limit)):
            reasons.append("outside_training_derived_band_limits")
    roughness = float(calculate_spectral_roughness(spectrum)) if spectrum.ndim == 1 else np.nan
    if np.isfinite(roughness):
        if roughness < acceptance_limits["roughness_low"] or roughness > acceptance_limits["roughness_high"]:
            reasons.append("outside_training_derived_roughness_limits")
    nearest_distance = np.nan
    if spectrum.ndim == 1 and spectrum.shape[0] == real_values.shape[1]:
        nearest_distance = float(calculate_nearest_real_distances(spectrum.reshape(1, -1), real_values)[0])
    metrics = {
        "Synthetic_Roughness": roughness,
        "Nearest_Real_Distance": nearest_distance,
        "Exact_Duplicate_Source_A": "exact_duplicate_source_a" in reasons,
        "Exact_Duplicate_Source_B": "exact_duplicate_source_b" in reasons,
        "Exact_Duplicate_Real_Training": "exact_duplicate_real_training_spectrum" in reasons,
    }
    return len(reasons) == 0, reasons, metrics


def calculate_nearest_real_distances(query_values: np.ndarray,
                                     real_values: np.ndarray,
                                     exclude_self: bool = False) -> np.ndarray:
    """Root-mean-square distance to the nearest real spectrum."""
    query = np.asarray(query_values, dtype=float)
    real = np.asarray(real_values, dtype=float)
    if query.ndim == 1:
        query = query.reshape(1, -1)
    distances = np.sqrt(((query[:, None, :] - real[None, :, :]) ** 2).mean(axis=2))
    if exclude_self and query.shape[0] == real.shape[0] and np.allclose(query, real, rtol=0, atol=0):
        np.fill_diagonal(distances, np.inf)
    return np.min(distances, axis=1)


def generate_synthetic_leaf_spectra(leaf_pool: pd.DataFrame,
                                    residual_pool: pd.DataFrame,
                                    wavelength_cols: list[str],
                                    allocation_table: pd.DataFrame,
                                    rng: np.random.Generator,
                                    source_tag: str,
                                    random_seed: int,
                                    alpha_low: float = SYNTHETIC_ALPHA_LOW,
                                    alpha_high: float = SYNTHETIC_ALPHA_HIGH,
                                    residual_scale: float = SYNTHETIC_RESIDUAL_SCALE) -> dict[str, pd.DataFrame]:
    """Generate accepted synthetic raw leaf-level spectra with rejection logging.

    alpha_low/alpha_high/residual_scale default to the locked module-level constants so
    every existing call site is unchanged. They are parameterized only so a sensitivity
    analysis can override them without touching the mixup formula or rejection logic.
    """
    synthetic_rows: list[dict[str, Any]] = []
    provenance_rows: list[dict[str, Any]] = []
    rejection_rows: list[dict[str, Any]] = []
    generation_rows: list[dict[str, Any]] = []
    safe_tag = re.sub(r"[^A-Za-z0-9]+", "_", source_tag).strip("_")
    acceptance_limits = derive_synthetic_acceptance_limits(leaf_pool, wavelength_cols)

    for _, allocation_row in allocation_table.iterrows():
        requested_count = int(allocation_row["Requested_Count"])
        label = allocation_row["Label"]
        week = allocation_row["Week"]
        if requested_count <= 0:
            generation_rows.append({
                **allocation_row.to_dict(),
                "Accepted_Count": 0,
                "Rejected_Count": 0,
                "Attempt_Count": 0,
                "Max_Attempts": 0,
                "Acceptance_Status": "not_requested",
            })
            continue
        strata = eligible_synthetic_pair_strata(leaf_pool, label, week)
        if pd.notna(allocation_row.get("DAT", np.nan)):
            same_dat_strata = strata.loc[strata["DAT"].eq(allocation_row["DAT"])]
            if not same_dat_strata.empty:
                strata = same_dat_strata.reset_index(drop=True)
        weights = strata["Leaf_Count"].to_numpy(dtype=float)
        weights = weights / weights.sum()
        accepted_count = 0
        rejected_count = 0
        attempt_count = 0
        max_attempts = max(1, requested_count * SYNTHETIC_MAX_ATTEMPTS_MULTIPLIER)

        while accepted_count < requested_count and attempt_count < max_attempts:
            attempt_count += 1
            stratum_pos = int(rng.choice(np.arange(len(strata)), p=weights))
            stratum = strata.iloc[stratum_pos]
            source_subset = subset_for_synthetic_stratum(leaf_pool, stratum)
            source_groups = source_subset["Sample_Group_ID"].drop_duplicates().to_numpy(str)
            group_a, group_b = rng.choice(source_groups, size=2, replace=False)
            row_a_positions = np.flatnonzero(source_subset["Sample_Group_ID"].astype(str).eq(group_a).to_numpy())
            row_b_positions = np.flatnonzero(source_subset["Sample_Group_ID"].astype(str).eq(group_b).to_numpy())
            row_a = source_subset.iloc[int(rng.choice(row_a_positions))]
            row_b = source_subset.iloc[int(rng.choice(row_b_positions))]
            if str(row_a["Sample_Group_ID"]) == str(row_b["Sample_Group_ID"]):
                raise AssertionError("Synthetic source plants must differ.")

            alpha = float(rng.uniform(alpha_low, alpha_high))
            spectrum_a = row_a[wavelength_cols].to_numpy(dtype=float)
            spectrum_b = row_b[wavelength_cols].to_numpy(dtype=float)
            target_dat = row_a["DAT"] if row_a["DAT"] == row_b["DAT"] else np.nan
            residual, residual_match_level, residual_meta = sample_matching_residual(
                residual_pool=residual_pool,
                label=label,
                week=week,
                dat=target_dat,
                wavelength_cols=wavelength_cols,
                rng=rng,
            )
            synthetic_spectrum = alpha * spectrum_a + (1.0 - alpha) * spectrum_b + residual_scale * residual
            is_valid, rejection_reasons, validation_metrics = validate_synthetic_leaf_spectrum(
                synthetic_spectrum=synthetic_spectrum,
                source_a=row_a,
                source_b=row_b,
                real_fold_training_spectra=leaf_pool,
                acceptance_limits=acceptance_limits,
                wavelength_cols=wavelength_cols,
            )
            base_attempt = {
                "Label": label,
                "Week": week,
                "DAT": target_dat,
                "Pairing_Level": stratum["Pairing_Level"],
                "Source_Plant_A": row_a["Sample_Group_ID"],
                "Source_Plant_B": row_b["Sample_Group_ID"],
                "Source_Leaf_A": row_a["Leaf_ID"],
                "Source_Leaf_B": row_b["Leaf_ID"],
                "Source_DAT_A": row_a["DAT"],
                "Source_DAT_B": row_b["DAT"],
                "Mixing_Alpha": alpha,
                "Noise_Method": "fold_training_repeated_scan_residual",
                "Noise_Scale": residual_scale,
                "Residual_Source_Plant": residual_meta["Residual_Source_Plant"],
                "Residual_Match_Level": residual_match_level,
                "Random_Seed": random_seed,
                "Generation_Attempt": attempt_count,
                **validation_metrics,
            }
            if not is_valid:
                rejected_count += 1
                rejection_rows.append({
                    **base_attempt,
                    "Rejection_Reasons": "; ".join(rejection_reasons),
                })
                continue

            accepted_count += 1
            synthetic_id = f"SYN_{safe_tag}_{len(synthetic_rows) + 1:05d}"
            dataset_suffix = f"DAT{int(target_dat)}" if pd.notna(target_dat) else "DATmixed"
            synthetic_row = {
                "Dataset_ID": f"Synthetic_{label}_{str(week).replace(' ', '')}_{dataset_suffix}",
                "Label": label,
                "Week": week,
                "DAT": target_dat,
                "Sample_Group_ID": synthetic_id,
                "Lettuce_Sample_ID": synthetic_id,
                "Leaf_ID": 1,
                "Area_Count": np.nan,
                "Scan_Count_Total": np.nan,
                "Scan_Count_Min": np.nan,
                "Scan_Count_Median": np.nan,
                "Scan_Count_Max": np.nan,
                "Synthetic_ID": synthetic_id,
                "Synthetic_Group_ID": synthetic_id,
                "Is_Synthetic": True,
            }
            synthetic_row.update({col: value for col, value in zip(wavelength_cols, synthetic_spectrum)})
            synthetic_rows.append(synthetic_row)
            provenance_rows.append({
                "Synthetic_ID": synthetic_id,
                "Synthetic_Group_ID": synthetic_id,
                "Is_Synthetic": True,
                **base_attempt,
            })

        status = "complete" if accepted_count == requested_count else "failed"
        generation_rows.append({
            **allocation_row.to_dict(),
            "Accepted_Count": accepted_count,
            "Rejected_Count": rejected_count,
            "Attempt_Count": attempt_count,
            "Max_Attempts": max_attempts,
            "Acceptance_Status": status,
            "Band_Limit_Method": acceptance_limits["band_limit_method"],
            "Roughness_Limit_Method": acceptance_limits["roughness_limit_method"],
        })
        if accepted_count != requested_count:
            raise RuntimeError(
                f"Could not generate requested synthetic spectra for label={label!r}, week={week!r}. "
                f"Requested {requested_count}, accepted {accepted_count}, rejected {rejected_count}, attempts {attempt_count}."
            )

    synthetic_leaf_df = pd.DataFrame(synthetic_rows)
    provenance_df = pd.DataFrame(provenance_rows)
    rejection_df = pd.DataFrame(rejection_rows)
    generation_summary = pd.DataFrame(generation_rows)
    allocation_summary = allocation_table.merge(
        generation_summary[["Label", "Week", "DAT", "Accepted_Count", "Rejected_Count", "Attempt_Count", "Acceptance_Status"]],
        on=["Label", "Week", "DAT"],
        how="left",
    )
    requested_total = int(allocation_table["Requested_Count"].sum())
    if len(synthetic_leaf_df) != requested_total:
        raise AssertionError("Accepted synthetic count does not match requested allocation.")
    if not synthetic_leaf_df.empty:
        if synthetic_leaf_df["Sample_Group_ID"].duplicated().any():
            raise AssertionError("Synthetic Sample_Group_ID values must be unique within a fold/scenario.")
        if (provenance_df["Source_Plant_A"].astype(str) == provenance_df["Source_Plant_B"].astype(str)).any():
            raise AssertionError("Synthetic source pairs must use different plants within each synthetic row.")
    return {
        "accepted_spectra": synthetic_leaf_df,
        "provenance": provenance_df,
        "rejections": rejection_df,
        "allocation_summary": allocation_summary,
        "generation_summary": generation_summary,
    }


def preprocess_synthetic_fold_spectra(synthetic_leaf_df: pd.DataFrame,
                                      wavelength_cols: list[str],
                                      wavelengths: np.ndarray) -> pd.DataFrame:
    """Pass synthetic spectra through the existing validated preprocessing path."""
    if synthetic_leaf_df.empty:
        return pd.DataFrame(columns=[*CONTEXT_COLS, *LEAF_ID_COLS, "y", *FINAL_FEATURE_COLS])
    synthetic_processed = preprocess_leaf_spectra(synthetic_leaf_df, wavelength_cols, wavelengths)
    synthetic_features = synthetic_processed["features"].copy()
    synthetic_features["Split"] = "synthetic_train_fold"
    synthetic_features["Is_Synthetic"] = True
    if not np.isfinite(synthetic_features[FINAL_FEATURE_COLS].to_numpy(float)).all():
        raise AssertionError("Synthetic features contain non-finite values after validated preprocessing.")
    if not synthetic_features["REIP_nm"].between(REIP_MIN, REIP_MAX).all():
        raise AssertionError("Synthetic REIP values are outside the locked red-edge search range.")
    return synthetic_features

def aggregate_fold_metrics(fold_metrics: pd.DataFrame) -> pd.DataFrame:
    metric_columns = [
        "Macro_F1", "ROC_AUC", "Precision_Tipburn", "Recall_Tipburn",
        "Balanced_Accuracy", "Accuracy", "TN", "FP", "FN", "TP",
    ]
    rows = []
    for scenario_cols, subset in fold_metrics.groupby(["Scenario", "Display_Name"], sort=False):
        scenario_name, display_name = scenario_cols
        for metric in metric_columns:
            values = subset[metric].astype(float)
            rows.append({
                "Scenario": scenario_name,
                "Display_Name": display_name,
                "Metric": metric,
                "Count": int(values.count()),
                "Mean": float(values.mean()),
                "Std": float(values.std()),
                "Median": float(values.median()),
                "Minimum": float(values.min()),
                "Maximum": float(values.max()),
            })
    return pd.DataFrame(rows)


def make_paired_combined_minus_original(fold_metrics: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    diff_metrics = ["Macro_F1", "ROC_AUC", "Precision_Tipburn", "Recall_Tipburn", "Balanced_Accuracy", "FP", "FN"]
    original = fold_metrics.loc[fold_metrics["Scenario"].eq("A_original_real_training_only")]
    combined = fold_metrics.loc[fold_metrics["Scenario"].eq("C_Mixed_FoldMatched_50_50")]
    if original.empty or combined.empty:
        # Return empty paired tables when A and C are not both included in the selected scenarios.
        empty_paired = pd.DataFrame(columns=["Seed", "Fold", "Comparison", *[f"{m}_Difference" for m in diff_metrics]])
        empty_summary = pd.DataFrame(columns=["Comparison", "Metric", "Count", "Mean", "Std", "Median", "Minimum", "Maximum"])
        return empty_paired, empty_summary
    merged = combined.merge(
        original,
        on=["Seed", "Fold"],
        suffixes=("_Combined", "_Original"),
    )
    rows = []
    for _, row in merged.iterrows():
        out = {
            "Seed": int(row["Seed"]),
            "Fold": int(row["Fold"]),
            "Comparison": "Combined minus Original",
        }
        for metric in diff_metrics:
            out[f"{metric}_Difference"] = row[f"{metric}_Combined"] - row[f"{metric}_Original"]
        rows.append(out)
    paired = pd.DataFrame(rows)
    summary_rows = []
    for metric in diff_metrics:
        col = f"{metric}_Difference"
        values = paired[col].astype(float)
        summary_rows.append({
            "Comparison": "Combined minus Original",
            "Metric": metric,
            "Count": int(values.count()),
            "Mean": float(values.mean()),
            "Std": float(values.std()),
            "Median": float(values.median()),
            "Minimum": float(values.min()),
            "Maximum": float(values.max()),
        })
    return paired, pd.DataFrame(summary_rows)


def run_development_synthetic_training_experiment(
    real_features: pd.DataFrame,
    real_leaf_spectra: pd.DataFrame,
    repeated_scan_residuals: pd.DataFrame,
    wavelength_cols: list[str],
    wavelengths: np.ndarray,
    forbidden_groups: set[str],
    feature_columns: list[str] = FINAL_FEATURE_COLS,
    repeat_seeds: list[int] = REPEAT_SEEDS,
    alpha_low: float = SYNTHETIC_ALPHA_LOW,
    alpha_high: float = SYNTHETIC_ALPHA_HIGH,
    residual_scale: float = SYNTHETIC_RESIDUAL_SCALE,
    scenarios: list[dict] = SYNTHETIC_SCENARIOS,
    synthetic_seed_offset: int = 0,
) -> dict[str, pd.DataFrame]:
    """Compare original-only, synthetic-only, and real+synthetic training within grouped CV.

    alpha_low/alpha_high/residual_scale/scenarios default to the locked module-level constants,
    so every existing call site behaves identically. They are parameterized only so a
    pre-committed sensitivity analysis can override the synthetic-generator settings and/or
    restrict which scenarios run, without changing the CV, model, threshold, or leakage guards.
    synthetic_seed_offset defaults to 0 (byte-identical to the locked run) and is added only to
    the per-fold synthetic-generation seed, so a draw-sensitivity diagnostic can redraw the
    synthetic leaves while holding folds, real rows, and C's real-subset selection fixed.
    """
    if feature_columns != FINAL_FEATURE_COLS:
        raise AssertionError("Synthetic experiment must use the locked FINAL_FEATURE_COLS.")
    real_features = real_features.reset_index(drop=True)
    real_leaf_spectra = real_leaf_spectra.reset_index(drop=True)
    if len(real_features) != len(real_leaf_spectra):
        raise AssertionError("Development feature rows and raw leaf-level spectra are not aligned.")
    if not real_features["Split"].eq("train_val").all():
        raise AssertionError("Synthetic experiment can only receive development rows.")

    X_real = real_features.loc[:, feature_columns].reset_index(drop=True)
    y_real = real_features["y"].to_numpy(int)
    groups_real = real_features["Sample_Group_ID"].to_numpy(str)
    if set(groups_real).intersection(forbidden_groups):
        raise AssertionError("Holdout groups leaked into synthetic experiment development inputs.")

    prediction_frames: list[pd.DataFrame] = []
    audit_rows: list[dict[str, Any]] = []
    convergence_audit_rows: list[dict[str, Any]] = []
    provenance_frames: list[pd.DataFrame] = []
    rejection_frames: list[pd.DataFrame] = []
    allocation_frames: list[pd.DataFrame] = []
    generation_frames: list[pd.DataFrame] = []
    synthetic_spectra_frames: list[pd.DataFrame] = []
    synthetic_feature_frames: list[pd.DataFrame] = []
    fold_metric_rows: list[dict[str, Any]] = []

    for seed in repeat_seeds:
        outer_cv = make_group_cv(y_real, groups_real, 5, seed)
        seen_by_scenario = {
            scenario["Scenario"]: np.zeros(len(y_real), dtype=bool)
            for scenario in scenarios
        }
        for fold_idx, (fit_idx, val_idx) in enumerate(outer_cv.split(X_real.to_numpy(float), y_real, groups_real), start=1):
            fit_groups = set(groups_real[fit_idx])
            val_groups = set(groups_real[val_idx])
            if fit_groups.intersection(val_groups):
                raise AssertionError("Training and validation plants overlap within a synthetic experiment fold.")
            if fit_groups.intersection(forbidden_groups) or val_groups.intersection(forbidden_groups):
                raise AssertionError("Holdout groups were used inside the synthetic experiment.")

            fold_leaf_pool = real_leaf_spectra.iloc[fit_idx].reset_index(drop=True)
            fold_residual_pool = repeated_scan_residuals.loc[
                repeated_scan_residuals["Sample_Group_ID"].astype(str).isin(fit_groups)
            ].reset_index(drop=True)
            if fold_residual_pool.empty:
                raise AssertionError("No fold-training residuals available for synthetic generation.")

            for scenario_idx, scenario in enumerate(scenarios, start=1):
                scenario_name = scenario["Scenario"]
                display_name = scenario["Display_Name"]
                fold_matched_50_50 = bool(scenario.get("Fold_Matched_50_50", False))
                fixed_real_per_label = None
                if fold_matched_50_50:
                    # Split fold-matched scenario C approximately 50/50 between real and synthetic rows.
                    fold_size = len(fit_idx)
                    c_real_total = fold_size // 2
                    c_synthetic_total = fold_size - c_real_total
                    if abs(c_real_total - c_synthetic_total) > 1:
                        raise AssertionError(
                            f"Scenario {scenario_name}: fold-matched real/synthetic totals differ "
                            f"by more than one row ({c_real_total} vs {c_synthetic_total})."
                        )
                    fold_label_available = {
                        label: int(np.sum(y_real[fit_idx] == label)) for label in (0, 1)
                    }
                    fixed_real_per_label = _balanced_two_label_targets(c_real_total, fold_label_available)
                    synthetic_count = c_synthetic_total
                else:
                    synthetic_count = int(round(len(fit_idx) * float(scenario["Synthetic_Ratio"])))
                synthetic_features = pd.DataFrame(columns=real_features.columns)
                provenance_df = pd.DataFrame()
                rejection_df = pd.DataFrame()
                allocation_summary = pd.DataFrame()
                generation_summary = pd.DataFrame()
                accepted_spectra = pd.DataFrame()

                fixed_real_X = None
                fixed_real_y = None
                fixed_real_meta = None
                if fixed_real_per_label is not None:
                    # Use the same group-preserving balanced selector as the fixed holdout comparison.
                    real_subset_seed = seed * 1_000_000 + fold_idx * 10_000 + scenario_idx * 100
                    try:
                        real_subset = select_balanced_real_subset(
                            feature_frame=X_real.iloc[fit_idx].reset_index(drop=True),
                            target_values=y_real[fit_idx],
                            meta_frame=real_features.iloc[fit_idx].reset_index(drop=True),
                            target_per_label=fixed_real_per_label,
                            random_seed=real_subset_seed,
                            group_col="Sample_Group_ID",
                            forbidden_groups=forbidden_groups,
                        )
                    except AssertionError as exc:
                        raise AssertionError(
                            f"Seed {seed} Fold {fold_idx} Scenario {scenario_name}: {exc}"
                        ) from exc
                    fixed_real_X = real_subset["X"]
                    fixed_real_y = real_subset["y"]
                    fixed_real_meta = real_subset["meta"]

                if synthetic_count > 0:
                    fold_random_seed = synthetic_seed_offset + seed * 100_000 + fold_idx * 1_000 + scenario_idx
                    synth_rng = np.random.default_rng(fold_random_seed)
                    allocation_table = allocate_synthetic_counts_by_label(
                        real_features.iloc[fit_idx],
                        fold_leaf_pool,
                        synthetic_count,
                    )
                    generation_result = generate_synthetic_leaf_spectra(
                        leaf_pool=fold_leaf_pool,
                        residual_pool=fold_residual_pool,
                        wavelength_cols=wavelength_cols,
                        allocation_table=allocation_table,
                        rng=synth_rng,
                        source_tag=f"seed{seed}_fold{fold_idx}_{scenario_name}",
                        random_seed=fold_random_seed,
                        alpha_low=alpha_low,
                        alpha_high=alpha_high,
                        residual_scale=residual_scale,
                    )
                    accepted_spectra = generation_result["accepted_spectra"]
                    provenance_df = generation_result["provenance"]
                    rejection_df = generation_result["rejections"]
                    allocation_summary = generation_result["allocation_summary"]
                    generation_summary = generation_result["generation_summary"]
                    synthetic_features = preprocess_synthetic_fold_spectra(
                        accepted_spectra,
                        wavelength_cols,
                        wavelengths,
                    )
                    for frame in [accepted_spectra, provenance_df, synthetic_features, allocation_summary, generation_summary, rejection_df]:
                        if not frame.empty:
                            frame.insert(0, "Scenario", scenario_name)
                            frame.insert(1, "Display_Name", display_name)
                            frame.insert(2, "Seed", seed)
                            frame.insert(3, "Fold", fold_idx)
                    provenance_frames.append(provenance_df)
                    rejection_frames.append(rejection_df)
                    allocation_frames.append(allocation_summary)
                    generation_frames.append(generation_summary)
                    synthetic_spectra_frames.append(accepted_spectra)
                    synthetic_feature_frames.append(synthetic_features)

                X_parts: list[np.ndarray] = []
                y_parts: list[np.ndarray] = []
                real_used_y = np.array([], dtype=int)
                real_used_groups: set[str] = set()
                if bool(scenario["Use_Real_Training"]):
                    if fixed_real_per_label is not None:
                        X_parts.append(fixed_real_X.loc[:, feature_columns].to_numpy(float))
                        y_parts.append(fixed_real_y)
                        real_used_y = fixed_real_y
                        real_used_groups = set(fixed_real_meta["Sample_Group_ID"].astype(str))
                    else:
                        X_parts.append(X_real.iloc[fit_idx].to_numpy(float))
                        y_parts.append(y_real[fit_idx])
                        real_used_y = y_real[fit_idx]
                        real_used_groups = set(groups_real[fit_idx])
                if synthetic_count > 0:
                    X_parts.append(synthetic_features.loc[:, feature_columns].to_numpy(float))
                    y_parts.append(synthetic_features["y"].to_numpy(int))
                if not X_parts:
                    raise AssertionError(f"Scenario {scenario_name} has no training rows.")

                X_fit = np.vstack(X_parts)
                y_fit = np.concatenate(y_parts)
                if set(np.unique(y_fit)) != {0, 1}:
                    raise AssertionError(f"Scenario {scenario_name} training fold lacks both classes.")

                synthetic_used_y = synthetic_features["y"].to_numpy(int) if synthetic_count > 0 else np.array([], dtype=int)
                synthetic_used_groups = (
                    set(synthetic_features["Sample_Group_ID"].astype(str))
                    if synthetic_count > 0 and not synthetic_features.empty else set()
                )
                holdout_rows_used_in_training = len((real_used_groups | synthetic_used_groups) & forbidden_groups)
                holdout_rows_used_in_generation = len(
                    (fit_groups | set(fold_residual_pool["Sample_Group_ID"].astype(str))) & forbidden_groups
                )
                if holdout_rows_used_in_training != 0:
                    raise AssertionError(f"Scenario {scenario_name}: holdout plants used in training.")
                if holdout_rows_used_in_generation != 0:
                    raise AssertionError(f"Scenario {scenario_name}: holdout plants used in synthetic generation.")

                fold_model = clone(build_locked_final_pipeline(RANDOM_STATE))
                with catch_convergence_warning() as fold_convergence_state:
                    fold_model.fit(X_fit, y_fit)
                convergence_audit_rows.append(build_convergence_record(
                    stage="Development synthetic A/B/C CV: outer-fold refit",
                    seed=seed,
                    outer_fold=fold_idx,
                    scenario=scenario_name,
                    estimator=fold_model,
                    convergence_triggered=fold_convergence_state["triggered"],
                ))
                X_val = X_real.iloc[val_idx].to_numpy(float)
                fold_score = continuous_model_score(fold_model, X_val)
                fold_pred = (fold_score >=0.5).astype(int)
                model_pred = fold_model.predict(X_val)
                if not np.array_equal(fold_pred, model_pred):
                    raise AssertionError("Explicit 0.5 threshold predictions differ from model.predict().")

                fold_meta = real_features.iloc[val_idx][[*CONTEXT_COLS, *LEAF_ID_COLS, "Split"]].copy()
                fold_meta["Scenario"] = scenario_name
                fold_meta["Display_Name"] = display_name
                fold_meta["Seed"] = seed
                fold_meta["Fold"] = fold_idx
                fold_meta["Model"] = FINAL_MODEL_NAME
                fold_meta["Locked_C"] = FINAL_MODEL_C
                fold_meta["Validation_Row_Source"] = "real"
                fold_meta["Synthetic_Rows_In_Validation"] = 0
                fold_meta["y_true"] = y_real[val_idx]
                fold_meta["y_pred"] = fold_pred
                fold_meta["Tipburn_Score"] = fold_score
                prediction_frames.append(fold_meta)
                if seen_by_scenario[scenario_name][val_idx].any():
                    raise AssertionError(f"Scenario {scenario_name}, seed {seed} assigned a validation row twice.")
                seen_by_scenario[scenario_name][val_idx] = True

                fold_metrics = binary_classification_metrics(
                    y_real[val_idx],
                    fold_pred,
                    fold_score,
                    "train_val_grouped_cv_real_validation_fold",
                    FINAL_MODEL_NAME,
                )
                fold_metrics.update({
                    "Scenario": scenario_name,
                    "Display_Name": display_name,
                    "Seed": seed,
                    "Fold": fold_idx,
                    "Synthetic_Ratio": scenario["Synthetic_Ratio"],
                    "Uses_Real_Training_Rows": bool(scenario["Use_Real_Training"]),
                    "Decision_Threshold": 0.5,
                })
                fold_metric_rows.append(fold_metrics)

                pairing_counts = provenance_df["Pairing_Level"].value_counts().to_dict() if not provenance_df.empty else {}
                residual_counts = provenance_df["Residual_Match_Level"].value_counts().to_dict() if not provenance_df.empty else {}
                rejected_count = int(len(rejection_df)) if not rejection_df.empty else 0
                accepted_count = int(len(accepted_spectra)) if not accepted_spectra.empty else 0
                real_healthy = int(np.sum(real_used_y == 0))
                real_tipburn = int(np.sum(real_used_y == 1))
                synthetic_healthy = int(np.sum(synthetic_used_y == 0))
                synthetic_tipburn = int(np.sum(synthetic_used_y == 1))
                audit_rows.append({
                    "Experiment_Name": "Development OOF CV",
                    "Scenario": scenario_name,
                    "Display_Name": display_name,
                    "Seed": seed,
                    "Fold": fold_idx,
                    # Record class counts using the same fields as the final holdout audit.
                    "Real_Healthy": real_healthy,
                    "Real_Tipburn": real_tipburn,
                    "Synthetic_Healthy": synthetic_healthy,
                    "Synthetic_Tipburn": synthetic_tipburn,
                    "Real_Total": int(len(real_used_y)),
                    "Synthetic_Total": int(len(synthetic_used_y)),
                    "Total_Training_Rows": int(len(y_fit)),
                    "Validation_Real_Rows": int(len(val_idx)),
                    "Validation_Synthetic_Rows": 0,
                    "Holdout_Rows_Used_In_Training": holdout_rows_used_in_training,
                    "Holdout_Rows_Used_In_Generation": holdout_rows_used_in_generation,
                    # Retain these column names for compatibility with existing report outputs.
                    "Real_Source_Leaves": int(len(fit_idx)),
                    "Real_Training_Leaves": int(len(real_used_y)),
                    "Requested_Synthetic_Leaves": int(synthetic_count),
                    "Accepted_Synthetic_Leaves": accepted_count,
                    "Rejected_Synthetic_Attempts": rejected_count,
                    "Total_Training_Leaves": int(len(y_fit)),
                    "Real_Validation_Leaves": int(len(val_idx)),
                    "Synthetic_Validation_Leaves": 0,
                    "Training_Validation_Plant_Overlap": 0,
                    "Holdout_Plants_Used": holdout_rows_used_in_training + holdout_rows_used_in_generation,
                    "Pairing_Level_Counts": str(pairing_counts),
                    "Residual_Match_Level_Counts": str(residual_counts),
                })

        for scenario_name, seen in seen_by_scenario.items():
            if not seen.all():
                raise AssertionError(f"Scenario {scenario_name}, seed {seed} left development rows without real OOF validation.")

    def concat_nonempty(frames: list[pd.DataFrame]) -> pd.DataFrame:
        nonempty = [frame for frame in frames if not frame.empty]
        return pd.concat(nonempty, ignore_index=True) if nonempty else pd.DataFrame()

    predictions = pd.concat(prediction_frames, ignore_index=True)
    fold_audit = pd.DataFrame(audit_rows)
    provenance = concat_nonempty(provenance_frames)
    rejections = concat_nonempty(rejection_frames)
    allocations = concat_nonempty(allocation_frames)
    generation_summary = concat_nonempty(generation_frames)
    synthetic_spectra = concat_nonempty(synthetic_spectra_frames)
    synthetic_features_all = concat_nonempty(synthetic_feature_frames)

    if not predictions["Split"].eq("train_val").all():
        raise AssertionError("Synthetic experiment predictions include non-development rows.")
    if not predictions["Validation_Row_Source"].eq("real").all():
        raise AssertionError("Synthetic experiment validation rows must all be genuine real leaves.")
    if set(predictions["Sample_Group_ID"].astype(str)).intersection(forbidden_groups):
        raise AssertionError("Holdout plants appear in synthetic experiment predictions.")
    if predictions.duplicated(["Scenario", "Seed", "Sample_Group_ID", "Leaf_ID"]).any():
        raise AssertionError("Duplicate OOF prediction rows found for a scenario/seed/leaf.")
    expected_prediction_count = len(real_features)
    prediction_counts = predictions.groupby(["Scenario", "Seed"]).size()
    if not prediction_counts.eq(expected_prediction_count).all():
        raise AssertionError("Each scenario/seed must validate every development leaf exactly once.")
    if not provenance.empty:
        if (provenance["Source_Plant_A"].astype(str) == provenance["Source_Plant_B"].astype(str)).any():
            raise AssertionError("A synthetic spectrum used the same plant for both source spectra.")
        source_groups = set(provenance["Source_Plant_A"].astype(str)).union(set(provenance["Source_Plant_B"].astype(str)))
        residual_groups = set(provenance["Residual_Source_Plant"].astype(str))
        if source_groups.intersection(forbidden_groups) or residual_groups.intersection(forbidden_groups):
            raise AssertionError("Holdout plants appear in synthetic spectrum provenance.")
    if not generation_summary.empty:
        if not generation_summary["Requested_Count"].eq(generation_summary["Accepted_Count"]).all():
            raise AssertionError("Synthetic generation silently reduced a requested count.")

    metric_rows: list[dict[str, Any]] = []
    for (scenario_name, seed), subset in predictions.groupby(["Scenario", "Seed"], sort=False):
        scenario = next(item for item in scenarios if item["Scenario"] == scenario_name)
        metrics = binary_classification_metrics(
            subset["y_true"].to_numpy(int),
            subset["y_pred"].to_numpy(int),
            subset["Tipburn_Score"].to_numpy(float),
            "train_val_grouped_cv_real_validation",
            FINAL_MODEL_NAME,
        )
        metrics.update({
            "Scenario": scenario_name,
            "Display_Name": scenario["Display_Name"],
            "Seed": int(seed),
            "Synthetic_Ratio": scenario["Synthetic_Ratio"],
            "Uses_Real_Training_Rows": bool(scenario["Use_Real_Training"]),
            "Feature_Order": ", ".join(feature_columns),
            "Decision_Threshold": 0.5,
        })
        metric_rows.append(metrics)

    seed_metrics = pd.DataFrame(metric_rows)
    metric_column_order = [
        "Scenario", "Display_Name", "Seed", "Synthetic_Ratio", "Uses_Real_Training_Rows",
        "Model", "Split", "n", "Macro_F1", "F1_macro", "ROC_AUC", "Precision_Tipburn", "Recall_Tipburn",
        "Precision_Healthy", "Recall_Healthy", "Balanced_Accuracy", "Accuracy", "TN", "FP", "FN", "TP",
        "Decision_Threshold", "Feature_Order",
    ]
    seed_metrics = seed_metrics.loc[:, metric_column_order].sort_values(["Scenario", "Seed"]).reset_index(drop=True)
    summary = (
        seed_metrics
        .groupby(["Scenario", "Display_Name", "Synthetic_Ratio", "Uses_Real_Training_Rows", "Model"], as_index=False)
        .agg(
            Seed_Count=("Seed", "nunique"),
            Macro_F1_mean=("Macro_F1", "mean"),
            Macro_F1_sd=("Macro_F1", "std"),
            ROC_AUC_mean=("ROC_AUC", "mean"),
            ROC_AUC_sd=("ROC_AUC", "std"),
            Tipburn_recall_mean=("Recall_Tipburn", "mean"),
            Tipburn_recall_sd=("Recall_Tipburn", "std"),
            Tipburn_precision_mean=("Precision_Tipburn", "mean"),
            Tipburn_precision_sd=("Precision_Tipburn", "std"),
            Balanced_accuracy_mean=("Balanced_Accuracy", "mean"),
            Balanced_accuracy_sd=("Balanced_Accuracy", "std"),
        )
        .sort_values("Scenario")
        .reset_index(drop=True)
    )
    fold_metrics_df = pd.DataFrame(fold_metric_rows).sort_values(["Scenario", "Seed", "Fold"]).reset_index(drop=True)
    fold_metric_aggregate = aggregate_fold_metrics(fold_metrics_df)
    paired_fold_differences, paired_fold_difference_summary = make_paired_combined_minus_original(fold_metrics_df)
    return {
        "summary": summary,
        "seed_metrics": seed_metrics,
        "fold_metrics": fold_metrics_df,
        "fold_metric_aggregate": fold_metric_aggregate,
        "paired_fold_differences": paired_fold_differences,
        "paired_fold_difference_summary": paired_fold_difference_summary,
        "fold_audit": fold_audit,
        "predictions": predictions,
        "provenance": provenance,
        "rejections": rejections,
        "allocations": allocations,
        "generation_summary": generation_summary,
        "synthetic_spectra": synthetic_spectra,
        "synthetic_features": synthetic_features_all,
        "convergence_audit": pd.DataFrame(convergence_audit_rows),
    }


## 10. Initialize Report State

Mutable report state used by the later sections.

In [10]:

validation_report_sections: list[str] = []
validation_report_appendix_sections: list[str] = []
report_metadata: dict[str, Any] = {}
# Collect convergence metadata without affecting model fitting, predictions, or metrics.
lr_convergence_audit_rows: list[dict[str, Any]] = []



## 11. Load, Preprocess, and Integrity Checks

Loads the four workbooks, preserves the fixed plant-level split, verifies no plant leakage, and appends dataset/split/audit tables to the HTML report.

In [11]:
raw_df, wavelength_cols, wavelengths = load_all_datasets(DATASET_DIR)
leaf_df = group_to_leaf_level(raw_df, wavelength_cols)
processed = preprocess_leaf_spectra(leaf_df, wavelength_cols, wavelengths)
features_df = processed["features"].copy()

split_labels, _ = split_final_test_by_sample(
    features_df,
    test_size=FINAL_TEST_SIZE,
    random_state=RANDOM_STATE,
)
features_df["Split"] = split_labels
leaf_df["Split"] = split_labels

train_mask = features_df["Split"].eq("train_val").to_numpy()
test_mask = features_df["Split"].eq("final_test").to_numpy()
features_train = features_df.loc[train_mask].reset_index(drop=True)
features_test = features_df.loc[test_mask].reset_index(drop=True)

assert_quality_checks(features_df, processed["raw"], processed["normalized"], wavelengths)
if set(features_df["Split"].unique()) != {"train_val", "final_test"}:
    raise AssertionError("Split column must contain exactly train_val and final_test.")
if features_df["Split"].isna().any():
    raise AssertionError("Every row must belong to exactly one split.")
if set(features_train["y"].unique()) != {0, 1} or set(features_test["y"].unique()) != {0, 1}:
    raise AssertionError("Both classes must exist in each split.")
if int(features_df.groupby("Sample_Group_ID")["Split"].nunique().max()) != 1:
    raise AssertionError("At least one plant occurs in more than one split.")
development_groups = set(features_train["Sample_Group_ID"].astype(str))
holdout_groups = set(features_test["Sample_Group_ID"].astype(str))
plant_overlap = development_groups.intersection(holdout_groups)
if plant_overlap:
    raise AssertionError(f"Development and holdout plant groups overlap: {sorted(plant_overlap)}")

audit_tables = make_audit_tables(raw_df, leaf_df, wavelength_cols)
missing_summary = audit_tables["missing_spectral_audit"]
validation_report_appendix_sections.append(
    dataframe_to_report_section(
        "Missing spectral-value audit",
        missing_summary,
        "Missing spectral values before leaf averaging. Zero means no scan rows were excluded.",
        section_id="missing-spectral-value-audit",
    )
)

locked_data = prepare_locked_final_data(features_df, train_mask, test_mask, FINAL_FEATURE_COLS)
X_train_final = locked_data["X_train"]
X_test_final = locked_data["X_test"]
y_train_final = locked_data["y_train"]
y_test_final = locked_data["y_test"]
groups_train_final = locked_data["groups_train"]
groups_test_final = locked_data["groups_test"]
meta_train_final = locked_data["meta_train"]
meta_test_final = locked_data["meta_test"]
print("Plant leakage and final-data integrity checks passed.")

registry_rows = []
for dataset_id, spec in DATASET_REGISTRY.items():
    registry_rows.append({
        "Dataset_ID": dataset_id,
        "Filename": spec.filename,
        "Label": spec.label,
        "Week": spec.week,
        "DAT": spec.dat,
        "Resolved_Path": str(resolve_dataset_path(DATASET_DIR, spec.filename)),
    })
dataset_registry_report = pd.DataFrame(registry_rows)

report_metadata.update({
    "Generated": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "Random state": RANDOM_STATE,
    "Final holdout size": FINAL_TEST_SIZE,
    "Repeated CV seeds": REPEAT_SEEDS,
    "Selected model": FINAL_MODEL_NAME,
    "Selected hyperparameter": f"clf__C = {FINAL_MODEL_C}",
    "Final feature order": ", ".join(FINAL_FEATURE_COLS),
    "Decision threshold": "0.5",
    "Prediction unit": "One lettuce leaf",
    "Grouping unit": "One lettuce plant / Sample_Group_ID",
    "Dataset filenames": "; ".join(dataset_registry_report["Filename"].tolist()),
    "Holdout policy": "Fixed plant-level 20% holdout; not used for retuning",
})

dataset_summary = pd.DataFrame([
    {"Metric": "Leaf-level rows", "Value": len(features_df)},
    {"Metric": "Wavelength bands", "Value": len(wavelengths)},
    {"Metric": "Wavelength range", "Value": f"{int(wavelengths.min())}-{int(wavelengths.max())} nm"},
    {"Metric": "Healthy leaves", "Value": int(features_df["Label"].eq("Healthy").sum())},
    {"Metric": "Tipburn leaves", "Value": int(features_df["Label"].eq("Tipburn").sum())},
    {"Metric": "Total plants", "Value": features_df["Sample_Group_ID"].nunique()},
])
validation_report_sections.append(dataframe_to_report_section(
    "Dataset and preprocessing",
    dataset_summary,
    "Summary of leaf averaging, smoothing, normalization and feature calculation.",
    section_id="dataset-and-preprocessing-summary",
))
split_summary = pd.DataFrame([
    {"Metric": "Development leaves", "Value": len(features_train)},
    {"Metric": "Holdout leaves", "Value": len(features_test)},
    {"Metric": "Development plants", "Value": len(development_groups)},
    {"Metric": "Holdout plants", "Value": len(holdout_groups)},
])
split_class_counts = (
    features_df
    .groupby(["Label", "Split"])
    .size()
    .unstack("Split", fill_value=0)
    .rename(columns={"train_val": "Development leaves", "final_test": "Holdout leaves"})
    .reset_index()
)
validation_report_sections.append(dataframe_to_report_section(
    "Plant-level data split",
    split_summary,
    "The same plant-level development and holdout split is used throughout the analysis.",
    section_id="fixed-plant-level-split-summary",
))
validation_report_sections.append(dataframe_to_report_section(
    "Class counts by split",
    split_class_counts,
    "Healthy and Tipburn leaf counts in each split.",
    section_id="split-class-counts",
))

measurement_cols = [
    "Dataset_ID", "Label", "Week", "DAT", "Sample_Group_ID", "Lettuce_Sample_ID", "Leaf_ID",
    "Area_Count", "Scan_Count_Total", "Scan_Count_Min", "Scan_Count_Median", "Scan_Count_Max", "Split",
]
measurement_exceptions = features_df.loc[
    (features_df["Area_Count"].ne(3))
    | (features_df["Scan_Count_Total"].ne(30))
    | (features_df["Scan_Count_Min"].ne(10))
    | (features_df["Scan_Count_Median"].ne(10))
    | (features_df["Scan_Count_Max"].ne(10)),
    measurement_cols,
].reset_index(drop=True)
print(f"Measurement-completeness exceptions retained for review: {len(measurement_exceptions)}")
if not measurement_exceptions.empty:
    validation_report_appendix_sections.append(
        dataframe_to_report_section(
            "Incomplete measurements",
            measurement_exceptions,
            "Leaves with area or scan counts different from the expected three areas and ten scans. These leaves were retained.",
            collapsible=True,
            section_id="measurement-completeness-exceptions",
        )
    )

Plant leakage and final-data integrity checks passed.
Measurement-completeness exceptions retained for review: 20


## 11.1 Smoothing-Window Sensitivity

This post-hoc robustness analysis compares the retained 11-band Savitzky–Golay setting with neighbouring smoothing windows using only the fixed development partition. The independent holdout plants are excluded from all smoothing-window comparisons.

In [12]:
development_leaf_df = (
    leaf_df.loc[train_mask]
    .reset_index(drop=True)
    .copy()
)

smoothing_development_groups = set(
    development_leaf_df[
        "Sample_Group_ID"
    ].astype(str)
)

if smoothing_development_groups != development_groups:
    raise AssertionError(
        "The smoothing analysis does not use exactly "
        "the fixed development plant set."
    )

if smoothing_development_groups.intersection(
    holdout_groups
):
    raise AssertionError(
        "Final-holdout plants entered the smoothing analysis."
    )


smoothing_results = build_reip_smoothing_sensitivity(
    development_leaf_df=development_leaf_df,
    wavelength_cols=wavelength_cols,
    wavelengths=wavelengths,
    windows=SMOOTHING_WINDOWS,
)

smoothing_leaf_results = smoothing_results[
    "leaf_results"
]

smoothing_plant_results = smoothing_results[
    "plant_results"
]

smoothing_effect_table = smoothing_results[
    "report_table"
]


# Confirm that the sensitivity helper exactly reproduces the
# operational 11-band REIP values from the main preprocessing path.
comparison_keys = [
    *CONTEXT_COLS,
    *LEAF_ID_COLS,
]

sensitivity_window_11 = (
    smoothing_leaf_results.loc[
        smoothing_leaf_results[
            "Window_Length"
        ].eq(DEFAULT_WINDOW),
        [
            *comparison_keys,
            "REIP_nm",
        ],
    ]
    .sort_values(comparison_keys)
    .reset_index(drop=True)
)

production_window_11 = (
    features_train[
        [
            *comparison_keys,
            "REIP_nm",
        ]
    ]
    .sort_values(comparison_keys)
    .reset_index(drop=True)
)

if not sensitivity_window_11[
    comparison_keys
].astype(str).equals(
    production_window_11[
        comparison_keys
    ].astype(str)
):
    raise AssertionError(
        "The 11-band sensitivity rows do not align "
        "with the main preprocessing rows."
    )

np.testing.assert_array_equal(
    sensitivity_window_11[
        "REIP_nm"
    ].to_numpy(),
    production_window_11[
        "REIP_nm"
    ].to_numpy(),
)

validation_report_sections.append(
    dataframe_to_report_section(
        "Smoothing window sensitivity analysis",
        smoothing_effect_table,
        (
            "Development-only check of how the smoothing window affects REIP. "
            "Cohen's d is Tipburn minus Healthy. The 11-band setting was retained."
        ),
        section_id=(
            "development-smoothing-window-sensitivity"
        ),
    )
)


print(
    "Development-only smoothing sensitivity completed."
)

print(
    "The 11-band results exactly reproduce the main "
    "preprocessing pipeline, and no holdout plant was used."
)

display(
    smoothing_effect_table.round(2)
)

Development-only smoothing sensitivity completed.
The 11-band results exactly reproduce the main preprocessing pipeline, and no holdout plant was used.


,Window (bands),Width (nm),Week 3,Week 4
0,3,15,-1.88,-1.74
1,5,25,-1.62,-1.65
2,7,35,-1.46,-1.74
3,9,45,-1.43,-1.78
4,11,55,-1.82,-2.04
5,13,65,-1.77,-1.82
6,15,75,-1.41,-1.60
7,21,105,-0.78,-1.48


## 12. Six-Feature Candidate EDA

Development-only six-feature descriptive statistics and feature distribution figure.

In [13]:
eda_descriptive = (
    features_train
    .melt(id_vars=["Label"], value_vars=EDA_FEATURE_COLS, var_name="Feature", value_name="Value")
    .groupby(["Feature", "Label"], as_index=False)
    .agg(
        Count=("Value", "size"),
        Mean=("Value", "mean"),
        Standard_deviation=("Value", "std"),
        Median=("Value", "median"),
        Minimum=("Value", "min"),
        Maximum=("Value", "max"),
    )
)
validation_report_sections.append(dataframe_to_report_section(
    "Six-feature candidate EDA",
    eda_descriptive,
    "Summary of the six candidate features in the development set. Values are in original units; blue rows are Healthy and red rows are Tipburn.",
    section_id="six-feature-candidate-eda",
    row_tint_col="Label",
    row_tint_colors={"Healthy": "#dbeafe", "Tipburn": "#fde2e0"},
))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, feature in zip(axes.flat, EDA_FEATURE_COLS):
    if feature == "REIP_nm":
        # Show REIP class composition on its 5 nm grid; use boxplots for the other features.
        reip_composition = (
            features_train.groupby([feature, "Label"]).size()
            .unstack("Label", fill_value=0)
            .reindex(columns=["Healthy", "Tipburn"], fill_value=0)
            .sort_index()
        )
        reip_totals = reip_composition.sum(axis=1)
        reip_pct = reip_composition.div(reip_totals, axis=0) * 100.0
        healthy_pct = reip_pct["Healthy"].to_numpy()
        tipburn_pct = reip_pct["Tipburn"].to_numpy()
        bar_x = np.arange(len(reip_composition))
        ax.bar(bar_x, healthy_pct, width=0.72, color="#2f6fbb", label="Healthy")
        ax.bar(bar_x, tipburn_pct, width=0.72, bottom=healthy_pct, color="#c84c3a", label="Tipburn")
        for xi in bar_x:
            if healthy_pct[xi] >= 4:
                ax.text(xi, healthy_pct[xi] / 2, f"{healthy_pct[xi]:.0f}% H", ha="center", va="center", color="white", fontsize=8)
            if tipburn_pct[xi] >= 4:
                ax.text(xi, healthy_pct[xi] + tipburn_pct[xi] / 2, f"{tipburn_pct[xi]:.0f}% T", ha="center", va="center", color="white", fontsize=8)
        ax.set_xticks(bar_x)
        ax.set_xticklabels([f"{int(v)}\nn = {int(reip_totals.iloc[i])}" for i, v in enumerate(reip_composition.index)])
        ax.set_yticks([0, 20, 40, 60, 80, 100])
        ax.set_ylim(-5, 109)  # Add vertical padding for readability.
        ax.set_ylabel("% of dev leaves")
        ax.set_title(feature)
        ax.set_xlabel("")
        continue
    sns.boxplot(data=features_train, x="Label", y=feature, hue="Label", legend=False, ax=ax, palette={"Healthy": "#2f6fbb", "Tipburn": "#c84c3a"})
    sns.stripplot(data=features_train, x="Label", y=feature, ax=ax, color="#111827", alpha=0.28, size=2.3)
    ax.set_title(feature)
    ax.set_xlabel("")
plt.tight_layout()
validation_report_sections.append(figure_to_base64_html(
    fig,
    "Six-feature distributions by label",
    "Boxplots compare Healthy and Tipburn for five features. REIP is shown in 5 nm groups. These plots are exploratory only.",
    section_id="six-feature-distributions-by-label",
))


## 13. Spectral Overview

Development-only normalized, derivative, and smoothed raw spectral overview figures.

In [14]:
normalized_train = processed["normalized"][train_mask]
fig, ax = plt.subplots(figsize=(11, 5.5))
for label, color in [("Healthy", "#2f6fbb"), ("Tipburn", "#c84c3a")]:
    mask = features_train["Label"].eq(label).to_numpy()
    label_values = normalized_train[mask]
    mean_curve = label_values.mean(axis=0)
    std_curve = label_values.std(axis=0)
    ax.plot(wavelengths, mean_curve, color=color, linewidth=2.4, label=f"{label} mean")
    ax.fill_between(wavelengths, mean_curve - std_curve, mean_curve + std_curve, color=color, alpha=0.14, linewidth=0)
ax.set_title("Development normalized spectral overview")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Row-wise min-max normalized reflectance")
ax.legend()
sns.despine(ax=ax)
plt.tight_layout()
validation_report_sections.append(figure_to_base64_html(
    fig,
    "Basic spectral overview",
    "Mean normalized spectra and standard-deviation bands for Healthy and Tipburn development leaves.",
    section_id="basic-spectral-overview",
))

## 14. PLS-DA Overview

Development-only PLS-DA score summaries and embedded score plots.

In [15]:
def make_pls_da_scores(values: np.ndarray, y: np.ndarray, meta: pd.DataFrame, title: str) -> tuple[pd.DataFrame, Any]:
    sk = require_sklearn()
    PLSRegression = sk["PLSRegression"]
    n_components = min(2, values.shape[1], values.shape[0] - 1)
    pls = PLSRegression(n_components=n_components, scale=True)
    scores = pls.fit_transform(values, y)[0]
    scores_df = meta[[*CONTEXT_COLS, *LEAF_ID_COLS, "Split"]].copy()
    scores_df["PLS_DA_1"] = scores[:, 0]
    scores_df["PLS_DA_2"] = scores[:, 1] if n_components > 1 else 0.0
    fig, ax = plt.subplots(figsize=(8.5, 6.2))
    for dataset_id in GROUP_ORDER:
        style = GROUP_STYLE[dataset_id]
        mask = scores_df["Dataset_ID"].eq(dataset_id)
        if mask.any():
            ax.scatter(scores_df.loc[mask, "PLS_DA_1"], scores_df.loc[mask, "PLS_DA_2"], label=style["label"], color=style["color"], alpha=0.78, s=42, edgecolor="white", linewidth=0.4)
    ax.axhline(0, color="#888", linewidth=0.8)
    ax.axvline(0, color="#888", linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("PLS-DA component 1")
    ax.set_ylabel("PLS-DA component 2")
    ax.legend(loc="best", frameon=True, fontsize=9)
    sns.despine(ax=ax)
    plt.tight_layout()
    return scores_df, fig

_, fig = make_pls_da_scores(
    features_train[EDA_FEATURE_COLS].to_numpy(float),
    features_train["y"].to_numpy(int),
    features_train,
    "Six-feature PLS-DA scores",
)
validation_report_sections.append(figure_to_base64_html(
    fig,
    "Six-feature PLS-DA",
    "Exploratory PLS-DA using the six candidate features. Separation here does not prove performance on unseen plants or select the final model.",
    section_id="spectral-and-six-feature-pls-da-overview",
))
_, fig = make_pls_da_scores(
    normalized_train,
    features_train["y"].to_numpy(int),
    features_train,
    "Full-spectrum PLS-DA scores",
)
validation_report_sections.append(figure_to_base64_html(
    fig,
    "Full-spectrum PLS-DA",
    "Exploratory PLS-DA using all processed wavelengths. Its axes are not directly comparable with the six-feature plot.",
    section_id="full-spectrum-pls-da-overview",
))


## 15. Development Model Comparison

Grouped development cross-validation comparison for engineered feature sets and candidate models.

In [16]:
y_train_all = features_train["y"].to_numpy(int)
groups_train_all = features_train["Sample_Group_ID"].to_numpy(str)
all_feature_set_metrics = []
for feature_set_name, feature_columns in FEATURE_SETS.items():
    feature_metrics, _ = run_grouped_cv(
        X=features_train.loc[:, feature_columns].reset_index(drop=True),
        y=y_train_all,
        groups=groups_train_all,
        meta=features_train,
        cv_splits=5,
        inner_splits=3,
        random_state=RANDOM_STATE,
        scenario_label=feature_set_name,
    )
    feature_metrics.insert(0, "Feature_Set", feature_set_name)
    feature_metrics.insert(1, "Feature_Count", len(feature_columns))
    feature_metrics.insert(2, "Feature_Names", ", ".join(feature_columns))
    all_feature_set_metrics.append(feature_metrics)
feature_set_model_comparison = (
    pd.concat(all_feature_set_metrics, ignore_index=True)
    .sort_values(
        "F1_macro",
        ascending=False,
    )
    .reset_index(drop=True)
)
# Identify the highest Macro-F1 within each feature set.
best_f1_by_feature_set = (
    feature_set_model_comparison
    .groupby("Feature_Set")["F1_macro"]
    .transform("max")
)
best_in_feature_set_mask = (
    feature_set_model_comparison["F1_macro"]
   .eq(best_f1_by_feature_set)
)
best_model_per_feature_set = (
    feature_set_model_comparison
    .loc[best_in_feature_set_mask]
    .reset_index(drop=True)
)
best_row_positions = set(
    feature_set_model_comparison
    .index[best_in_feature_set_mask]
)
full_feature_pool = FEATURE_SETS["6_features"]
feature_set_display = feature_set_model_comparison.copy()
feature_set_display["Best_In_Feature_Set"] = feature_set_display.index.isin(best_row_positions)
feature_set_display["Excluded_Features"] = feature_set_display["Feature_Names"].apply(
    lambda names: ", ".join(f for f in full_feature_pool if f not in names.split(", ")) or "None"
)
display_split_name = feature_set_model_comparison["Split"].iloc[0]
display_n = int(feature_set_model_comparison["n"].iloc[0])
feature_set_display = feature_set_display.drop(columns=["Split", "n", "Feature_Names"])
validation_report_sections.append(dataframe_to_report_section(
    "Model comparison",
    feature_set_display,
    f"Nested plant-grouped CV results for all feature sets and classifiers "
    f"(split: {display_split_name}, n={display_n}). "
    "Excluded_Features lists features omitted from the six-feature pool. "
    "Best_In_Feature_Set marks the highest Macro-F1 result within each "
    "feature set. Tipburn recall and ROC-AUC are supporting metrics. "
    "This screening stage is separate from final model selection.",
    section_id="feature-set-and-model-comparison",
))

## 16. Repeated Grouped-CV Finalist Comparison

Repeated plant-grouped cross-validation for the locked final Logistic Regression candidate and the SVM comparator.

In [17]:
FINAL_CANDIDATES = {
    "6_features_SVM": {
        "model": "SVM_RBF",
        "features": ["REIP_nm", "NDRE", "Red_Edge_Slope", "NDVI", "NIR_Mean_Norm", "NIR_Mean_SmoothedRaw"],
    },
    "5_features_LogisticRegression": {
        "model": FINAL_MODEL_NAME,
        "features": FINAL_FEATURE_COLS,
    },
}
all_repeated_metrics = []
all_repeated_prediction_rows = []

for candidate_name, candidate_config in FINAL_CANDIDATES.items():
    model_name = candidate_config["model"]
    feature_columns = candidate_config["features"]

    for seed in REPEAT_SEEDS:
        candidate_metrics, candidate_predictions = run_grouped_cv(
            X=features_train.loc[:, feature_columns].reset_index(drop=True),
            y=y_train_all,
            groups=groups_train_all,
            meta=features_train,
            cv_splits=5,
            inner_splits=3,
            random_state=seed,
            model_names=[model_name],
            scenario_label=candidate_name,
        )

        candidate_predictions["Candidate"] = candidate_name
        all_repeated_prediction_rows.append(candidate_predictions)

        metric_row = candidate_metrics.iloc[0].to_dict()
        metric_row["Candidate"] = candidate_name
        metric_row["Seed"] = seed
        metric_row["Feature_Count"] = len(feature_columns)
        metric_row["Feature_Names"] = ", ".join(feature_columns)
        all_repeated_metrics.append(metric_row)

repeated_cv_predictions = pd.concat(
    all_repeated_prediction_rows,
    ignore_index=True,
)
repeated_cv_results = pd.DataFrame(all_repeated_metrics)
result_columns = ["Candidate", "Seed", "Feature_Count", "Model", "F1_macro", "ROC_AUC", "Precision_Tipburn", "Recall_Tipburn", "Precision_Healthy", "Recall_Healthy", "Balanced_Accuracy", "TN", "FP", "FN", "TP", "Feature_Names"]
repeated_cv_results = repeated_cv_results[result_columns].sort_values(["Candidate", "Seed"]).reset_index(drop=True)
repeated_cv_summary = (
    repeated_cv_results
    .groupby(["Candidate", "Feature_Count", "Model"], as_index=False)
    .agg(
        F1_mean=("F1_macro", "mean"),
        F1_sd=("F1_macro", "std"),
        Tipburn_recall_mean=("Recall_Tipburn", "mean"),
        Tipburn_recall_sd=("Recall_Tipburn", "std"),
        Tipburn_precision_mean=("Precision_Tipburn", "mean"),
        Tipburn_precision_sd=("Precision_Tipburn", "std"),
        Balanced_accuracy_mean=("Balanced_Accuracy", "mean"),
        Balanced_accuracy_sd=("Balanced_Accuracy", "std"),
        ROC_AUC_mean=("ROC_AUC", "mean"),
        ROC_AUC_sd=("ROC_AUC", "std"),
        FP_mean=("FP", "mean"),
        FP_sd=("FP", "std"),
        FN_mean=("FN", "mean"),
        FN_sd=("FN", "std"),
    )
    .sort_values(["F1_mean", "Tipburn_recall_mean", "ROC_AUC_mean"], ascending=[False, False, False])
    .reset_index(drop=True)
)

# Summarise how often each C value was selected across repeated grouped-CV folds.
best_param_by_fold = (
    repeated_cv_predictions
    .loc[
        (
            repeated_cv_predictions["Candidate"]
            == "5_features_LogisticRegression"
        )
        & (
            repeated_cv_predictions["Model"]
            == FINAL_MODEL_NAME
        ),
        ["CV_Seed", "Fold", "Best_Params"],
    ]
    .drop_duplicates()
    .sort_values(["CV_Seed", "Fold"])
    .reset_index(drop=True)
)

best_param_summary = (
    best_param_by_fold
    .groupby("Best_Params", as_index=False)
    .size()
    .rename(columns={"size": "Selection_Count"})
    .sort_values(
        "Selection_Count",
        ascending=False,
    )
    .reset_index(drop=True)
)

validation_report_sections.append(
    dataframe_to_report_section(
        "Best C selection frequency",
        best_param_summary,
        (
            "Number of times each C value was selected by inner CV across the 25 outer folds."
        ),
        section_id="logistic-regression-best-parameter-selection-frequency",
    )
)

validation_report_appendix_sections.append(
    dataframe_to_report_section(
        "Best C by fold",
        best_param_by_fold,
        "C value selected within each repeated plant-grouped fold.",
        collapsible=True,
        section_id="logistic-regression-best-parameter-by-fold",
    )
)

locked_candidate_df = pd.DataFrame([
    {
        "Model": FINAL_MODEL_NAME,
        "Locked_C": FINAL_MODEL_C,
        "Feature_Order": ", ".join(FINAL_FEATURE_COLS),
        "Development_Matrix_Shape": str(X_train_final.shape),
        "Holdout_Matrix_Shape": str(X_test_final.shape),
        "Development_Plants": len(set(groups_train_final)),
        "Holdout_Plants": len(set(groups_test_final)),
    }
])
validation_report_sections.append(dataframe_to_report_section(
    "Locked final model",
    locked_candidate_df,
    "Five-feature Logistic Regression selected using development-data validation.",
    section_id="locked-final-candidate",
))

### 16.1 Decoupled 2x2 Finalist Comparison (Model x Feature Count)

The earlier comparison combined model and feature-count differences. This analysis therefore tested Logistic Regression and RBF SVM using both five and six features to separate these effects.

Candidates were ranked by mean Macro-F1 across repeated grouped CV. Near-ties were resolved using lower variability, higher Tipburn recall and fewer features. Only development data were used; the final holdout remained untouched.

In [18]:
# Compare Logistic Regression and RBF SVM with five and six features on development data only.

two_by_two_dev_groups = set(features_train["Sample_Group_ID"].astype(str))
if not two_by_two_dev_groups:
    raise AssertionError("features_train is empty; expected a non-empty development-only set of plants.")
if two_by_two_dev_groups & holdout_groups:
    raise AssertionError(
        "features_train overlaps holdout_groups -- the 2x2 comparison must run on development-only data."
    )

TWO_BY_TWO_CANDIDATES = {
    "6_features_LogisticRegression": {
        "model": FINAL_MODEL_NAME,
        "features": FEATURE_SETS["6_features"],
    },
    "6_features_SVM_RBF": {
        "model": "SVM_RBF",
        "features": FEATURE_SETS["6_features"],
    },
    # Reuse cached five-feature Logistic Regression results to avoid an identical refit.
    "5_features_SVM_RBF": {
        "model": "SVM_RBF",
        "features": FINAL_FEATURE_COLS,
    },
}

two_by_two_all_metrics = []
for candidate_name, candidate_config in TWO_BY_TWO_CANDIDATES.items():
    model_name = candidate_config["model"]
    feature_columns = candidate_config["features"]
    for seed in REPEAT_SEEDS:
        candidate_metrics, candidate_predictions = run_grouped_cv(
            X=features_train.loc[:, feature_columns].reset_index(drop=True),
            y=y_train_all,
            groups=groups_train_all,
            meta=features_train,
            cv_splits=5,
            inner_splits=3,
            random_state=seed,
            model_names=[model_name],
            scenario_label=candidate_name,
        )
        metric_row = candidate_metrics.iloc[0].to_dict()
        metric_row["Candidate"] = candidate_name
        metric_row["Seed"] = seed
        metric_row["Feature_Count"] = len(feature_columns)
        metric_row["Feature_Names"] = ", ".join(feature_columns)
        two_by_two_all_metrics.append(metric_row)

_cached_5lr_rows = [
    dict(_row) for _row in all_repeated_metrics
    if _row["Candidate"] == "5_features_LogisticRegression"
]
if len(_cached_5lr_rows) != len(REPEAT_SEEDS):
    raise AssertionError(
        f"Expected {len(REPEAT_SEEDS)} cached 5_features_LogisticRegression rows from the repeated finalist comparison, "
        f"got {len(_cached_5lr_rows)}."
    )
two_by_two_all_metrics.extend(_cached_5lr_rows)

two_by_two_result_columns = [
    "Candidate", "Seed", "Feature_Count", "Model", "F1_macro", "ROC_AUC",
    "Precision_Tipburn", "Recall_Tipburn", "Precision_Healthy", "Recall_Healthy",
    "TN", "FP", "FN", "TP", "Feature_Names",
]
two_by_two_results = (
    pd.DataFrame(two_by_two_all_metrics)[two_by_two_result_columns]
    .sort_values(["Candidate", "Seed"])
    .reset_index(drop=True)
)

two_by_two_summary = (
    two_by_two_results
    .groupby(["Candidate", "Feature_Count", "Model"], as_index=False)
    .agg(
        F1_mean=("F1_macro", "mean"),
        F1_sd=("F1_macro", "std"),
        Tipburn_recall_mean=("Recall_Tipburn", "mean"),
        Tipburn_recall_sd=("Recall_Tipburn", "std"),
        Tipburn_precision_mean=("Precision_Tipburn", "mean"),
        Tipburn_precision_sd=("Precision_Tipburn", "std"),
        ROC_AUC_mean=("ROC_AUC", "mean"),
        ROC_AUC_sd=("ROC_AUC", "std"),
        FP_mean=("FP", "mean"),
        FP_sd=("FP", "std"),
        FN_mean=("FN", "mean"),
        FN_sd=("FN", "std"),
    )
    .sort_values("F1_mean", ascending=False)
    .reset_index(drop=True)
)

# Apply the fixed tie rule: mean Macro-F1, lower SD, higher Tipburn recall, then fewer features.
TWO_BY_TWO_TIE_THRESHOLD = float(two_by_two_summary["F1_sd"].max())
two_by_two_leader = two_by_two_summary.iloc[0]
two_by_two_tie_band_lower = float(two_by_two_leader["F1_mean"] - TWO_BY_TWO_TIE_THRESHOLD)
two_by_two_tied = two_by_two_summary.loc[
    two_by_two_summary["F1_mean"] >= two_by_two_tie_band_lower
].reset_index(drop=True)
two_by_two_selected = (
    two_by_two_tied
    .sort_values(["F1_sd", "Tipburn_recall_mean", "Feature_Count"], ascending=[True, False, True])
    .iloc[0]
)

print(f"Tie threshold (largest seed-to-seed macro-F1 SD across the 4 candidates): {TWO_BY_TWO_TIE_THRESHOLD:.4f}")
print(f"Leader by mean macro-F1: {two_by_two_leader['Candidate']} (F1_mean={two_by_two_leader['F1_mean']:.4f})")
print(f"Tie band: candidates with F1_mean >= {two_by_two_tie_band_lower:.4f}")
print(f"Tied candidates: {two_by_two_tied['Candidate'].tolist()}")
print(
    f"Selected candidate: {two_by_two_selected['Candidate']} "
    f"(Model={two_by_two_selected['Model']}, Feature_Count={int(two_by_two_selected['Feature_Count'])}, "
    f"F1_mean={two_by_two_selected['F1_mean']:.4f}, F1_sd={two_by_two_selected['F1_sd']:.4f}, "
    f"Tipburn_recall_mean={two_by_two_selected['Tipburn_recall_mean']:.4f})"
)

# Compare model families at each fixed feature count.
print("\n=== Decoupled read-out: model effect (LogisticRegression vs SVM_RBF) ===")
for feature_count in (6, 5):
    lr_row = two_by_two_summary.loc[
        (two_by_two_summary["Feature_Count"] == feature_count)
        & (two_by_two_summary["Model"] == FINAL_MODEL_NAME)
    ].iloc[0]
    svm_row = two_by_two_summary.loc[
        (two_by_two_summary["Feature_Count"] == feature_count)
        & (two_by_two_summary["Model"] == "SVM_RBF")
    ].iloc[0]
    print(
        f"{feature_count} features: LR F1_mean={lr_row['F1_mean']:.4f} vs SVM F1_mean={svm_row['F1_mean']:.4f} "
        f"(diff LR-SVM={lr_row['F1_mean'] - svm_row['F1_mean']:+.4f}); "
        f"LR Tipburn_recall_mean={lr_row['Tipburn_recall_mean']:.4f} vs "
        f"SVM Tipburn_recall_mean={svm_row['Tipburn_recall_mean']:.4f}"
    )

# Compare five versus six features within each model family.
print("\n=== Decoupled read-out: feature-count effect (6 vs 5 features -- the NIR_Mean_Norm test) ===")
for model_name in (FINAL_MODEL_NAME, "SVM_RBF"):
    row_6 = two_by_two_summary.loc[
        (two_by_two_summary["Feature_Count"] == 6) & (two_by_two_summary["Model"] == model_name)
    ].iloc[0]
    row_5 = two_by_two_summary.loc[
        (two_by_two_summary["Feature_Count"] == 5) & (two_by_two_summary["Model"] == model_name)
    ].iloc[0]
    print(
        f"{model_name}: F1_mean@6={row_6['F1_mean']:.4f} vs F1_mean@5={row_5['F1_mean']:.4f} "
        f"(diff 6-5={row_6['F1_mean'] - row_5['F1_mean']:+.4f}); "
        f"Tipburn_recall_mean@6={row_6['Tipburn_recall_mean']:.4f} vs "
        f"Tipburn_recall_mean@5={row_5['Tipburn_recall_mean']:.4f}"
    )

two_by_two_summary_display = two_by_two_summary.sort_values("F1_sd", ascending=True).reset_index(drop=True)
validation_report_sections.append(dataframe_to_report_section(
    "Four finalist comparison",
    two_by_two_summary_display,
    "Repeated grouped-CV results for five or six features with Logistic Regression "
    "or RBF SVM. The holdout was not used. Selected: "
    f"{two_by_two_selected['Candidate']}.",
    section_id="two-by-two-finalist-comparison",
))
validation_report_sections.append(dataframe_to_report_section(
    "Finalist results by seed",
    two_by_two_results,
    "Repeated grouped-CV results for each finalist and seed.",
    collapsible=True,
    section_id="two-by-two-finalist-seed-level-results",
))

Tie threshold (largest seed-to-seed macro-F1 SD across the 4 candidates): 0.0205
Leader by mean macro-F1: 6_features_LogisticRegression (F1_mean=0.9104)
Tie band: candidates with F1_mean >= 0.8898
Tied candidates: ['6_features_LogisticRegression', '5_features_LogisticRegression', '5_features_SVM_RBF', '6_features_SVM_RBF']
Selected candidate: 5_features_LogisticRegression (Model=LogisticRegression, Feature_Count=5, F1_mean=0.9048, F1_sd=0.0063, Tipburn_recall_mean=0.8806)

=== Decoupled read-out: model effect (LogisticRegression vs SVM_RBF) ===
6 features: LR F1_mean=0.9104 vs SVM F1_mean=0.8950 (diff LR-SVM=+0.0154); LR Tipburn_recall_mean=0.8944 vs SVM Tipburn_recall_mean=0.8667
5 features: LR F1_mean=0.9048 vs SVM F1_mean=0.9034 (diff LR-SVM=+0.0014); LR Tipburn_recall_mean=0.8806 vs SVM Tipburn_recall_mean=0.8875

=== Decoupled read-out: feature-count effect (6 vs 5 features -- the NIR_Mean_Norm test) ===
LogisticRegression: F1_mean@6=0.9104 vs F1_mean@5=0.9048 (diff 6-5=+0.0056); 

## 17. Dummy Baseline

Development grouped-CV dummy classifier baseline, metrics, and predictions.

In [19]:
X_dummy = X_train_final.to_numpy(float)
y_dummy = y_train_final.copy()
groups_dummy = groups_train_final.copy()
dummy_cv_seed = RANDOM_STATE
outer_cv = make_group_cv(y_dummy, groups_dummy, 5, dummy_cv_seed)
y_true_all, y_pred_all, y_score_all = [], [], []
dummy_prediction_rows = []
for fold_idx, (fit_idx, val_idx) in enumerate(outer_cv.split(X_dummy, y_dummy, groups_dummy), start=1):
    dummy_model = DummyClassifier(strategy="most_frequent")
    dummy_model.fit(X_dummy[fit_idx], y_dummy[fit_idx])
    fold_pred = dummy_model.predict(X_dummy[val_idx])
    probabilities = dummy_model.predict_proba(X_dummy[val_idx])
    class_positions = {int(class_label): idx for idx, class_label in enumerate(dummy_model.classes_)}
    fold_score = probabilities[:, class_positions[1]] if 1 in class_positions else np.zeros(len(val_idx), dtype=float)
    y_true_all.extend(y_dummy[val_idx].tolist())
    y_pred_all.extend(fold_pred.tolist())
    y_score_all.extend(fold_score.tolist())
    fold_meta = meta_train_final.iloc[val_idx][[*CONTEXT_COLS, *LEAF_ID_COLS, "Split"]].copy()
    fold_meta["Fold"] = fold_idx
    fold_meta["CV_Seed"] = dummy_cv_seed
    fold_meta["Model"] = "Dummy_MostFrequent"
    fold_meta["y_true"] = y_dummy[val_idx]
    fold_meta["y_pred"] = fold_pred
    fold_meta["Tipburn_Score"] = fold_score
    dummy_prediction_rows.append(fold_meta)
dummy_metrics = pd.DataFrame([
    metric_summary(np.asarray(y_true_all, dtype=int), np.asarray(y_pred_all, dtype=int), np.asarray(y_score_all, dtype=float), "Dummy_MostFrequent", "train_val_grouped_cv")
])
dummy_metrics["CV_Seed"] = dummy_cv_seed
reference_lr = repeated_cv_summary.loc[repeated_cv_summary["Candidate"].eq("5_features_LogisticRegression") & repeated_cv_summary["Model"].eq(FINAL_MODEL_NAME)]
if not reference_lr.empty:
    dummy_metrics["Reference_LR_Repeated_CV_F1_mean"] = float(reference_lr["F1_mean"].iloc[0])
    dummy_metrics["Reference_LR_Repeated_CV_F1_sd"] = float(reference_lr["F1_sd"].iloc[0])
dummy_predictions = pd.concat(dummy_prediction_rows, ignore_index=True)
validation_report_sections.append(dataframe_to_report_section("Dummy baseline", dummy_metrics, "Most-frequent-class baseline using the same plant-grouped development folds.", section_id="dummy-baseline"))
validation_report_sections.append(dataframe_to_report_section("Dummy predictions", dummy_predictions, "Complete OOF predictions from the dummy baseline.", collapsible=True, section_id="dummy-baseline-predictions"))

## 18. Plant-Level Label-Permutation Check

Plant-level label-shuffle check comparing the real-label grouped-CV score against shuffled-label scores.

In [20]:
N_PERMUTATIONS = 199
PERMUTATION_SEED = RANDOM_STATE
REAL_LABEL_CV_SEED = RANDOM_STATE
X_perm = X_train_final.to_numpy(float)
y_perm_original = y_train_final.copy()
groups_perm = groups_train_final.copy()
locked_lr_pipeline = build_locked_final_pipeline(RANDOM_STATE)

def grouped_oof_macro_f1(X: np.ndarray, y: np.ndarray, groups: np.ndarray, estimator, cv_seed: int) -> float:
    cv = make_group_cv(y, groups, 5, cv_seed)
    oof_pred = np.full(len(y), -1, dtype=int)
    for fit_idx, val_idx in cv.split(X, y, groups):
        fold_model = clone(estimator)
        fold_model.fit(X[fit_idx], y[fit_idx])
        oof_pred[val_idx] = fold_model.predict(X[val_idx])
    if (oof_pred < 0).any():
        raise RuntimeError("Some development rows did not receive an OOF prediction.")
    return float(f1_score(y, oof_pred, average="macro", zero_division=0))

real_label_f1 = grouped_oof_macro_f1(X_perm, y_perm_original, groups_perm, locked_lr_pipeline, REAL_LABEL_CV_SEED)
plant_label_table = pd.DataFrame({"Sample_Group_ID": groups_perm, "y": y_perm_original}).drop_duplicates().sort_values("Sample_Group_ID").reset_index(drop=True)
rng = np.random.default_rng(PERMUTATION_SEED)
permutation_rows = []
for permutation_id in range(1, N_PERMUTATIONS + 1):
    permutation_cv_seed = RANDOM_STATE + permutation_id
    shuffled_group_labels = rng.permutation(plant_label_table["y"].to_numpy(int))
    group_to_shuffled_label = dict(zip(plant_label_table["Sample_Group_ID"], shuffled_group_labels))
    shuffled_y = np.asarray([group_to_shuffled_label[group] for group in groups_perm], dtype=int)
    shuffled_f1 = grouped_oof_macro_f1(X_perm, shuffled_y, groups_perm, locked_lr_pipeline, permutation_cv_seed)
    permutation_rows.append({"Permutation": permutation_id, "Permutation_Seed": PERMUTATION_SEED, "CV_Seed": permutation_cv_seed, "Shuffled_Macro_F1": shuffled_f1})
permutation_scores = pd.DataFrame(permutation_rows)
permutation_p_value = (1 + int((permutation_scores["Shuffled_Macro_F1"] >= real_label_f1).sum())) / (N_PERMUTATIONS + 1)
permutation_summary = pd.DataFrame([
    {
        "Real_Label_Macro_F1": real_label_f1,
        "Real_Label_CV_Seed": REAL_LABEL_CV_SEED,
        "Permutation_Seed": PERMUTATION_SEED,
        "Permutation_Count": N_PERMUTATIONS,
        "Mean_Shuffled_Macro_F1": permutation_scores["Shuffled_Macro_F1"].mean(),
        "Std_Shuffled_Macro_F1": permutation_scores["Shuffled_Macro_F1"].std(),
        "Maximum_Shuffled_Macro_F1": permutation_scores["Shuffled_Macro_F1"].max(),
        "Permutation_P_Value": permutation_p_value,
    }
])
validation_report_sections.append(dataframe_to_report_section("Plant-label permutation diagnostic", permutation_summary, "Plant-level shuffled-label diagnostic using the locked five-feature Logistic Regression pipeline.", section_id="plant-label-permutation-diagnostic"))

## 19. Week x Label Tipburn-Score Summary

Development out-of-fold Tipburn-score summaries by Week and Label, plus plant-level deltas.

In [21]:
development_meta_cols = ["Dataset_ID", "Week", "DAT", "Label", "Sample_Group_ID", "Lettuce_Sample_ID", "Leaf_ID", "Split"]
X_week_oof = X_train_final.to_numpy(float)
y_week_oof = y_train_final.copy()
groups_week_oof = groups_train_final.copy()
meta_week_oof = meta_train_final.loc[:, development_meta_cols].copy()
balance_check = (
    meta_week_oof.groupby(["Week", "Label"])
    .agg(Leaf_Count=("Leaf_ID", "size"), Plant_Count=("Sample_Group_ID", "nunique"))
    .reset_index()
)
if not balance_check["Leaf_Count"].eq(72).all() or not balance_check["Plant_Count"].eq(24).all():
    raise AssertionError("Development Week x Label internal balance check failed.")
week_label_oof_prediction_frames = []
locked_week_lr_pipeline = build_locked_final_pipeline(RANDOM_STATE)
for seed in REPEAT_SEEDS:
    outer_cv = make_group_cv(y_week_oof, groups_week_oof, 5, seed)
    seed_seen = np.zeros(len(y_week_oof), dtype=bool)
    for fold_idx, (fit_idx, val_idx) in enumerate(outer_cv.split(X_week_oof, y_week_oof, groups_week_oof), start=1):
        if seed_seen[val_idx].any():
            raise RuntimeError(f"Seed {seed} assigned at least one row twice.")
        fold_model = clone(locked_week_lr_pipeline)
        fold_model.fit(X_week_oof[fit_idx], y_week_oof[fit_idx])
        fold_pred = fold_model.predict(X_week_oof[val_idx])
        fold_score = continuous_model_score(fold_model, X_week_oof[val_idx])
        fold_meta = meta_week_oof.iloc[val_idx].copy()
        fold_meta["Seed"] = seed
        fold_meta["Fold"] = fold_idx
        fold_meta["Model"] = FINAL_MODEL_NAME
        fold_meta["Locked_C"] = FINAL_MODEL_C
        fold_meta["y_true"] = y_week_oof[val_idx]
        fold_meta["y_pred"] = fold_pred
        fold_meta["Tipburn_Score"] = fold_score
        week_label_oof_prediction_frames.append(fold_meta)
        seed_seen[val_idx] = True
    if not seed_seen.all():
        raise RuntimeError(f"Seed {seed} left development rows without OOF predictions.")
week_label_oof_predictions_all_seeds = pd.concat(week_label_oof_prediction_frames, ignore_index=True)
if not week_label_oof_predictions_all_seeds["Split"].eq("train_val").all():
    raise AssertionError("OOF predictions include non-development rows.")
if set(week_label_oof_predictions_all_seeds["Sample_Group_ID"].astype(str)).intersection(holdout_groups):
    raise AssertionError("OOF predictions contain holdout plants.")
if week_label_oof_predictions_all_seeds.duplicated(["Seed", "Sample_Group_ID", "Leaf_ID"]).any():
    raise AssertionError("Duplicate OOF prediction rows found for Seed + Sample_Group_ID + Leaf_ID.")
if not week_label_oof_predictions_all_seeds["Tipburn_Score"].between(0, 1).all():
    raise AssertionError("OOF Tipburn scores are outside [0, 1].")
leaf_score_group_cols = ["Dataset_ID", "Sample_Group_ID", "Lettuce_Sample_ID", "Leaf_ID", "Week", "DAT", "Label", "y_true", "Split"]
development_oof_leaf_scores = (
    week_label_oof_predictions_all_seeds
    .groupby(leaf_score_group_cols, as_index=False)
    .agg(
        Mean_Tipburn_Score=("Tipburn_Score", "mean"),
        Median_Tipburn_Score=("Tipburn_Score", "median"),
        Score_SD_Across_Seeds=("Tipburn_Score", "std"),
        Minimum_Tipburn_Score=("Tipburn_Score", "min"),
        Maximum_Tipburn_Score=("Tipburn_Score", "max"),
        Seed_Count=("Seed", "nunique"),
    )
)
if not development_oof_leaf_scores["Seed_Count"].eq(len(REPEAT_SEEDS)).all():
    raise AssertionError("Unexpected Seed_Count in OOF leaf score table.")
validation_report_sections.append(dataframe_to_report_section(
    "Development OOF scores",
    development_oof_leaf_scores,
    "Mean OOF Tipburn score for each development leaf. Holdout and synthetic rows are excluded.",
    collapsible=True,
    section_id="development-oof-leaf-level-scores",
))

# Evaluate development OOF scores at threshold 0.5 without holdout or synthetic rows.
development_oof_y_true = development_oof_leaf_scores["y_true"].to_numpy(int)
development_oof_scores = development_oof_leaf_scores["Mean_Tipburn_Score"].to_numpy(float)
development_oof_y_pred = (development_oof_scores >= 0.5).astype(int)
development_oof_metrics = pd.DataFrame([
    binary_classification_metrics(
        development_oof_y_true, development_oof_y_pred, development_oof_scores,
        "development_oof_leaf_level", FINAL_MODEL_NAME,
    )
])
development_oof_metrics["Decision_Threshold"] = 0.5
print(
    "Development OOF evaluation "
    "(locked original model, threshold 0.5): "
    f"Macro-F1={development_oof_metrics.loc[0, 'F1_macro']:.4f}, "
    f"ROC AUC={development_oof_metrics.loc[0, 'ROC_AUC']:.4f}, "
    f"Balanced Accuracy="
    f"{development_oof_metrics.loc[0, 'Balanced_Accuracy']:.4f}"
)
validation_report_sections.append(dataframe_to_report_section(
    "Development OOF results",
    development_oof_metrics,
    "Performance of the final model on development OOF predictions at threshold 0.5. Holdout and synthetic rows are excluded.",
    section_id="development-oof-leaf-level-evaluation",
))
fig, ax = plt.subplots(figsize=(5.8, 4.8))
development_oof_cm = np.array([
    [development_oof_metrics.loc[0, "TN"], development_oof_metrics.loc[0, "FP"]],
    [development_oof_metrics.loc[0, "FN"], development_oof_metrics.loc[0, "TP"]],
])
sns.heatmap(development_oof_cm, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=["Healthy", "Tipburn"], yticklabels=["Healthy", "Tipburn"], ax=ax)
ax.set_title(f"Development OOF confusion matrix\nF1={development_oof_metrics.loc[0, 'F1_macro']:.3f}, AUC={development_oof_metrics.loc[0, 'ROC_AUC']:.3f}")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.tight_layout()
validation_report_sections.append(figure_to_base64_html(fig, "Development OOF confusion matrix", "Confusion matrix for development OOF predictions at threshold 0.5.", section_id="development-oof-confusion-matrix"))
week_label_oof_leaf_summary = (
    development_oof_leaf_scores
    .groupby(["Week", "DAT", "Label"], as_index=False)
    .agg(
        Leaf_Count=("Mean_Tipburn_Score", "size"),
        Plant_Count=("Sample_Group_ID", "nunique"),
        Mean=("Mean_Tipburn_Score", "mean"),
        Median=("Mean_Tipburn_Score", "median"),
        Std=("Mean_Tipburn_Score", "std"),
        Minimum=("Mean_Tipburn_Score", "min"),
        Maximum=("Mean_Tipburn_Score", "max"),
    )
)
development_oof_plant_scores = (
    development_oof_leaf_scores
    .groupby(["Dataset_ID", "Sample_Group_ID", "Week", "DAT", "Label"], as_index=False)
    .agg(Leaf_Count=("Leaf_ID", "nunique"), Plant_Mean_Tipburn_Score=("Mean_Tipburn_Score", "mean"))
)
plant_week_mean_scores = development_oof_plant_scores.groupby(["Label", "Week"])["Plant_Mean_Tipburn_Score"].mean().unstack("Week")
week_label_fixed_lr_oof_week_effect_delta = (
    plant_week_mean_scores.loc[:, ["Week 3", "Week 4"]]
    .reset_index()
    .rename(columns={"Week 3": "Week_3_Mean_Tipburn_Score", "Week 4": "Week_4_Mean_Tipburn_Score"})
)
week_label_fixed_lr_oof_week_effect_delta["Week4_Minus_Week3"] = (
    week_label_fixed_lr_oof_week_effect_delta["Week_4_Mean_Tipburn_Score"]
    - week_label_fixed_lr_oof_week_effect_delta["Week_3_Mean_Tipburn_Score"]
)
validation_report_sections.append(dataframe_to_report_section(
    "Development OOF Week x Label score summary",
    week_label_oof_leaf_summary,
    "Leaf-level summary of averaged out-of-fold Tipburn probabilities. Each leaf contributes once after averaging across repeated grouped-CV seeds.",
    section_id="development-oof-week-label-score-summary",
))
validation_report_sections.append(dataframe_to_report_section(
    "Week-effect delta table",
    week_label_fixed_lr_oof_week_effect_delta,
    "Plant-level Week 4 minus Week 3 mean Tipburn-score delta by true label. This is descriptive and was not used to change the model, threshold, features or hyperparameters.",
    section_id="week-effect-delta-table",
))

Development OOF evaluation (locked original model, threshold 0.5): Macro-F1=0.9166, ROC AUC=0.9594, Balanced Accuracy=0.9167


## 20. Development-Only Synthetic Training Experiment

A separate experiment compares original-only, synthetic-only, and original-plus-synthetic training inside the existing plant-grouped development CV. Validation folds remain genuine real leaves, and the fixed 20% holdout is not used.

In [22]:
leaf_train_final = leaf_df.loc[train_mask].reset_index(drop=True)
repeated_scan_residuals = make_repeated_scan_residuals(raw_df, wavelength_cols)
synthetic_experiment_results = run_development_synthetic_training_experiment(
    real_features=meta_train_final,
    real_leaf_spectra=leaf_train_final,
    repeated_scan_residuals=repeated_scan_residuals,
    wavelength_cols=wavelength_cols,
    wavelengths=wavelengths,
    forbidden_groups=holdout_groups,
    feature_columns=FINAL_FEATURE_COLS,
    repeat_seeds=REPEAT_SEEDS,
)
synthetic_experiment_summary = synthetic_experiment_results["summary"]
synthetic_experiment_seed_metrics = synthetic_experiment_results["seed_metrics"]
synthetic_experiment_fold_metrics = synthetic_experiment_results["fold_metrics"]
synthetic_experiment_paired_fold_difference_summary = synthetic_experiment_results["paired_fold_difference_summary"]
synthetic_experiment_fold_audit = synthetic_experiment_results["fold_audit"]
synthetic_experiment_predictions = synthetic_experiment_results["predictions"]
synthetic_experiment_provenance = synthetic_experiment_results["provenance"]
synthetic_experiment_rejections = synthetic_experiment_results["rejections"]
synthetic_experiment_generation_summary = synthetic_experiment_results["generation_summary"]
synthetic_experiment_spectra = synthetic_experiment_results["synthetic_spectra"]
synthetic_experiment_features = synthetic_experiment_results["synthetic_features"]
synthetic_experiment_convergence_audit = synthetic_experiment_results["convergence_audit"]
lr_convergence_audit_rows.extend(synthetic_experiment_convergence_audit.to_dict("records"))

if not synthetic_experiment_predictions["Validation_Row_Source"].eq("real").all():
    raise AssertionError("Synthetic experiment validation rows must all be real leaves.")
if synthetic_experiment_predictions["Synthetic_Rows_In_Validation"].sum() != 0:
    raise AssertionError("Synthetic rows appeared in validation predictions.")
if set(synthetic_experiment_predictions["Sample_Group_ID"].astype(str)).intersection(holdout_groups):
    raise AssertionError("Holdout plants appeared in synthetic experiment validation predictions.")
if not synthetic_experiment_fold_audit["Synthetic_Validation_Leaves"].eq(0).all():
    raise AssertionError("Fold audit shows synthetic validation leaves.")
if not synthetic_experiment_fold_audit["Holdout_Plants_Used"].eq(0).all():
    raise AssertionError("Fold audit shows holdout plants used in the synthetic experiment.")
if not synthetic_experiment_generation_summary.empty:
    if not synthetic_experiment_generation_summary["Requested_Count"].eq(synthetic_experiment_generation_summary["Accepted_Count"]).all():
        raise AssertionError("Synthetic generation did not meet every requested count.")

# Verify A, B, and fold-matched C use equal training sizes, with C split nearly 50/50.
DEV_C_SCENARIO_NAME = "C_Mixed_FoldMatched_50_50"
fold_matched_c_audit = synthetic_experiment_fold_audit.loc[
    synthetic_experiment_fold_audit["Scenario"].eq(DEV_C_SCENARIO_NAME)
].reset_index(drop=True)
if fold_matched_c_audit.empty:
    raise AssertionError(f"{DEV_C_SCENARIO_NAME} produced no fold-audit rows.")

_fold_matched_c_checks_zero = {
    "Validation_Synthetic_Rows": 0,
    "Holdout_Rows_Used_In_Training": 0,
    "Holdout_Rows_Used_In_Generation": 0,
}
for _col, _expected in _fold_matched_c_checks_zero.items():
    if not fold_matched_c_audit[_col].eq(_expected).all():
        raise AssertionError(
            f"{DEV_C_SCENARIO_NAME} fold audit column {_col} is not always {_expected}: "
            f"{sorted(fold_matched_c_audit[_col].unique().tolist())}"
        )

ab_c_fold_size_match_audit = (
    synthetic_experiment_fold_audit
    .pivot_table(index=["Seed", "Fold"], columns="Scenario", values="Total_Training_Rows", aggfunc="first")
    .rename(columns={
        "A_original_real_training_only": "A_Total_Training_Rows",
        "B_synthetic_only_equal_real_count": "B_Total_Training_Rows",
        DEV_C_SCENARIO_NAME: "C_Total_Training_Rows",
    })
    .loc[:, ["A_Total_Training_Rows", "B_Total_Training_Rows", "C_Total_Training_Rows"]]
    .reset_index()
)
if ab_c_fold_size_match_audit[["A_Total_Training_Rows", "B_Total_Training_Rows", "C_Total_Training_Rows"]].isna().any().any():
    raise AssertionError("A/B/C are not all present for every seed/fold in the fold-size-match audit.")
if not (
    ab_c_fold_size_match_audit["A_Total_Training_Rows"].eq(ab_c_fold_size_match_audit["B_Total_Training_Rows"])
    & ab_c_fold_size_match_audit["A_Total_Training_Rows"].eq(ab_c_fold_size_match_audit["C_Total_Training_Rows"])
).all():
    raise AssertionError(
        "A_Total_Training_Rows == B_Total_Training_Rows == C_Total_Training_Rows failed for some seed/fold: "
        f"{ab_c_fold_size_match_audit.loc[~(ab_c_fold_size_match_audit['A_Total_Training_Rows'].eq(ab_c_fold_size_match_audit['B_Total_Training_Rows']) & ab_c_fold_size_match_audit['A_Total_Training_Rows'].eq(ab_c_fold_size_match_audit['C_Total_Training_Rows']))].to_dict('records')}"
    )

_c_real_vs_synth_diff = (fold_matched_c_audit["Real_Total"] - fold_matched_c_audit["Synthetic_Total"]).abs()
if not (_c_real_vs_synth_diff <= 1).all():
    raise AssertionError(
        f"{DEV_C_SCENARIO_NAME}: Real_Total and Synthetic_Total differ by more than one row in some fold: "
        f"{sorted(_c_real_vs_synth_diff.unique().tolist())}"
    )
if not (fold_matched_c_audit["Real_Total"] + fold_matched_c_audit["Synthetic_Total"] == fold_matched_c_audit["Total_Training_Rows"]).all():
    raise AssertionError(f"{DEV_C_SCENARIO_NAME}: Real_Total + Synthetic_Total does not equal Total_Training_Rows in some fold.")

print(
    f"{DEV_C_SCENARIO_NAME}: {len(fold_matched_c_audit)} seed/fold rows. "
    "A/B/C total training rows are equal per seed/fold (all == len(fit_idx)), and C's "
    "Real_Total/Synthetic_Total differ by at most one row per fold."
)

synthetic_policy_table = pd.DataFrame([
    {"Policy": "Synthetic generation scope", "Value": "Generated separately inside each plant-grouped development training fold"},
    {"Policy": "Validation rows", "Value": "Genuine real development leaves only"},
    {"Policy": "Holdout usage", "Value": "Fixed 20% holdout is not used for generation, selection, tuning, or validation in this experiment"},
    {"Policy": "Source spectra", "Value": "Raw leaf-level spectra after repeated-scan and area-to-leaf averaging, before smoothing/normalization/features"},
    {"Policy": "Allocation", "Value": "Deterministic Label x Week allocation; label totals differ by at most one and week totals follow fold-training week proportions within label"},
    {"Policy": "Pairing rule", "Value": "Same Label and Week, preferably same DAT, different Sample_Group_ID values"},
    {"Policy": "Residual fallback", "Value": "Label+Week+DAT, then Label+Week, then Week, then any residual from current fold-training plants"},
    {"Policy": "Alpha", "Value": f"Uniform({SYNTHETIC_ALPHA_LOW}, {SYNTHETIC_ALPHA_HIGH})"},
    {"Policy": "Noise", "Value": f"{SYNTHETIC_RESIDUAL_SCALE} x real repeated-scan residual from fold-training rows"},
    {"Policy": "Rejection loop", "Value": f"Maximum attempts = requested count x {SYNTHETIC_MAX_ATTEMPTS_MULTIPLIER}; requested counts cannot be silently reduced"},
    {"Policy": "Feature extraction", "Value": "Synthetic features are produced only by preprocess_leaf_spectra()"},
    {"Policy": "Model", "Value": f"Locked {FINAL_MODEL_NAME} with clf__C={FINAL_MODEL_C} and threshold 0.5"},
])

report_metadata.update({
    "Synthetic experiment": "Development CV only; holdout untouched and synthetic-free",
    "Synthetic alpha range": f"{SYNTHETIC_ALPHA_LOW}-{SYNTHETIC_ALPHA_HIGH}",
    "Synthetic residual scale": SYNTHETIC_RESIDUAL_SCALE,
    "Synthetic combined ratio": SYNTHETIC_COMBINED_RATIO,
})
validation_report_sections.append(dataframe_to_report_section(
    "Synthetic experiment policy",
    synthetic_policy_table,
    "Rules enforced for the added synthetic-data experiment. These rules do not modify the original-only pipeline or the fixed final holdout evaluation.",
    section_id="synthetic-experiment-policy",
))

# Use Seed 42, Fold 1, and scenario C as the representative synthetic-data example.
REPRESENTATIVE_SYNTHETIC_SEED = 42
REPRESENTATIVE_SYNTHETIC_FOLD = 1
REPRESENTATIVE_SYNTHETIC_SCENARIO = "C_Mixed_FoldMatched_50_50"
representative_mask = (
    synthetic_experiment_spectra["Seed"].eq(REPRESENTATIVE_SYNTHETIC_SEED)
    & synthetic_experiment_spectra["Fold"].eq(REPRESENTATIVE_SYNTHETIC_FOLD)
    & synthetic_experiment_spectra["Scenario"].eq(REPRESENTATIVE_SYNTHETIC_SCENARIO)
)
representative_synthetic_spectra = synthetic_experiment_spectra.loc[representative_mask].reset_index(drop=True)
representative_synthetic_features = synthetic_experiment_features.loc[
    synthetic_experiment_features["Seed"].eq(REPRESENTATIVE_SYNTHETIC_SEED)
    & synthetic_experiment_features["Fold"].eq(REPRESENTATIVE_SYNTHETIC_FOLD)
    & synthetic_experiment_features["Scenario"].eq(REPRESENTATIVE_SYNTHETIC_SCENARIO)
].reset_index(drop=True)
representative_provenance = synthetic_experiment_provenance.loc[
    synthetic_experiment_provenance["Seed"].eq(REPRESENTATIVE_SYNTHETIC_SEED)
    & synthetic_experiment_provenance["Fold"].eq(REPRESENTATIVE_SYNTHETIC_FOLD)
    & synthetic_experiment_provenance["Scenario"].eq(REPRESENTATIVE_SYNTHETIC_SCENARIO)
].reset_index(drop=True)
representative_generation = synthetic_experiment_generation_summary.loc[
    synthetic_experiment_generation_summary["Seed"].eq(REPRESENTATIVE_SYNTHETIC_SEED)
    & synthetic_experiment_generation_summary["Fold"].eq(REPRESENTATIVE_SYNTHETIC_FOLD)
    & synthetic_experiment_generation_summary["Scenario"].eq(REPRESENTATIVE_SYNTHETIC_SCENARIO)
].reset_index(drop=True)
# Summarise the representative real fold, acceptance limits, and duplicate checks.
rep_cv = make_group_cv(y_train_final, groups_train_final, 5, REPRESENTATIVE_SYNTHETIC_SEED)
rep_fit_idx, _ = next(
    (fit_idx, val_idx)
    for fold_number, (fit_idx, val_idx) in enumerate(rep_cv.split(X_train_final.to_numpy(float), y_train_final, groups_train_final), start=1)
    if fold_number == REPRESENTATIVE_SYNTHETIC_FOLD
)
representative_real_spectra = leaf_train_final.iloc[rep_fit_idx].reset_index(drop=True)
representative_real_features = meta_train_final.iloc[rep_fit_idx].reset_index(drop=True)
rep_real_values = representative_real_spectra[wavelength_cols].to_numpy(float)
rep_synth_values = representative_synthetic_spectra[wavelength_cols].to_numpy(float)
rep_limits = derive_synthetic_acceptance_limits(representative_real_spectra, wavelength_cols)
# Pair each scenario's gate outcome with limits derived from its real training fold.
_dup_cols = ["Exact_Duplicate_Source_A", "Exact_Duplicate_Source_B", "Exact_Duplicate_Real_Training"]
_perfold_scenarios = [
    ("B_synthetic_only_equal_real_count", "B (synthetic-only)", "synthetic-validation-per-fold-gate-b"),
    ("C_Mixed_FoldMatched_50_50", "C (mixed 50/50)", "synthetic-validation-per-fold-gate-c"),
]
_perfold_rows = {_scen: [] for _scen, _lbl, _sid in _perfold_scenarios}
for _seed in REPEAT_SEEDS:
    _cv = make_group_cv(y_train_final, groups_train_final, 5, _seed)
    for _fnum, (_fit, _val) in enumerate(_cv.split(X_train_final.to_numpy(float), y_train_final, groups_train_final), start=1):
        _lim = derive_synthetic_acceptance_limits(leaf_train_final.iloc[_fit].reset_index(drop=True), wavelength_cols)
        _limvals = {
            "Band_Low": float(np.min(_lim["band_low"])), "Band_High": float(np.max(_lim["band_high"])),
            "Roughness_Low": float(_lim["roughness_low"]), "Roughness_High": float(_lim["roughness_high"]),
            "Suspicious_Threshold": float(_lim["nearest_real_suspicious_threshold"]),
        }
        for _scen, _lbl, _sid in _perfold_scenarios:
            _prov = synthetic_experiment_provenance[
                synthetic_experiment_provenance["Seed"].eq(_seed)
                & synthetic_experiment_provenance["Fold"].eq(_fnum)
                & synthetic_experiment_provenance["Scenario"].eq(_scen)
            ]
            if not synthetic_experiment_rejections.empty and "Seed" in synthetic_experiment_rejections.columns:
                _rej_n = int((synthetic_experiment_rejections["Seed"].eq(_seed)
                              & synthetic_experiment_rejections["Fold"].eq(_fnum)
                              & synthetic_experiment_rejections["Scenario"].eq(_scen)).sum())
            else:
                _rej_n = 0
            _dups = int(_prov[_dup_cols].to_numpy(bool).any(axis=1).sum()) if (not _prov.empty and all(_c in _prov.columns for _c in _dup_cols)) else 0
            _nn = pd.to_numeric(_prov.get("Nearest_Real_Distance"), errors="coerce") if not _prov.empty else pd.Series(dtype=float)
            _rgh = pd.to_numeric(_prov.get("Synthetic_Roughness"), errors="coerce") if not _prov.empty else pd.Series(dtype=float)
            _spec = synthetic_experiment_spectra[
                synthetic_experiment_spectra["Seed"].eq(_seed)
                & synthetic_experiment_spectra["Fold"].eq(_fnum)
                & synthetic_experiment_spectra["Scenario"].eq(_scen)
            ]
            _specvals = _spec[wavelength_cols].to_numpy(float) if not _spec.empty else np.empty((0, len(wavelength_cols)))
            _perfold_rows[_scen].append({
                "Seed": _seed, "Fold": _fnum, **_limvals,
                "Band_Min_Accepted": float(_specvals.min()) if _specvals.size else float("nan"),
                "Band_Max_Accepted": float(_specvals.max()) if _specvals.size else float("nan"),
                "Roughness_Min_Accepted": float(_rgh.min()) if _rgh.notna().any() else float("nan"),
                "Roughness_Max_Accepted": float(_rgh.max()) if _rgh.notna().any() else float("nan"),
                "Min_Nearest_Real_Distance": float(_nn.min()) if _nn.notna().any() else float("nan"),
                "Mean_Nearest_Real_Distance": float(_nn.mean()) if _nn.notna().any() else float("nan"),
                "Accepted_Synthetic": int(len(_prov)),
                "Rejected_Attempts": _rej_n,
                "Exact_Duplicate_Rows": _dups,
            })
for _scen, _lbl, _sid in _perfold_scenarios:
    _tbl = pd.DataFrame(_perfold_rows[_scen])
    validation_report_appendix_sections.append(dataframe_to_report_section(
        f"Per-fold acceptance check - Scenario {_lbl}",
        _tbl,
        f"Acceptance limits, synthetic statistics and gate result for each development fold in Scenario {_lbl}.",
        collapsible=True,
        section_id=_sid,
    ))
exact_duplicate_summary = pd.DataFrame([{
    "Exact_Duplicate_Source_A": int(representative_provenance["Exact_Duplicate_Source_A"].sum()) if not representative_provenance.empty else 0,
    "Exact_Duplicate_Source_B": int(representative_provenance["Exact_Duplicate_Source_B"].sum()) if not representative_provenance.empty else 0,
    "Exact_Duplicate_Real_Training": int(representative_provenance["Exact_Duplicate_Real_Training"].sum()) if not representative_provenance.empty else 0,
}])

suspicious_close = representative_provenance.loc[
    representative_provenance["Nearest_Real_Distance"].le(rep_limits["nearest_real_suspicious_threshold"])
].copy()
validation_report_appendix_sections.append(dataframe_to_report_section(
    "Suspicious synthetic rows",
    suspicious_close.head(50),
    "Synthetic rows at or below the distance threshold. An empty table means none were too close.",
    collapsible=len(suspicious_close) > 12,
    section_id="synthetic-validation-suspiciously-close-synthetic-rows",
))
feature_range_rows = []
for feature in FINAL_FEATURE_COLS:
    real_min = float(representative_real_features[feature].min())
    real_max = float(representative_real_features[feature].max())
    synth_values = representative_synthetic_features[feature].astype(float)
    feature_range_rows.append({
        "Feature": feature,
        "Real_Min": real_min,
        "Real_Max": real_max,
        "Synthetic_Min": float(synth_values.min()),
        "Synthetic_Max": float(synth_values.max()),
        "Synthetic_Below_Real_Min": int(synth_values.lt(real_min).sum()),
        "Synthetic_Above_Real_Max": int(synth_values.gt(real_max).sum()),
    })
feature_range_checks = pd.DataFrame(feature_range_rows)
validation_report_appendix_sections.append(dataframe_to_report_section(
    "Feature-range checks",
    feature_range_checks,
    "Synthetic feature ranges compared with real training ranges in the example fold.",
    section_id="synthetic-validation-feature-range-checks",
))

fig, ax = plt.subplots(figsize=(9, 5.2))
for row in rep_real_values[:80]:
    ax.plot(wavelengths, row, color="#4c78a8", alpha=0.08, linewidth=0.8)
for row in rep_synth_values[:80]:
    ax.plot(wavelengths, row, color="#f58518", alpha=0.10, linewidth=0.8)
ax.set_title("Representative fold raw spectral overlays")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Raw reflectance")
ax.plot([], [], color="#4c78a8", label="Real training leaves")
ax.plot([], [], color="#f58518", label="Synthetic accepted leaves")
ax.legend()
sns.despine(ax=ax)
plt.tight_layout()
validation_report_appendix_sections.append(figure_to_base64_html(
    fig,
    "Real and synthetic spectra",
    "Real and accepted synthetic spectra from the fixed example fold: Seed 42, Fold 1.",
    section_id="synthetic-validation-real-vs-synthetic-overlays",
))

real_mean = rep_real_values.mean(axis=0)
real_std = rep_real_values.std(axis=0)
synth_mean = rep_synth_values.mean(axis=0)
synth_std = rep_synth_values.std(axis=0)
fig, ax = plt.subplots(figsize=(9, 5.2))
ax.plot(wavelengths, real_mean, color="#4c78a8", linewidth=2.0, label="Real mean")
ax.fill_between(wavelengths, real_mean - real_std, real_mean + real_std, color="#4c78a8", alpha=0.18, label="Real +/- 1 SD")
ax.plot(wavelengths, synth_mean, color="#f58518", linewidth=2.0, label="Synthetic mean")
ax.fill_between(wavelengths, synth_mean - synth_std, synth_mean + synth_std, color="#f58518", alpha=0.18, label="Synthetic +/- 1 SD")
ax.set_title("Representative fold mean and SD spectra")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Raw reflectance")
ax.legend()
sns.despine(ax=ax)
plt.tight_layout()
validation_report_appendix_sections.append(figure_to_base64_html(
    fig,
    "Mean real and synthetic spectra",
    "Mean ± 1 SD raw spectra for real and synthetic leaves in the representative development fold.",
    section_id="synthetic-validation-mean-sd-spectra",
))

feature_plot_df = pd.concat([
    representative_real_features.assign(Source_Type="Real"),
    representative_synthetic_features.assign(Source_Type="Synthetic"),
], ignore_index=True)
feature_long = feature_plot_df.melt(id_vars="Source_Type", value_vars=FINAL_FEATURE_COLS, var_name="Feature", value_name="Value")
# Plot REIP by grid composition and the remaining features as real-versus-synthetic boxplots.
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
_reip_bins = feature_plot_df["REIP_nm"].round().astype(int)
for ax, feature in zip(axes.flat, FINAL_FEATURE_COLS):
    if feature == "REIP_nm":
        reip_comp = (
            pd.DataFrame({"REIP_bin": _reip_bins, "Source_Type": feature_plot_df["Source_Type"]})
            .groupby(["REIP_bin", "Source_Type"]).size()
            .unstack("Source_Type", fill_value=0)
            .reindex(columns=["Real", "Synthetic"], fill_value=0)
            .sort_index()
        )
        reip_tot = reip_comp.sum(axis=1)
        reip_pct = reip_comp.div(reip_tot, axis=0) * 100.0
        real_pct = reip_pct["Real"].to_numpy()
        synth_pct = reip_pct["Synthetic"].to_numpy()
        bar_x = np.arange(len(reip_comp))
        ax.bar(bar_x, real_pct, width=0.72, color="#2f6fbb", label="Real")
        ax.bar(bar_x, synth_pct, width=0.72, bottom=real_pct, color="#c84c3a", label="Synthetic")
        for xi in bar_x:
            if real_pct[xi] >= 4:
                ax.text(xi, real_pct[xi] / 2, f"{real_pct[xi]:.0f}% R", ha="center", va="center", color="white", fontsize=8)
            if synth_pct[xi] >= 4:
                ax.text(xi, real_pct[xi] + synth_pct[xi] / 2, f"{synth_pct[xi]:.0f}% S", ha="center", va="center", color="white", fontsize=8)
        ax.set_xticks(bar_x)
        ax.set_xticklabels([f"{int(v)}\nn = {int(reip_tot.iloc[i])}" for i, v in enumerate(reip_comp.index)])
        ax.set_yticks([0, 20, 40, 60, 80, 100])
        ax.set_ylim(-5, 109)
        ax.set_ylabel("% of representative-fold leaves")
        ax.set_title(feature)
        ax.set_xlabel("")
        continue
    sns.boxplot(data=feature_plot_df, x="Source_Type", y=feature, hue="Source_Type", legend=False, ax=ax, palette={"Real": "#2f6fbb", "Synthetic": "#c84c3a"})
    sns.stripplot(data=feature_plot_df, x="Source_Type", y=feature, ax=ax, color="#111827", alpha=0.28, size=2.3)
    ax.set_title(feature)
    ax.set_xlabel("")
for ax in axes.flat[len(FINAL_FEATURE_COLS):]:
    ax.set_visible(False)
plt.tight_layout()
validation_report_appendix_sections.append(figure_to_base64_html(
    fig,
    "Final-feature distributions",
    "Real and synthetic distributions for the five final features in the example fold.",
    section_id="synthetic-validation-final-feature-distributions",
))
feature_stats = (
    feature_long
    .groupby(["Source_Type", "Feature"], as_index=False)
    .agg(Count=("Value", "size"), Mean=("Value", "mean"), Std=("Value", "std"), Median=("Value", "median"), Minimum=("Value", "min"), Maximum=("Value", "max"))
)
validation_report_appendix_sections.append(dataframe_to_report_section(
    "Real and synthetic feature summary",
    feature_stats,
    "Summary statistics for real and synthetic final features in the example fold.",
    section_id="synthetic-validation-descriptive-feature-statistics",
))

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
pca_values = feature_plot_df[FINAL_FEATURE_COLS].to_numpy(float)
pca_scaled = StandardScaler().fit_transform(pca_values)
pca_coords = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(pca_scaled)
pca_df = pd.DataFrame({"PC1": pca_coords[:, 0], "PC2": pca_coords[:, 1], "Source_Type": feature_plot_df["Source_Type"].to_numpy()})
fig, ax = plt.subplots(figsize=(7.5, 5.8))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="Source_Type", alpha=0.75, ax=ax)
ax.set_title("Representative fold PCA of locked features")
sns.despine(ax=ax)
plt.tight_layout()
validation_report_appendix_sections.append(figure_to_base64_html(
    fig,
    "Real and synthetic PCA",
    "PCA comparison of real and synthetic rows using the five final features.",
    section_id="synthetic-validation-pca-comparison",
))

print("Synthetic training experiment completed without holdout or synthetic-validation leakage.")

C_Mixed_FoldMatched_50_50: 25 seed/fold rows. A/B/C total training rows are equal per seed/fold (all == len(fit_idx)), and C's Real_Total/Synthetic_Total differ by at most one row per fold.
Synthetic training experiment completed without holdout or synthetic-validation leakage.


In [23]:
# Pool acceptance checks across all 25 development folds and both synthetic scenarios.
_gate_prov = synthetic_experiment_provenance.copy()
_gate_rej = synthetic_experiment_rejections.copy()
_dup_cols = ["Exact_Duplicate_Source_A", "Exact_Duplicate_Source_B", "Exact_Duplicate_Real_Training"]

pooled_gate_rows = []
if not _gate_prov.empty:
    for scenario_name, scenario_prov in _gate_prov.groupby("Scenario", sort=True):
        folds_covered = scenario_prov[["Seed", "Fold"]].drop_duplicates().shape[0]
        accepted_rows = int(len(scenario_prov))
        has_dup_cols = all(col in scenario_prov.columns for col in _dup_cols)
        dup_rows = int(scenario_prov[_dup_cols].to_numpy(bool).any(axis=1).sum()) if has_dup_cols else np.nan
        nearest = pd.to_numeric(scenario_prov.get("Nearest_Real_Distance"), errors="coerce")
        scenario_rej = _gate_rej.loc[_gate_rej["Scenario"].eq(scenario_name)] if not _gate_rej.empty else _gate_rej
        rejected_attempts = int(len(scenario_rej))
        min_nearest = float(nearest.min()) if nearest.notna().any() else np.nan
        gate_pass = ((dup_rows == 0) if not pd.isna(dup_rows) else False) and (
            np.isnan(min_nearest) or min_nearest > 0.0
        )
        pooled_gate_rows.append({
            "Scenario": scenario_name,
            "Folds_Covered": folds_covered,
            "Accepted_Synthetic_Rows": accepted_rows,
            "Exact_Duplicate_Rows": dup_rows,
            "Min_Nearest_Real_Distance": min_nearest,
            "Mean_Nearest_Real_Distance": float(nearest.mean()) if nearest.notna().any() else np.nan,
            "Rejected_Attempts": rejected_attempts,
            "Gate_Status": "PASS" if gate_pass else "REVIEW",
        })

pooled_gate_summary = pd.DataFrame(pooled_gate_rows)
if pooled_gate_summary.empty:
    pooled_gate_summary = pd.DataFrame([{"Scenario": "None", "Gate_Status": "No synthetic rows generated"}])

validation_report_appendix_sections.append(dataframe_to_report_section(
    "All-fold acceptance summary",
    pooled_gate_summary,
    "Acceptance results across all 25 development folds for Scenarios B and C. Zero duplicates and a positive minimum distance confirm that accepted synthetic leaves are unique.",
    section_id="synthetic-validation-pooled-gate-summary",
))

# Summarise rejection reasons across all folds and synthetic scenarios.
CANONICAL_REJECTION_REASONS = [
    "incorrect_wavelength_length",
    "non_finite_values",
    "exact_duplicate_source_a",
    "exact_duplicate_source_b",
    "exact_duplicate_real_training_spectrum",
    "outside_training_derived_band_limits",
    "outside_training_derived_roughness_limits",
]
_rej_scenarios = sorted(_gate_prov["Scenario"].unique().tolist()) if not _gate_prov.empty else ["All synthetic-bearing scenarios"]
if not _gate_rej.empty and "Rejection_Reasons" in _gate_rej.columns:
    _observed_reasons = (
        _gate_rej.assign(Rejection_Reason=_gate_rej["Rejection_Reasons"].str.split("; "))
        .explode("Rejection_Reason")
        .groupby(["Scenario", "Rejection_Reason"], dropna=False)
        .size()
        .reset_index(name="Rejected_Attempts")
    )
else:
    _observed_reasons = pd.DataFrame(columns=["Scenario", "Rejection_Reason", "Rejected_Attempts"])
# Include zero-count reasons so every scenario uses the same seven-category audit.
_rej_grid = pd.MultiIndex.from_product(
    [_rej_scenarios, CANONICAL_REJECTION_REASONS], names=["Scenario", "Rejection_Reason"]
).to_frame(index=False)
pooled_rejection_reasons = _rej_grid.merge(_observed_reasons, on=["Scenario", "Rejection_Reason"], how="left")
pooled_rejection_reasons["Rejected_Attempts"] = pooled_rejection_reasons["Rejected_Attempts"].fillna(0).astype(int)
pooled_rejection_reasons = pooled_rejection_reasons.sort_values(
    ["Scenario", "Rejected_Attempts", "Rejection_Reason"], ascending=[True, False, True]
).reset_index(drop=True)

# Report the pooled envelope of fold-specific acceptance limits.
_pool_band_low, _pool_band_high, _pool_rough_low, _pool_rough_high = [], [], [], []
for _rr_seed in REPEAT_SEEDS:
    _rr_cv = make_group_cv(y_train_final, groups_train_final, 5, _rr_seed)
    for _rr_fit, _rr_val in _rr_cv.split(X_train_final.to_numpy(float), y_train_final, groups_train_final):
        _rr_lim = derive_synthetic_acceptance_limits(leaf_train_final.iloc[_rr_fit].reset_index(drop=True), wavelength_cols)
        _pool_band_low.append(float(np.min(_rr_lim["band_low"])))
        _pool_band_high.append(float(np.max(_rr_lim["band_high"])))
        _pool_rough_low.append(float(_rr_lim["roughness_low"]))
        _pool_rough_high.append(float(_rr_lim["roughness_high"]))
REJECTION_REASON_LIMIT = {
    "incorrect_wavelength_length": ("structural: length == wavelength_count", np.nan, np.nan),
    "non_finite_values": ("structural: all values finite", np.nan, np.nan),
    "exact_duplicate_source_a": ("structural: != parent A spectrum", np.nan, np.nan),
    "exact_duplicate_source_b": ("structural: != parent B spectrum", np.nan, np.nan),
    "exact_duplicate_real_training_spectrum": ("structural: != any real training spectrum", np.nan, np.nan),
    "outside_training_derived_band_limits": ("Band_Low..Band_High (per-fold min/max +/- max(1.5*IQR, eps))", min(_pool_band_low), max(_pool_band_high)),
    "outside_training_derived_roughness_limits": ("Roughness_Low..Roughness_High (per-fold min/max +/- 3*IQR)", min(_pool_rough_low), max(_pool_rough_high)),
}
pooled_rejection_reasons["Enforced_Limit"] = pooled_rejection_reasons["Rejection_Reason"].map(lambda r: REJECTION_REASON_LIMIT.get(r, ("", np.nan, np.nan))[0])
pooled_rejection_reasons["Pooled_Limit_Low"] = pooled_rejection_reasons["Rejection_Reason"].map(lambda r: REJECTION_REASON_LIMIT.get(r, ("", np.nan, np.nan))[1])
pooled_rejection_reasons["Pooled_Limit_High"] = pooled_rejection_reasons["Rejection_Reason"].map(lambda r: REJECTION_REASON_LIMIT.get(r, ("", np.nan, np.nan))[2])

validation_report_appendix_sections.append(dataframe_to_report_section(
    "Synthetic rejection reasons",
    pooled_rejection_reasons,
    "Rejected synthetic draws by reason across all folds for Scenarios B and C. Rejected draws were not used for training.",
    section_id="synthetic-validation-pooled-rejection-reasons",
))

## 21. Development OOF Leaf-Level Misclassification Summary

Compact one-row-per-leaf summary from Experiment 1 development OOF CV only. This uses the existing A/B/C OOF misclassification output and does not use or mix in final holdout rows.

In [24]:
SCENARIO_TO_LEAF_SUMMARY_SUFFIX = {
    "A_original_real_training_only": "A_Original",
    "B_synthetic_only_equal_real_count": "B_Synthetic",
    # Use "C_Mixed" as the short table suffix for C_Mixed_FoldMatched_50_50.
    "C_Mixed_FoldMatched_50_50": "C_Mixed",
}

required_oof_cols = {
    "Scenario", "Seed", "Fold", "Sample_Group_ID", "Leaf_ID", "Label", "Week", "DAT",
    "y_true", "y_pred", "Tipburn_Score", "Validation_Row_Source", "Synthetic_Rows_In_Validation",
}
missing_oof_cols = sorted(required_oof_cols - set(synthetic_experiment_predictions.columns))
if missing_oof_cols:
    raise AssertionError(f"Synthetic OOF prediction table is missing columns: {missing_oof_cols}")

oof_leaf_source = synthetic_experiment_predictions.loc[
    synthetic_experiment_predictions["Scenario"].isin(SCENARIO_TO_LEAF_SUMMARY_SUFFIX)
].copy()
oof_leaf_source["Scenario_Short"] = oof_leaf_source["Scenario"].map(SCENARIO_TO_LEAF_SUMMARY_SUFFIX)

if not oof_leaf_source["Validation_Row_Source"].eq("real").all():
    raise AssertionError("Leaf-level OOF summary source contains non-real validation rows.")
if int(oof_leaf_source["Synthetic_Rows_In_Validation"].sum()) != 0:
    raise AssertionError("Leaf-level OOF summary source contains synthetic validation rows.")
if not oof_leaf_source["Split"].eq("train_val").all():
    raise AssertionError("Leaf-level OOF summary source contains non-development rows.")
if set(oof_leaf_source["Sample_Group_ID"].astype(str)).intersection(holdout_groups):
    raise AssertionError("Leaf-level OOF summary source contains final-holdout plants.")

oof_leaf_source["Error_Type"] = np.select(
    [
        oof_leaf_source["y_true"].eq(0) & oof_leaf_source["y_pred"].eq(1),
        oof_leaf_source["y_true"].eq(1) & oof_leaf_source["y_pred"].eq(0),
    ],
    ["FP", "FN"],
    default="Correct",
)
oof_leaf_source["Is_FP"] = oof_leaf_source["Error_Type"].eq("FP").astype(int)
oof_leaf_source["Is_FN"] = oof_leaf_source["Error_Type"].eq("FN").astype(int)
oof_leaf_source["Is_Wrong"] = oof_leaf_source["Error_Type"].isin(["FP", "FN"]).astype(int)

leaf_id_cols = ["Sample_Group_ID", "Leaf_ID", "Label", "Week", "DAT"]
leaf_scenario_summary = (
    oof_leaf_source
    .groupby([*leaf_id_cols, "Scenario_Short"], as_index=False)
    .agg(
        N_FP=("Is_FP", "sum"),
        N_FN=("Is_FN", "sum"),
        Total_Wrong=("Is_Wrong", "sum"),
        Mean_Tipburn_Score=("Tipburn_Score", "mean"),
    )
)

scenario_suffix_order = ["A_Original", "B_Synthetic", "C_Mixed"]
base_leaf_index = (
    oof_leaf_source[leaf_id_cols]
    .drop_duplicates()
    .sort_values(["Sample_Group_ID", "Leaf_ID", "Label"])
    .reset_index(drop=True)
)

def pivot_leaf_metric(metric: str, prefix: str) -> pd.DataFrame:
    wide = (
        leaf_scenario_summary
        .pivot_table(
            index=leaf_id_cols,
            columns="Scenario_Short",
            values=metric,
            aggfunc="first",
        )
        .reindex(columns=scenario_suffix_order)
        .reset_index()
    )
    wide = wide.rename(columns={suffix: f"{prefix}_{suffix}" for suffix in scenario_suffix_order})
    return wide

oof_leaf_level_misclassification_summary = base_leaf_index.copy()
for metric, prefix in [
    ("N_FP", "N_FP"),
    ("N_FN", "N_FN"),
    ("Total_Wrong", "Total_Wrong"),
    ("Mean_Tipburn_Score", "Mean_Tipburn_Score"),
]:
    oof_leaf_level_misclassification_summary = oof_leaf_level_misclassification_summary.merge(
        pivot_leaf_metric(metric, prefix),
        on=leaf_id_cols,
        how="left",
    )

count_cols = [
    "N_FP_A_Original", "N_FP_B_Synthetic", "N_FP_C_Mixed",
    "N_FN_A_Original", "N_FN_B_Synthetic", "N_FN_C_Mixed",
    "Total_Wrong_A_Original", "Total_Wrong_B_Synthetic", "Total_Wrong_C_Mixed",
]
for col in count_cols:
    oof_leaf_level_misclassification_summary[col] = (
        oof_leaf_level_misclassification_summary[col].fillna(0).astype(int)
    )
score_cols = [
    "Mean_Tipburn_Score_A_Original",
    "Mean_Tipburn_Score_B_Synthetic",
    "Mean_Tipburn_Score_C_Mixed",
]
for col in score_cols:
    oof_leaf_level_misclassification_summary[col] = oof_leaf_level_misclassification_summary[col].astype(float)

oof_leaf_level_misclassification_summary["Total_Wrong_All"] = oof_leaf_level_misclassification_summary[
    ["Total_Wrong_A_Original", "Total_Wrong_B_Synthetic", "Total_Wrong_C_Mixed"]
].sum(axis=1).astype(int)

def interpret_oof_leaf_errors(row: pd.Series) -> str:
    wrong_by_scenario = {
        "A": int(row["Total_Wrong_A_Original"]),
        "B": int(row["Total_Wrong_B_Synthetic"]),
        "C_Mixed": int(row["Total_Wrong_C_Mixed"]),
    }
    wrong_scenarios = [key for key, value in wrong_by_scenario.items() if value > 0]
    if not wrong_scenarios:
        return "No OOF errors across A/B/C_Mixed."
    if len(wrong_scenarios) == len(wrong_by_scenario):
        return "Wrong in all three OOF scenarios (A, B, C_Mixed)."
    if wrong_scenarios == ["A"]:
        return "Wrong only with original training."
    if wrong_scenarios == ["B"]:
        return "Wrong only with synthetic training."
    if wrong_scenarios == ["C_Mixed"]:
        return "Wrong only with fold-size-matched ~50/50 mixed training."
    return "Wrong in scenarios " + "/".join(wrong_scenarios) + "."

oof_leaf_level_misclassification_summary["Short_Interpretation"] = oof_leaf_level_misclassification_summary.apply(
    interpret_oof_leaf_errors,
    axis=1,
)

oof_leaf_level_misclassification_summary = (
    oof_leaf_level_misclassification_summary
    .rename(columns={"Sample_Group_ID": "Lettuce_ID"})
    .loc[:, [
        "Lettuce_ID", "Leaf_ID", "Label", "Week", "DAT",
        "N_FP_A_Original", "N_FP_B_Synthetic", "N_FP_C_Mixed",
        "N_FN_A_Original", "N_FN_B_Synthetic", "N_FN_C_Mixed",
        "Total_Wrong_A_Original", "Total_Wrong_B_Synthetic", "Total_Wrong_C_Mixed",
        "Total_Wrong_All",
        "Mean_Tipburn_Score_A_Original", "Mean_Tipburn_Score_B_Synthetic", "Mean_Tipburn_Score_C_Mixed",
        "Short_Interpretation",
    ]]
)

if oof_leaf_level_misclassification_summary.duplicated(["Lettuce_ID", "Leaf_ID", "Label"]).any():
    raise AssertionError("Leaf-level OOF summary is not unique by Lettuce_ID + Leaf_ID + Label.")

oof_leaf_level_misclassification_summary = (
    oof_leaf_level_misclassification_summary
    .loc[oof_leaf_level_misclassification_summary["Total_Wrong_All"].gt(0)]
    .reset_index(drop=True)
)

print(
    "OOF leaf-level misclassification summary rows: "
    f"{len(oof_leaf_level_misclassification_summary)}"
)

validation_report_appendix_sections.append(
    dataframe_to_report_section(
        "Development OOF leaf-level misclassification summary",
        oof_leaf_level_misclassification_summary,
        "Development leaves misclassified in at least one repeated-CV evaluation for Scenario A, B or C. Holdout rows are excluded.",
        collapsible=True,
        section_id="development-oof-leaf-level-misclassification-summary",
    )
)

OOF leaf-level misclassification summary rows: 51


### 21.1 Threshold Sensitivity on Development 

This analysis examined how different classification thresholds affected development Macro-F1 and balanced accuracy. It was diagnostic only; the locked threshold remained at 0.5.

In [25]:
# Sweep development OOF thresholds diagnostically; keep the deployed threshold at 0.5.
if len(development_oof_leaf_scores) != len(meta_train_final):
    raise AssertionError(
        f"Expected {len(meta_train_final)} development leaves in development_oof_leaf_scores, "
        f"got {len(development_oof_leaf_scores)}."
    )

THRESHOLD_SWEEP = np.round(np.arange(0.30, 0.70 + 1e-9, 0.05), 2)
_threshold_y_true = development_oof_leaf_scores["y_true"].to_numpy(int)
_threshold_scores = development_oof_leaf_scores["Mean_Tipburn_Score"].to_numpy(float)

threshold_sensitivity_rows = []
for threshold in THRESHOLD_SWEEP:
    threshold_pred = (_threshold_scores >= threshold).astype(int)
    threshold_metrics = binary_classification_metrics(
        _threshold_y_true, threshold_pred, _threshold_scores,
        "development_oof_threshold_sweep", FINAL_MODEL_NAME,
    )
    threshold_sensitivity_rows.append({
        "Threshold": float(threshold),
        "Macro_F1": threshold_metrics["Macro_F1"],
        "Balanced_Accuracy": threshold_metrics["Balanced_Accuracy"],
        "FP": threshold_metrics["FP"],
        "FN": threshold_metrics["FN"],
    })
threshold_sensitivity_table = pd.DataFrame(threshold_sensitivity_rows)

# Scan all observed OOF scores to locate the exact Macro-F1 peak beyond the 0.05 grid.
_fine_thresholds = np.unique(np.round(_threshold_scores, 4))
_fine_macro_f1 = np.array([
    binary_classification_metrics(
        _threshold_y_true, (_threshold_scores >= t).astype(int), _threshold_scores,
        "development_oof_fine_scan", FINAL_MODEL_NAME,
    )["Macro_F1"]
    for t in _fine_thresholds
])
_peak_idx = int(np.argmax(_fine_macro_f1))
_peak_threshold = float(_fine_thresholds[_peak_idx])
_peak_macro_f1 = float(_fine_macro_f1[_peak_idx])
_macro_f1_at_0_5 = float(
    threshold_sensitivity_table.loc[threshold_sensitivity_table["Threshold"].eq(0.5), "Macro_F1"].iloc[0]
)
threshold_sensitivity_note = (
    f"The best development threshold was {_peak_threshold:.2f} "
    f"(Macro-F1 {_peak_macro_f1:.4f}); at 0.5 it was {_macro_f1_at_0_5:.4f}. "
    "The holdout was not used."
)

print(threshold_sensitivity_note)

validation_report_sections.append(dataframe_to_report_section(
    "Threshold sensitivity on development data",
    threshold_sensitivity_table,
    "Development-only threshold check from 0.30 to 0.70 using original-model OOF scores. " + threshold_sensitivity_note,
    row_tint_col="Threshold",
    row_tint_colors={"0.5": "#ffe9a8"},
    section_id="threshold-sensitivity-development-oof",
))

The best development threshold was 0.48 (Macro-F1 0.9166); at 0.5 it was 0.9166. The holdout was not used.


### 21.2 Finalised Misclassification Analysis

This analysis identified repeatedly misclassified development leaves and examined their spectral patterns to understand the model’s main error behaviour.

In [26]:
# Analyse Scenario A development OOF errors without refitting or accessing the holdout.
_seed_wrong = week_label_oof_predictions_all_seeds.copy()
_seed_wrong["Is_Wrong"] = (
    _seed_wrong["y_true"]
    .ne(_seed_wrong["y_pred"])
    .astype(int)
)

_seeds_wrong_per_leaf = (
    _seed_wrong
    .groupby(
        ["Sample_Group_ID", "Leaf_ID"],
        as_index=False,
    )
    .agg(
        Seeds_Wrong=("Is_Wrong", "sum"),
        Seeds_Total=("Seed", "nunique"),
    )
)

if not _seeds_wrong_per_leaf["Seeds_Total"].eq(
    len(REPEAT_SEEDS)
).all():
    raise AssertionError(
        "Expected every development leaf to have exactly "
        "len(REPEAT_SEEDS) seed predictions."
    )

original_model_leaf_misclassification = (
    development_oof_leaf_scores
    .merge(
        _seeds_wrong_per_leaf,
        on=["Sample_Group_ID", "Leaf_ID"],
        how="left",
        validate="one_to_one",
    )
)

if original_model_leaf_misclassification[
    "Seeds_Wrong"
].isna().any():
    raise AssertionError(
        "Some development leaves are missing per-seed "
        "wrongness counts."
    )

original_model_leaf_misclassification[
    "Predicted_Class_0_5"
] = np.where(
    original_model_leaf_misclassification[
        "Mean_Tipburn_Score"
    ] >= 0.5,
    "Tipburn",
    "Healthy",
)

original_model_leaf_misclassification[
    "Predicted_Label_Binary"
] = (
    original_model_leaf_misclassification[
        "Mean_Tipburn_Score"
    ] >= 0.5
).astype(int)

original_model_leaf_misclassification[
    "Correct_Wrong"
] = np.where(
    original_model_leaf_misclassification[
        "Predicted_Label_Binary"
    ].eq(
        original_model_leaf_misclassification["y_true"]
    ),
    "Correct",
    "Wrong",
)

original_model_leaf_misclassification = (
    original_model_leaf_misclassification
    .rename(
        columns={
            "Sample_Group_ID": "Lettuce_ID",
            "Mean_Tipburn_Score": "Mean_OOF_Score",
        }
    )
    .loc[
        :,
        [
            "Lettuce_ID",
            "Leaf_ID",
            "Label",
            "Week",
            "Predicted_Class_0_5",
            "Correct_Wrong",
            "Seeds_Wrong",
            "Mean_OOF_Score",
        ],
    ]
    .sort_values(
        ["Correct_Wrong", "Seeds_Wrong"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

if len(original_model_leaf_misclassification) != len(
    development_oof_leaf_scores
):
    raise AssertionError(
        "Per-leaf classification table row count does not "
        "match the development OOF leaf count."
    )

print(
    "Original-model development leaf classifications: "
    f"{len(original_model_leaf_misclassification)} leaves."
)

# Retain the complete leaf-level table in the final HTML only.
validation_report_appendix_sections.append(
    dataframe_to_report_section(
        "Original-model development leaf classifications",
        original_model_leaf_misclassification,
        "Complete OOF classifications from the original model using mean scores across seeds and threshold 0.5.",
        collapsible=True,
        section_id=(
            "original-model-development-leaf-classifications"
        ),
    )
)

# Identify Tipburn leaves misclassified in all repeated-CV seeds.
consistent_false_negatives = (
    original_model_leaf_misclassification
    .loc[
        (
            original_model_leaf_misclassification["Label"]
            .astype(str)
            .str.lower()
            .eq("tipburn")
        )
        & (
            original_model_leaf_misclassification[
                "Seeds_Wrong"
            ].eq(len(REPEAT_SEEDS))
        )
    ]
    .reset_index(drop=True)
)

n_consistent_fn = len(consistent_false_negatives)

n_week4_consistent_fn = int(
    consistent_false_negatives["Week"]
    .astype(str)
    .eq("Week 4")
    .sum()
)

if n_consistent_fn > 0:
    week4_fn_summary_note = (
        f"{n_week4_consistent_fn} of {n_consistent_fn} consistent false "
        f"negatives were from Week 4. Each was misclassified in all "
        f"{len(REPEAT_SEEDS)} repeated-CV seeds."
    )
else:
    week4_fn_summary_note = (
        "No consistent false negatives were found for the "
        f"original model across all {len(REPEAT_SEEDS)} seeds."
    )

print(week4_fn_summary_note)

validation_report_sections.append(
    dataframe_to_report_section(
        "Consistent false negatives",
        consistent_false_negatives,
        week4_fn_summary_note,
        section_id=(
            "consistent-false-negatives-original-model"
        ),
    )
)

# Compare consistent false-negative spectra with correctly classified
# Tipburn spectra.
correctly_classified_tipburn = (
    original_model_leaf_misclassification
    .loc[
        (
            original_model_leaf_misclassification["Label"]
            .astype(str)
            .str.lower()
            .eq("tipburn")
        )
        & (
            original_model_leaf_misclassification[
                "Correct_Wrong"
            ].eq("Correct")
        )
    ]
)

_n_spectra_examples = min(
    6,
    n_consistent_fn,
    len(correctly_classified_tipburn),
)

if _n_spectra_examples > 0:
    _fn_sample = consistent_false_negatives.sample(
        n=_n_spectra_examples,
        random_state=RANDOM_STATE,
    )

    _correct_sample = correctly_classified_tipburn.sample(
        n=_n_spectra_examples,
        random_state=RANDOM_STATE,
    )

    def _lookup_leaf_spectrum(
        lettuce_id,
        leaf_id,
    ) -> np.ndarray:
        match = leaf_train_final.loc[
            (
                leaf_train_final["Sample_Group_ID"]
                .astype(str)
                .eq(str(lettuce_id))
            )
            & (
                leaf_train_final["Leaf_ID"]
                .astype(str)
                .eq(str(leaf_id))
            )
        ]

        if match.empty:
            raise AssertionError(
                "No leaf-level spectrum found for "
                f"{lettuce_id}/{leaf_id}."
            )

        return match.iloc[0][
            wavelength_cols
        ].to_numpy(dtype=float)

    fig, ax = plt.subplots(
        figsize=(9, 5.2)
    )

    for _, row in _correct_sample.iterrows():
        ax.plot(
            wavelengths,
            _lookup_leaf_spectrum(
                row["Lettuce_ID"],
                row["Leaf_ID"],
            ),
            color="#4c78a8",
            alpha=0.75,
            linewidth=1.2,
        )

    for _, row in _fn_sample.iterrows():
        ax.plot(
            wavelengths,
            _lookup_leaf_spectrum(
                row["Lettuce_ID"],
                row["Leaf_ID"],
            ),
            color="#e45756",
            alpha=0.75,
            linewidth=1.2,
        )

    ax.plot(
        [],
        [],
        color="#4c78a8",
        label="Correctly classified Tipburn leaves",
    )

    ax.plot(
        [],
        [],
        color="#e45756",
        label="Consistent false-negative Tipburn leaves",
    )

    ax.set_title(
        "Original model: misclassified vs correctly "
        "classified Tipburn spectra"
    )
    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Raw reflectance")
    ax.legend()

    sns.despine(ax=ax)
    plt.tight_layout()

    validation_report_sections.append(
        figure_to_base64_html(
            fig,
            (
                "False-negative and correct Tipburn spectra"
            ),
            "Sample spectra from consistent false negatives and correctly classified Tipburn leaves.",
            section_id=(
                "original-model-misclassified-vs-correct-spectra"
            ),
        )
    )

    plt.close(fig)

else:
    print(
        "Skipping misclassified-versus-correct spectra figure: "
        "not enough examples in one of the two groups."
    )

# Check whether mixed training preserves, fixes, or introduces
# Scenario A errors.
_a_wrong_mask = (
    oof_leaf_level_misclassification_summary[
        "Total_Wrong_A_Original"
    ].gt(0)
)

_c_wrong_mask = (
    oof_leaf_level_misclassification_summary[
        "Total_Wrong_C_Mixed"
    ].gt(0)
)

_n_a_wrong = int(
    _a_wrong_mask.sum()
)

_n_retained = int(
    (
        _a_wrong_mask
        & _c_wrong_mask
    ).sum()
)

_n_c_only = int(
    (
        ~_a_wrong_mask
        & _c_wrong_mask
    ).sum()
)

_n_new_fn = int(
    (
        oof_leaf_level_misclassification_summary[
            "N_FN_A_Original"
        ].eq(0)
        & oof_leaf_level_misclassification_summary[
            "N_FN_C_Mixed"
        ].gt(0)
    ).sum()
)

cross_scenario_comparison_note = (
    f"Of the {_n_a_wrong} development leaves ever misclassified "
    "under the original Scenario A model, "
    f"{_n_retained} remain misclassified under the fold-size-matched "
    "mixed model. The mixed model therefore mostly retains Scenario "
    "A's errors rather than correcting them. It also introduces "
    f"{_n_c_only} errors among leaves that were never wrong under "
    f"Scenario A, including {_n_new_fn} new false negatives. This is "
    "a descriptive comparison of existing development OOF results "
    "and is not evidence that either scenario should replace the "
    "locked original model."
)

print(cross_scenario_comparison_note)

Original-model development leaf classifications: 288 leaves.
10 of 13 consistent false negatives were from Week 4. Each was misclassified in all 5 repeated-CV seeds.
Of the 35 development leaves ever misclassified under the original Scenario A model, 33 remain misclassified under the fold-size-matched mixed model. The mixed model therefore mostly retains Scenario A's errors rather than correcting them. It also introduces 10 errors among leaves that were never wrong under Scenario A, including 5 new false negatives. This is a descriptive comparison of existing development OOF results and is not evidence that either scenario should replace the locked original model.


## 22. Development Augmentation-Size Experiments

C1 and C2 examine whether increasing the training-set size with synthetic spectra changes development performance. C1 retains the complete real training fold and adds synthetic rows equal to 50% of the fold size, while C2 adds synthetic rows equal to 100% of the fold size. These experiments are separate from the size-matched A, B and C comparison. The model, feature set, threshold, alpha range and residual-noise scale remain unchanged.

In [27]:
# Evaluate C1/C2 as full real folds plus added synthetic rows; locked settings remain unchanged.

AUGMENTATION_DEV_SCENARIOS = [
    {
        "Scenario": "C1_augmented_50pct",
        "Display_Name": "C1 (augmentation): full real training fold + synthetic rows equal to 50% of fold size",
        "Synthetic_Ratio": 0.5,
        "Use_Real_Training": True,
    },
    {
        "Scenario": "C2_augmented_100pct",
        "Display_Name": "C2 (augmentation): full real training fold + synthetic rows equal to 100% of fold size",
        "Synthetic_Ratio": 1.0,
        "Use_Real_Training": True,
    },
]

augmentation_dev_results = run_development_synthetic_training_experiment(
    real_features=meta_train_final,
    real_leaf_spectra=leaf_train_final,
    repeated_scan_residuals=repeated_scan_residuals,
    wavelength_cols=wavelength_cols,
    wavelengths=wavelengths,
    forbidden_groups=holdout_groups,
    feature_columns=FINAL_FEATURE_COLS,
    repeat_seeds=REPEAT_SEEDS,
    scenarios=AUGMENTATION_DEV_SCENARIOS,
)
augmentation_dev_summary = augmentation_dev_results["summary"]

print("Augmentation Development C variants (C1/C2): full real fold + added synthetic.")
augmentation_dev_comparison_table = pd.concat(
    [synthetic_experiment_summary, augmentation_dev_summary],
    ignore_index=True,
)
augmentation_dev_comparison_table = augmentation_dev_comparison_table.sort_values("Macro_F1_mean", ascending=False).reset_index(drop=True)
SCENARIO_DISPLAY_NAMES = {
    "A_original_real_training_only": "A: Original",
    "B_synthetic_only_equal_real_count": "B: Synthetic-only",
    "C_Mixed_FoldMatched_50_50": "C: Mixed 50/50",
    "C1_augmented_50pct": "C1: Augmented +50%",
    "C2_augmented_100pct": "C2: Augmented +100%",
}
augmentation_dev_comparison_table = augmentation_dev_comparison_table.assign(
    Scenario=augmentation_dev_comparison_table["Scenario"].map(SCENARIO_DISPLAY_NAMES).fillna(augmentation_dev_comparison_table["Scenario"])
)
print(augmentation_dev_comparison_table[["Scenario", "Display_Name", "Macro_F1_mean", "ROC_AUC_mean"]].to_string())


# Add augmentation results to the report list; the complete HTML is written later.
validation_report_appendix_sections.append(dataframe_to_report_section(
    "Development augmentation results",
    augmentation_dev_comparison_table,
    "Development results for A (real only), B (synthetic only), C (50% real and 50% synthetic), and C1/C2, which add synthetic rows equal to 50% or 100% of the real training-fold size.",
    section_id="development-augmentation-size-experiments",
))

# Plot pooled OOF confusion matrices for C1 and C2.
augmentation_dev_predictions = augmentation_dev_results["predictions"]
augmentation_dev_scenario_order = [scenario["Scenario"] for scenario in AUGMENTATION_DEV_SCENARIOS]
# Compare locked A/B/C results with augmentation variants C1/C2.
abc_scenario_order = ["A_original_real_training_only", "B_synthetic_only_equal_real_count", "C_Mixed_FoldMatched_50_50"]
combined_conf_predictions = pd.concat([synthetic_experiment_predictions, augmentation_dev_predictions], ignore_index=True)
conf_scenario_order = abc_scenario_order + augmentation_dev_scenario_order
fig, axes = plt.subplots(1, len(conf_scenario_order), figsize=(4.8 * len(conf_scenario_order), 4.6))
axes = np.atleast_1d(axes)
for ax, scenario_name in zip(axes, conf_scenario_order):
    subset = combined_conf_predictions.loc[combined_conf_predictions["Scenario"].eq(scenario_name)]
    scenario_metrics = binary_classification_metrics(
        subset["y_true"].to_numpy(int), subset["y_pred"].to_numpy(int), subset["Tipburn_Score"].to_numpy(float),
        "train_val_grouped_cv_real_validation", FINAL_MODEL_NAME,
    )
    cm_values = np.array([[scenario_metrics["TN"], scenario_metrics["FP"]], [scenario_metrics["FN"], scenario_metrics["TP"]]])
    sns.heatmap(cm_values, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=["Healthy", "Tipburn"], yticklabels=["Healthy", "Tipburn"], ax=ax)
    _scen_label = SCENARIO_DISPLAY_NAMES.get(scenario_name, scenario_name)
    ax.set_title(f"{_scen_label}\nMacro-F1={scenario_metrics['F1_macro']:.3f}, AUC={scenario_metrics['ROC_AUC']:.3f}", fontsize=9)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
plt.tight_layout()
validation_report_appendix_sections.append(figure_to_base64_html(
    fig,
    "Development confusion matrices",
    "Pooled OOF confusion matrices for A, B, C, C1 and C2 at threshold 0.5.",
    section_id="development-augmentation-size-experiments-confusion-matrices",
))

# Compute paired Macro-F1 differences from A for every development scenario.
_main_fold_metrics = synthetic_experiment_fold_metrics[["Scenario", "Seed", "Fold", "Macro_F1"]]
_augmentation_fold_metrics = augmentation_dev_results["fold_metrics"][["Scenario", "Seed", "Fold", "Macro_F1"]]
_all_fold_metrics = pd.concat([_main_fold_metrics, _augmentation_fold_metrics], ignore_index=True)
_a_fold_macro = (
    _all_fold_metrics.loc[_all_fold_metrics["Scenario"].eq("A_original_real_training_only"), ["Seed", "Fold", "Macro_F1"]]
    .rename(columns={"Macro_F1": "Macro_F1_A"})
)
_macro_contrast_specs = [
    ("B_synthetic_only_equal_real_count", "B - A", "Size-matched (both trained on N rows)"),
    ("C_Mixed_FoldMatched_50_50", "C - A", "Size-matched (both trained on N rows)"),
    ("C1_augmented_50pct", "C1 - A", "Augmentation: full real fold + 50% synthetic (training grows to ~1.5N)"),
    ("C2_augmented_100pct", "C2 - A", "Augmentation: full real fold + 100% synthetic (training grows to ~2N)"),
]
_macro_rows = []
for _scen, _label, _note in _macro_contrast_specs:
    _sub = (
        _all_fold_metrics.loc[_all_fold_metrics["Scenario"].eq(_scen), ["Seed", "Fold", "Macro_F1"]]
        .merge(_a_fold_macro, on=["Seed", "Fold"], how="inner")
    )
    if _sub.empty:
        continue
    _diff = _sub["Macro_F1"] - _sub["Macro_F1_A"]
    _macro_rows.append({
        "Contrast": _label,
        "Design_Note": _note,
        "Folds": int(len(_diff)),
        "Mean_Delta_Macro_F1": float(_diff.mean()),
        "Std_Delta_Macro_F1": float(_diff.std()),
        "Median_Delta_Macro_F1": float(_diff.median()),
        "Min_Delta_Macro_F1": float(_diff.min()),
        "Max_Delta_Macro_F1": float(_diff.max()),
        "Folds_Better_Than_A": int((_diff > 0).sum()),
    })
macro_f1_vs_a_summary = pd.DataFrame(_macro_rows)
validation_report_appendix_sections.append(dataframe_to_report_section(
    "Macro-F1 compared with Scenario A",
    macro_f1_vs_a_summary,
    "Per-fold Macro-F1 difference from original-only Scenario A. B and C use equal training sizes; C1 and C2 add synthetic rows to all real training rows.",
    section_id="development-macro-f1-vs-a-comparison",
))


# Pool synthetic generation counts across all development folds.
_gen_all = pd.concat([
    synthetic_experiment_generation_summary,
    augmentation_dev_results["generation_summary"],
], ignore_index=True)
generation_scenario_summary = (
    _gen_all.groupby("Scenario", as_index=False)
    .agg(Requested=("Requested_Count", "sum"), Accepted=("Accepted_Count", "sum"),
         Rejected=("Rejected_Count", "sum"), Attempts=("Attempt_Count", "sum"))
)
if not generation_scenario_summary["Scenario"].eq("A_original_real_training_only").any():
    generation_scenario_summary = pd.concat([
        pd.DataFrame([{"Scenario": "A_original_real_training_only", "Requested": 0, "Accepted": 0, "Rejected": 0, "Attempts": 0}]),
        generation_scenario_summary,
    ], ignore_index=True)
_gen_scenario_order = {"A_original_real_training_only": 0, "B_synthetic_only_equal_real_count": 1, "C_Mixed_FoldMatched_50_50": 2, "C1_augmented_50pct": 3, "C2_augmented_100pct": 4}
generation_scenario_summary = (
    generation_scenario_summary.assign(_ord=generation_scenario_summary["Scenario"].map(_gen_scenario_order).fillna(99))
    .sort_values("_ord").drop(columns="_ord").reset_index(drop=True)
)
generation_scenario_summary = generation_scenario_summary.assign(
    Scenario=generation_scenario_summary["Scenario"].map(SCENARIO_DISPLAY_NAMES).fillna(generation_scenario_summary["Scenario"])
)
validation_report_appendix_sections.append(dataframe_to_report_section(
    "Synthetic generation summary",
    generation_scenario_summary,
    "Requested, accepted and rejected synthetic counts across all development folds for each scenario.",
    section_id="synthetic-validation-generation-summary",
))

Augmentation Development C variants (C1/C2): full real fold + added synthetic.
              Scenario                                                                                                                                                                                                          Display_Name  Macro_F1_mean  ROC_AUC_mean
0   C1: Augmented +50%                                                                                                                                 C1 (augmentation): full real training fold + synthetic rows equal to 50% of fold size       0.910369      0.957571
1  C2: Augmented +100%                                                                                                                                C2 (augmentation): full real training fold + synthetic rows equal to 100% of fold size       0.908965      0.957436
2          A: Original                                                                                                     

## 23. Synthetic-Generator Alpha/Noise Sensitivity

Pre-committed 12-point grid (3 alpha ranges x 4 noise scales) over the synthetic-generator's mixup alpha range and residual-noise scale, run through the existing repeated plant-grouped Development CV experiment (5 seeds x 5 folds) for scenarios A, B, and C_Mixed_FoldMatched_50_50 only. Development CV only; the fixed final holdout is never loaded, fit, or evaluated in this cell. The grid, objective, and metrics were pre-committed before running and are not modified based on results. 


In [28]:
# Test the predefined alpha/noise grid on development CV only; the holdout is not accessed.

SENSITIVITY_ALPHA_RANGES = [(0.2, 0.8), (0.3, 0.7), (0.4, 0.6)]
SENSITIVITY_NOISE_SCALES = [0.1, 0.2, 0.3, 0.5]
SENSITIVITY_SCENARIO_NAMES = [
    "A_original_real_training_only",
    "B_synthetic_only_equal_real_count",
    "C_Mixed_FoldMatched_50_50",
]
SENSITIVITY_SCENARIOS = [s for s in SYNTHETIC_SCENARIOS if s["Scenario"] in SENSITIVITY_SCENARIO_NAMES]
if len(SENSITIVITY_SCENARIOS) != 3:
    raise AssertionError("Expected exactly 3 scenarios (A, B, C_Mixed_FoldMatched_50_50) for the sensitivity analysis.")
if {s["Scenario"] for s in SENSITIVITY_SCENARIOS} != set(SENSITIVITY_SCENARIO_NAMES):
    raise AssertionError("Sensitivity scenario subset does not match the expected 3 scenario names.")

SENSITIVITY_METRICS = ["Macro_F1", "Balanced_Accuracy", "ROC_AUC", "Recall_Tipburn", "Precision_Tipburn", "FP", "FN"]
SENSITIVITY_SCENARIO_SHORT = {
    "A_original_real_training_only": "A",
    "B_synthetic_only_equal_real_count": "B",
    "C_Mixed_FoldMatched_50_50": "C",
}


def paired_seed_fold_diff(fold_metrics_df: pd.DataFrame,
                          treatment_scenario: str,
                          baseline_scenario: str,
                          metrics: list[str]) -> dict[str, float]:
    """Paired per-(Seed,Fold) treatment-minus-baseline differences: mean/median/SD/win/tie/loss."""
    left = fold_metrics_df.loc[fold_metrics_df["Scenario"].eq(treatment_scenario)]
    right = fold_metrics_df.loc[fold_metrics_df["Scenario"].eq(baseline_scenario)]
    merged = left.merge(right, on=["Seed", "Fold"], suffixes=("_treat", "_base"))
    if len(merged) != 25:
        raise AssertionError(
            f"Expected 25 paired Seed/Fold rows for {treatment_scenario} vs {baseline_scenario}, got {len(merged)}."
        )
    result: dict[str, float] = {}
    for metric in metrics:
        diff = merged[f"{metric}_treat"] - merged[f"{metric}_base"]
        result[f"{metric}_Diff_Mean"] = float(diff.mean())
        result[f"{metric}_Diff_Median"] = float(diff.median())
        result[f"{metric}_Diff_SD"] = float(diff.std())
        result[f"{metric}_Win"] = int((diff > 0).sum())
        result[f"{metric}_Tie"] = int((diff == 0).sum())
        result[f"{metric}_Loss"] = int((diff < 0).sum())
    return result


sensitivity_rows: list[dict[str, Any]] = []
sensitivity_combo_index = 0
for alpha_low, alpha_high in SENSITIVITY_ALPHA_RANGES:
    for noise_scale in SENSITIVITY_NOISE_SCALES:
        sensitivity_combo_index += 1
        row: dict[str, Any] = {
            "Combo_Index": sensitivity_combo_index,
            "Alpha_Low": alpha_low,
            "Alpha_High": alpha_high,
            "Noise_Scale": noise_scale,
            "Status": "ok",
            "Error_Message": "",
        }
        try:
            combo_result = run_development_synthetic_training_experiment(
                real_features=meta_train_final,
                real_leaf_spectra=leaf_train_final,
                repeated_scan_residuals=repeated_scan_residuals,
                wavelength_cols=wavelength_cols,
                wavelengths=wavelengths,
                forbidden_groups=holdout_groups,
                feature_columns=FINAL_FEATURE_COLS,
                repeat_seeds=REPEAT_SEEDS,
                alpha_low=alpha_low,
                alpha_high=alpha_high,
                residual_scale=noise_scale,
                scenarios=SENSITIVITY_SCENARIOS,
            )
            fold_metrics_combo = combo_result["fold_metrics"]
            if len(fold_metrics_combo) != len(SENSITIVITY_SCENARIOS) * 25:
                raise AssertionError(
                    f"Expected {len(SENSITIVITY_SCENARIOS) * 25} fold-metric rows, got {len(fold_metrics_combo)}."
                )
            if set(fold_metrics_combo["Scenario"].unique()) != set(SENSITIVITY_SCENARIO_NAMES):
                raise AssertionError("Sensitivity fold-metrics scenario set does not match the expected 3 scenarios.")

            mean_macro_f1_by_scenario = fold_metrics_combo.groupby("Scenario")["Macro_F1"].mean().to_dict()
            for scenario_name, short in SENSITIVITY_SCENARIO_SHORT.items():
                row[f"{short}_Mean_Macro_F1"] = float(mean_macro_f1_by_scenario[scenario_name])

            c_vs_a = paired_seed_fold_diff(
                fold_metrics_combo, "C_Mixed_FoldMatched_50_50", "A_original_real_training_only", SENSITIVITY_METRICS
            )
            b_vs_a = paired_seed_fold_diff(
                fold_metrics_combo, "B_synthetic_only_equal_real_count", "A_original_real_training_only", SENSITIVITY_METRICS
            )
            for key, value in c_vs_a.items():
                row[f"C_minus_A_{key}"] = value
            for key, value in b_vs_a.items():
                row[f"B_minus_A_{key}"] = value

            macro_f1_rank = sorted(
                [(short, row[f"{short}_Mean_Macro_F1"]) for short in ["A", "B", "C"]],
                key=lambda item: item[1],
                reverse=True,
            )
            row["Development_Ranking_By_Macro_F1"] = " > ".join(f"{name}({value:.4f})" for name, value in macro_f1_rank)
            print(
                f"[{sensitivity_combo_index}/12] alpha=({alpha_low},{alpha_high}) noise={noise_scale}: "
                f"A={row['A_Mean_Macro_F1']:.4f} B={row['B_Mean_Macro_F1']:.4f} C={row['C_Mean_Macro_F1']:.4f}"
            )
        except (AssertionError, RuntimeError) as exc:
            # Record generation failures so every predefined grid point remains in the results.
            row["Status"] = "failed"
            row["Error_Message"] = str(exc)
            print(f"[{sensitivity_combo_index}/12] alpha=({alpha_low},{alpha_high}) noise={noise_scale}: FAILED - {exc}")

        sensitivity_rows.append(row)

if len(sensitivity_rows) != 12:
    raise AssertionError(f"Expected 12 grid combinations, got {len(sensitivity_rows)}.")

synthetic_sensitivity_results = pd.DataFrame(sensitivity_rows)

n_failed = int(synthetic_sensitivity_results["Status"].eq("failed").sum())

ok_results = synthetic_sensitivity_results.loc[synthetic_sensitivity_results["Status"].eq("ok")]
if not ok_results.empty:
# Optimise mean development Macro-F1 for the locked fold-matched mixed scenario.
    best_row = ok_results.loc[ok_results["C_Mean_Macro_F1"].idxmax()]
    current_settings_row = ok_results.loc[
        (ok_results["Alpha_Low"] == SYNTHETIC_ALPHA_LOW)
        & (ok_results["Alpha_High"] == SYNTHETIC_ALPHA_HIGH)
        & (ok_results["Noise_Scale"] == SYNTHETIC_RESIDUAL_SCALE)
    ]
    summary_cols = [
        "Combo_Index", "Alpha_Low", "Alpha_High", "Noise_Scale",
        "A_Mean_Macro_F1", "B_Mean_Macro_F1", "C_Mean_Macro_F1",
        "Development_Ranking_By_Macro_F1",
    ]
    print("\n=== Pre-committed objective: mean development macro-F1 for C_Mixed_FoldMatched_50_50 ===")
    print("(Reported for characterisation only -- not a basis for changing the locked synthetic settings.)")
    print("\nBest combination by objective:")
    print(best_row[summary_cols].to_string())
    print("\nCurrent-settings row (alpha 0.3-0.7, noise 0.2) for comparison:")
    if current_settings_row.empty:
        print("  (current settings are not part of the pre-committed grid)")
    else:
        print(current_settings_row[summary_cols].to_string(index=False))

print("\nFull 12-row A/B/C ranking (all combinations, per the framing note -- not just the best):")
print(synthetic_sensitivity_results[
    ["Combo_Index", "Alpha_Low", "Alpha_High", "Noise_Scale", "Status", "Development_Ranking_By_Macro_F1"]
].to_string(index=False))

_alpha_noise_cols = [
    "Combo_Index", "Alpha_Low", "Alpha_High", "Noise_Scale",
    "A_Mean_Macro_F1", "B_Mean_Macro_F1", "C_Mean_Macro_F1",
    "Development_Ranking_By_Macro_F1", "Status",
]
alpha_noise_sensitivity_display = synthetic_sensitivity_results.reindex(columns=_alpha_noise_cols)
validation_report_appendix_sections.append(dataframe_to_report_section(
    "Alpha and noise sensitivity",
    alpha_noise_sensitivity_display,
    "Development-only results for the 3 x 4 alpha and noise grid. This checks whether the synthetic conclusions change under different settings. The holdout and locked settings were unchanged.",
    collapsible=True,
    section_id="synthetic-alpha-noise-sensitivity",
))

[1/12] alpha=(0.2,0.8) noise=0.1: A=0.9081 B=0.8942 C=0.9024
[2/12] alpha=(0.2,0.8) noise=0.2: A=0.9081 B=0.8942 C=0.9024
[3/12] alpha=(0.2,0.8) noise=0.3: A=0.9081 B=0.8928 C=0.9017
[4/12] alpha=(0.2,0.8) noise=0.5: A=0.9081 B=0.8928 C=0.9017
[5/12] alpha=(0.3,0.7) noise=0.1: A=0.9081 B=0.8928 C=0.9024
[6/12] alpha=(0.3,0.7) noise=0.2: A=0.9081 B=0.8928 C=0.9017
[7/12] alpha=(0.3,0.7) noise=0.3: A=0.9081 B=0.8914 C=0.9017
[8/12] alpha=(0.3,0.7) noise=0.5: A=0.9081 B=0.8921 C=0.9017
[9/12] alpha=(0.4,0.6) noise=0.1: A=0.9081 B=0.8915 C=0.9038
[10/12] alpha=(0.4,0.6) noise=0.2: A=0.9081 B=0.8915 C=0.9038
[11/12] alpha=(0.4,0.6) noise=0.3: A=0.9081 B=0.8908 C=0.9038
[12/12] alpha=(0.4,0.6) noise=0.5: A=0.9081 B=0.8921 C=0.9038

=== Pre-committed objective: mean development macro-F1 for C_Mixed_FoldMatched_50_50 ===
(Reported for characterisation only -- not a basis for changing the locked synthetic settings.)

Best combination by objective:
Combo_Index                                    

## 24. Supplementary Wavelength-Pair Index Search

This diagnostic tested whether a newly selected two-wavelength index could improve the locked five-feature model. The wavelength pair was chosen within each plant-grouped training fold only and compared against the unchanged five-feature baseline. The holdout was not used, and the results did not modify the locked model or final feature set.

In [29]:
# Search development folds for an additional two-wavelength index without changing the locked pipeline.

sk = require_sklearn()
roc_auc_score = sk["roc_auc_score"]

WAVELENGTH_SEARCH_MIN_GAP_NM = 20.0
WAVELENGTH_SEARCH_DENOM_MIN_ABS = 0.02
WAVELENGTH_SEARCH_BASELINE_LABEL = "Locked_5_Feature_Baseline"
WAVELENGTH_SEARCH_PLUS_INDEX_LABEL = "Locked_5_Feature_Plus_DataDriven_Index"

# Reuse per-leaf smoothing because it has no cross-sample statistics or leakage.
smoothed_dev = processed["smoothed"][train_mask]
if smoothed_dev.shape[0] != len(X_train_final):
    raise AssertionError("Development smoothed-reflectance matrix is not aligned with X_train_final.")

wavelength_search_candidate_pairs = [
    (i, j)
    for i in range(len(wavelengths))
    for j in range(i + 1, len(wavelengths))
    if (wavelengths[j] - wavelengths[i]) >= WAVELENGTH_SEARCH_MIN_GAP_NM
]
if not wavelength_search_candidate_pairs:
    raise AssertionError("No candidate wavelength pairs satisfy the minimum gap constraint.")


def select_best_wavelength_pair(smoothed_fit: np.ndarray, y_fit: np.ndarray) -> dict[str, Any]:
    """Select the (lambda1, lambda2) pair most separable for y_fit, using fit rows only."""
    if smoothed_fit.shape[0] != len(y_fit):
        raise AssertionError("Fit-fold smoothed reflectance and labels are not aligned.")
    grid = np.stack([
        interpolate_each_row(smoothed_fit, wavelengths, wl) for wl in wavelengths
    ])  # wavelengths x fit rows
    best_score = -1.0
    best_pair = None
    n_excluded = 0
    for i, j in wavelength_search_candidate_pairs:
        ra, rb = grid[i], grid[j]
        denom = ra + rb
        if np.min(np.abs(denom)) < WAVELENGTH_SEARCH_DENOM_MIN_ABS:
            n_excluded += 1
            continue
        index_values = (rb - ra) / (denom + EPS)
        if np.std(index_values) < EPS:
            continue
        auc = roc_auc_score(y_fit, index_values)
        score = max(auc, 1.0 - auc)
        if score > best_score:
            best_score = score
            best_pair = (i, j)
    if best_pair is None:
        raise AssertionError("No candidate wavelength pair was usable for this fold.")
    return {
        "Lambda1_nm": float(wavelengths[best_pair[0]]),
        "Lambda2_nm": float(wavelengths[best_pair[1]]),
        "Selection_Score": float(best_score),
        "N_Rows_Scored": int(smoothed_fit.shape[0]),
        "N_Candidates_Evaluated": len(wavelength_search_candidate_pairs),
        "N_Candidates_Excluded_Near_Zero_Denominator": n_excluded,
    }


def apply_wavelength_pair(smoothed_subset: np.ndarray, lambda1: float, lambda2: float) -> np.ndarray:
    ra = interpolate_each_row(smoothed_subset, wavelengths, lambda1)
    rb = interpolate_each_row(smoothed_subset, wavelengths, lambda2)
    index_values = (rb - ra) / (ra + rb + EPS)
    if not np.isfinite(index_values).all():
        raise AssertionError("Applied wavelength-pair index produced non-finite values.")
    return index_values


wavelength_search_fold_choices: list[dict[str, Any]] = []
wavelength_search_fold_metric_rows: list[dict[str, Any]] = []

for seed in REPEAT_SEEDS:
    outer_cv = make_group_cv(y_train_final, groups_train_final, 5, seed)
    for fold_idx, (fit_idx, val_idx) in enumerate(
        outer_cv.split(X_train_final.to_numpy(float), y_train_final, groups_train_final), start=1
    ):
        fit_groups = set(groups_train_final[fit_idx])
        val_groups = set(groups_train_final[val_idx])
        if fit_groups.intersection(val_groups):
            raise AssertionError("Training and validation plants overlap within the wavelength-search fold.")
        if fit_groups.intersection(holdout_groups) or val_groups.intersection(holdout_groups):
            raise AssertionError("Holdout plants were used inside the wavelength-search experiment.")

# Select the wavelength pair from the fit fold only; exclude validation and holdout rows.
        selection = select_best_wavelength_pair(smoothed_dev[fit_idx], y_train_final[fit_idx])
        if selection["N_Rows_Scored"] != len(fit_idx):
            raise AssertionError("Wavelength-pair selection did not score exactly the fit-fold rows.")
        lambda1_star = selection["Lambda1_nm"]
        lambda2_star = selection["Lambda2_nm"]
        if abs(lambda1_star - lambda2_star) < WAVELENGTH_SEARCH_MIN_GAP_NM:
            raise AssertionError("Selected wavelength pair violates the minimum-gap constraint.")

        index_fit = apply_wavelength_pair(smoothed_dev[fit_idx], lambda1_star, lambda2_star)
        index_val = apply_wavelength_pair(smoothed_dev[val_idx], lambda1_star, lambda2_star)

        X_fit_5 = X_train_final.iloc[fit_idx].to_numpy(float)
        X_val_5 = X_train_final.iloc[val_idx].to_numpy(float)
        X_fit_6 = np.hstack([X_fit_5, index_fit.reshape(-1, 1)])
        X_val_6 = np.hstack([X_val_5, index_val.reshape(-1, 1)])
        y_fit = y_train_final[fit_idx]
        y_val = y_train_final[val_idx]

        for arm_label, X_fit_arm, X_val_arm in [
            (WAVELENGTH_SEARCH_BASELINE_LABEL, X_fit_5, X_val_5),
            (WAVELENGTH_SEARCH_PLUS_INDEX_LABEL, X_fit_6, X_val_6),
        ]:
# Fit identical locked pipelines in both arms so only the input features differ.
            arm_model = clone(build_locked_final_pipeline(RANDOM_STATE))
            arm_model.fit(X_fit_arm, y_fit)
            arm_score = continuous_model_score(arm_model, X_val_arm)
            arm_pred = (arm_score >= 0.5).astype(int)
            if not np.array_equal(arm_pred, arm_model.predict(X_val_arm)):
                raise AssertionError("Explicit 0.5 threshold predictions differ from model.predict().")
            arm_metrics = binary_classification_metrics(
                y_val, arm_pred, arm_score, "train_val_grouped_cv_wavelength_search", arm_label
            )
            arm_metrics.update({
                "Scenario": arm_label,
                "Display_Name": arm_label,
                "Seed": seed,
                "Fold": fold_idx,
            })
            wavelength_search_fold_metric_rows.append(arm_metrics)

        wavelength_search_fold_choices.append({
            "Seed": seed,
            "Fold": fold_idx,
            **selection,
        })

wavelength_search_fold_metrics = pd.DataFrame(wavelength_search_fold_metric_rows)
wavelength_search_fold_choices_df = pd.DataFrame(wavelength_search_fold_choices)
if len(wavelength_search_fold_choices_df) != len(REPEAT_SEEDS) * 5:
    raise AssertionError("Wavelength-search did not produce exactly one selection per seed/fold.")

wavelength_search_metric_summary = aggregate_fold_metrics(wavelength_search_fold_metrics)

# Calculate paired per-fold differences between the added-index and baseline arms.
_baseline_folds = wavelength_search_fold_metrics.loc[
    wavelength_search_fold_metrics["Scenario"].eq(WAVELENGTH_SEARCH_BASELINE_LABEL)
]
_plus_index_folds = wavelength_search_fold_metrics.loc[
    wavelength_search_fold_metrics["Scenario"].eq(WAVELENGTH_SEARCH_PLUS_INDEX_LABEL)
]
_paired_diff_metrics = ["Macro_F1", "ROC_AUC", "Precision_Tipburn", "Recall_Tipburn", "Balanced_Accuracy"]
_wavelength_search_merged = _plus_index_folds.merge(
    _baseline_folds, on=["Seed", "Fold"], suffixes=("_Plus_Index", "_Baseline"),
)
wavelength_search_paired_diff = pd.DataFrame([
    {
        "Seed": int(row["Seed"]),
        "Fold": int(row["Fold"]),
        **{
            f"{metric}_Difference": row[f"{metric}_Plus_Index"] - row[f"{metric}_Baseline"]
            for metric in _paired_diff_metrics
        },
    }
    for _, row in _wavelength_search_merged.iterrows()
])
wavelength_search_paired_diff_summary = pd.DataFrame([
    {
        "Metric": metric,
        "Mean_Difference": float(wavelength_search_paired_diff[f"{metric}_Difference"].mean()),
        "Std_Difference": float(wavelength_search_paired_diff[f"{metric}_Difference"].std()),
        "Wins": int((wavelength_search_paired_diff[f"{metric}_Difference"] > 0).sum()),
        "Ties": int((wavelength_search_paired_diff[f"{metric}_Difference"] == 0).sum()),
        "Losses": int((wavelength_search_paired_diff[f"{metric}_Difference"] < 0).sum()),
    }
    for metric in _paired_diff_metrics
])

# Summarise wavelength-pair selection stability across the 25 folds.
wavelength_search_pair_stability = (
    wavelength_search_fold_choices_df
    .groupby(["Lambda1_nm", "Lambda2_nm"], as_index=False)
    .size()
    .rename(columns={"size": "Selection_Count"})
    .sort_values("Selection_Count", ascending=False)
    .reset_index(drop=True)
)
wavelength_search_stability_stats = pd.DataFrame([{
    "Total_Folds": len(wavelength_search_fold_choices_df),
    "Distinct_Pairs_Chosen": len(wavelength_search_pair_stability),
    "Most_Frequent_Pair": (
        f"({wavelength_search_pair_stability.loc[0, 'Lambda1_nm']:.0f}, "
        f"{wavelength_search_pair_stability.loc[0, 'Lambda2_nm']:.0f}) nm"
    ),
    "Most_Frequent_Pair_Count": int(wavelength_search_pair_stability.loc[0, "Selection_Count"]),
    "Lambda1_Std_Across_Folds": float(wavelength_search_fold_choices_df["Lambda1_nm"].std()),
    "Lambda2_Std_Across_Folds": float(wavelength_search_fold_choices_df["Lambda2_nm"].std()),
}])


def _search_stat(scenario: str, metric: str, stat: str) -> float:
    row = wavelength_search_metric_summary.loc[
        wavelength_search_metric_summary["Scenario"].eq(scenario)
        & wavelength_search_metric_summary["Metric"].eq(metric)
    ]
    return float(row[stat].iloc[0])


print(
    "Wavelength-pair search (25 seed/fold draws, development-only):\n"
    f"  Baseline (5 features):      Macro_F1 = {_search_stat(WAVELENGTH_SEARCH_BASELINE_LABEL, 'Macro_F1', 'Mean'):.4f} "
    f"+/- {_search_stat(WAVELENGTH_SEARCH_BASELINE_LABEL, 'Macro_F1', 'Std'):.4f}, "
    f"ROC_AUC = {_search_stat(WAVELENGTH_SEARCH_BASELINE_LABEL, 'ROC_AUC', 'Mean'):.4f} "
    f"+/- {_search_stat(WAVELENGTH_SEARCH_BASELINE_LABEL, 'ROC_AUC', 'Std'):.4f}\n"
    f"  Plus data-driven index (6): Macro_F1 = {_search_stat(WAVELENGTH_SEARCH_PLUS_INDEX_LABEL, 'Macro_F1', 'Mean'):.4f} "
    f"+/- {_search_stat(WAVELENGTH_SEARCH_PLUS_INDEX_LABEL, 'Macro_F1', 'Std'):.4f}, "
    f"ROC_AUC = {_search_stat(WAVELENGTH_SEARCH_PLUS_INDEX_LABEL, 'ROC_AUC', 'Mean'):.4f} "
    f"+/- {_search_stat(WAVELENGTH_SEARCH_PLUS_INDEX_LABEL, 'ROC_AUC', 'Std'):.4f}\n"
    f"  Most frequent pair: {wavelength_search_stability_stats.loc[0, 'Most_Frequent_Pair']} in "
    f"{wavelength_search_stability_stats.loc[0, 'Most_Frequent_Pair_Count']}/{wavelength_search_stability_stats.loc[0, 'Total_Folds']} folds "
    f"({wavelength_search_stability_stats.loc[0, 'Distinct_Pairs_Chosen']} distinct pairs chosen overall)."
)

validation_report_appendix_sections.append(dataframe_to_report_section(
    "Wavelength-pair metric comparison",
    wavelength_search_metric_summary,
    "Development CV comparison of the final five-feature model with and without a two-wavelength index. Each pair was selected using training-fold data only.",
    section_id="wavelength-search-metric-comparison",
))
validation_report_appendix_sections.append(dataframe_to_report_section(
    "Wavelength-pair differences",
    wavelength_search_paired_diff_summary,
    "Per-fold performance difference between the model with the index and the five-feature model.",
    section_id="wavelength-search-paired-diff-summary",
))
validation_report_appendix_sections.append(dataframe_to_report_section(
    "Wavelength-pair selection frequency",
    wavelength_search_pair_stability,
    "Number of times each wavelength pair was selected across the 25 folds.",
    section_id="wavelength-search-pair-stability",
))
validation_report_appendix_sections.append(dataframe_to_report_section(
    "Wavelength-pair by fold",
    wavelength_search_fold_choices_df,
    "Selected wavelength pair and training score for each seed and fold.",
    collapsible=True,
    section_id="wavelength-search-fold-choices",
))

Wavelength-pair search (25 seed/fold draws, development-only):
  Baseline (5 features):      Macro_F1 = 0.9081 +/- 0.0439, ROC_AUC = 0.9599 +/- 0.0301
  Plus data-driven index (6): Macro_F1 = 0.9078 +/- 0.0483, ROC_AUC = 0.9730 +/- 0.0226
  Most frequent pair: (590, 610) nm in 7/25 folds (8 distinct pairs chosen overall).


## 25. Supplementary Synthetic-Draw Sensitivity

This diagnostic tested whether the development performance difference between synthetic-only and mixed training changed across repeated synthetic draws. Cross-validation folds, real training data and model settings were fixed, while only the synthetic-generation seed varied across 10 repetitions. The analysis quantified synthetic-draw variation relative to fold-to-fold variation without changing the locked model or holdout evaluation.

In [30]:
# Repeat synthetic draws on fixed development folds without changing the model or holdout.

DRAW_SENSITIVITY_REPS = 10
DRAW_SENSITIVITY_OFFSET_STEP = 10_000  # Keep redraw seeds outside the base seed range.
DRAW_SENSITIVITY_B_SCENARIO = "B_synthetic_only_equal_real_count"
DRAW_SENSITIVITY_C_SCENARIO = "C_Mixed_FoldMatched_50_50"

# Confirm offset 0 exactly reproduces the locked synthetic run.
draw_sensitivity_baseline = run_development_synthetic_training_experiment(
    real_features=meta_train_final,
    real_leaf_spectra=leaf_train_final,
    repeated_scan_residuals=repeated_scan_residuals,
    wavelength_cols=wavelength_cols,
    wavelengths=wavelengths,
    forbidden_groups=holdout_groups,
    feature_columns=FINAL_FEATURE_COLS,
    repeat_seeds=REPEAT_SEEDS,
    scenarios=SYNTHETIC_SCENARIOS,
    synthetic_seed_offset=0,
)
pd.testing.assert_frame_equal(
    draw_sensitivity_baseline["summary"], synthetic_experiment_summary, check_exact=True
)
print("Verification gate passed: synthetic_seed_offset=0 reproduces synthetic_experiment_summary exactly.")

# Run ten redraws while holding folds and real rows fixed; only synthetic seeds change.
draw_sensitivity_fold_frames = []
draw_sensitivity_provenance_frames = []
for draw_rep in range(1, DRAW_SENSITIVITY_REPS + 1):
    _rep_result = run_development_synthetic_training_experiment(
        real_features=meta_train_final,
        real_leaf_spectra=leaf_train_final,
        repeated_scan_residuals=repeated_scan_residuals,
        wavelength_cols=wavelength_cols,
        wavelengths=wavelengths,
        forbidden_groups=holdout_groups,
        feature_columns=FINAL_FEATURE_COLS,
        repeat_seeds=REPEAT_SEEDS,
        scenarios=SYNTHETIC_SCENARIOS,
        synthetic_seed_offset=draw_rep * DRAW_SENSITIVITY_OFFSET_STEP,
    )
    _rep_folds = _rep_result["fold_metrics"].copy()
    _rep_folds["Draw_Rep"] = draw_rep
    draw_sensitivity_fold_frames.append(_rep_folds)
    if not _rep_result["provenance"].empty:
        draw_sensitivity_provenance_frames.append(_rep_result["provenance"])

draw_sensitivity_fold_metrics = pd.concat(draw_sensitivity_fold_frames, ignore_index=True)

# Recheck pooled provenance for holdout leakage.
draw_sensitivity_provenance = (
    pd.concat(draw_sensitivity_provenance_frames, ignore_index=True)
    if draw_sensitivity_provenance_frames else pd.DataFrame()
)
if not draw_sensitivity_provenance.empty:
    _src_groups = (
        set(draw_sensitivity_provenance["Source_Plant_A"].astype(str))
        | set(draw_sensitivity_provenance["Source_Plant_B"].astype(str))
        | set(draw_sensitivity_provenance["Residual_Source_Plant"].astype(str))
    )
    if _src_groups.intersection(holdout_groups):
        raise AssertionError("Holdout plants appeared in draw-sensitivity synthetic provenance.")

# Calculate Macro-F1(C) - Macro-F1(B) for each seed, fold, and redraw.
draw_sensitivity_wide = draw_sensitivity_fold_metrics.pivot_table(
    index=["Seed", "Fold", "Draw_Rep"], columns="Scenario", values="Macro_F1"
).reset_index()
for _needed in (DRAW_SENSITIVITY_B_SCENARIO, DRAW_SENSITIVITY_C_SCENARIO):
    if _needed not in draw_sensitivity_wide.columns:
        raise AssertionError(f"Scenario {_needed} missing from draw-sensitivity fold metrics.")
draw_sensitivity_wide["delta"] = (
    draw_sensitivity_wide[DRAW_SENSITIVITY_C_SCENARIO] - draw_sensitivity_wide[DRAW_SENSITIVITY_B_SCENARIO]
)
_expected_rows = len(REPEAT_SEEDS) * 5 * DRAW_SENSITIVITY_REPS
if len(draw_sensitivity_wide) != _expected_rows:
    raise AssertionError(
        f"Expected {_expected_rows} (Seed,Fold,Draw_Rep) delta rows, got {len(draw_sensitivity_wide)}."
    )

# Estimate draw-level variation within each fixed seed/fold.
_sd_draw_per_fold = draw_sensitivity_wide.groupby(["Seed", "Fold"])["delta"].std(ddof=1)
draw_sensitivity_sd_draw = float(_sd_draw_per_fold.mean())
# Estimate fold-level variation within each redraw.
_sd_fold_per_rep = draw_sensitivity_wide.groupby("Draw_Rep")["delta"].std(ddof=1)
draw_sensitivity_sd_fold = float(_sd_fold_per_rep.mean())
_var_draw = draw_sensitivity_sd_draw ** 2
_var_fold = draw_sensitivity_sd_fold ** 2
draw_sensitivity_var_ratio = float(_var_draw / (_var_draw + _var_fold)) if (_var_draw + _var_fold) > 0 else 0.0
draw_sensitivity_max_se_reduction_pct = float((1.0 - np.sqrt(1.0 - draw_sensitivity_var_ratio)) * 100.0)

# Check delta-sign consistency across redraws within each seed/fold.
_fold_pos_counts = draw_sensitivity_wide.groupby(["Seed", "Fold"])["delta"].apply(lambda s: int((s > 0).sum()))
_fold_neg_counts = draw_sensitivity_wide.groupby(["Seed", "Fold"])["delta"].apply(lambda s: int((s < 0).sum()))
_n_folds_total = len(_fold_pos_counts)
draw_sensitivity_n_folds_unanimous = int((
    (_fold_pos_counts == DRAW_SENSITIVITY_REPS) | (_fold_neg_counts == DRAW_SENSITIVITY_REPS)
).sum())
# Check mean-delta sign consistency across all redraws.
draw_sensitivity_headline_mean_delta_per_rep = draw_sensitivity_wide.groupby("Draw_Rep")["delta"].mean()
_headline_signs = np.sign(draw_sensitivity_headline_mean_delta_per_rep.to_numpy())
_headline_signs_nonzero = _headline_signs[_headline_signs != 0]
draw_sensitivity_headline_sign_stable = bool(
    len(_headline_signs_nonzero) == DRAW_SENSITIVITY_REPS and len(set(_headline_signs_nonzero.tolist())) == 1
)
draw_sensitivity_headline_mean_delta = float(draw_sensitivity_headline_mean_delta_per_rep.mean())
draw_sensitivity_headline_min_delta = float(draw_sensitivity_headline_mean_delta_per_rep.min())
draw_sensitivity_headline_max_delta = float(draw_sensitivity_headline_mean_delta_per_rep.max())

# Apply the decision rule defined before inspecting redraw results.
if draw_sensitivity_var_ratio < 0.25 and draw_sensitivity_headline_sign_stable:
    draw_sensitivity_decision = "KEEP current independent-draw design"
    draw_sensitivity_verdict = (
        f"Var_ratio = {draw_sensitivity_var_ratio:.3f} (< 0.25) and the mean-delta sign held in all "
        f"{DRAW_SENSITIVITY_REPS} replicates. Synthetic-draw noise is small relative to fold noise "
        f"(SD_draw = {draw_sensitivity_sd_draw:.4f} vs SD_fold = {draw_sensitivity_sd_fold:.4f}); a "
        "shared fold-specific pool would cut the B-vs-C standard error by at most "
        f"{draw_sensitivity_max_se_reduction_pct:.1f}% (less in practice, since only half of B's "
        "synthetic rows would be shared with C). The current design is retained."
    )
elif draw_sensitivity_var_ratio > 0.50 or not draw_sensitivity_headline_sign_stable:
    draw_sensitivity_decision = "SWITCH to a shared fold-specific synthetic pool"
    draw_sensitivity_verdict = (
        f"Var_ratio = {draw_sensitivity_var_ratio:.3f} "
        f"({'> 0.50' if draw_sensitivity_var_ratio > 0.50 else 'sign-unstable'}); the mean-delta sign "
        f"{'held' if draw_sensitivity_headline_sign_stable else 'FLIPPED'} across the "
        f"{DRAW_SENSITIVITY_REPS} replicates. Synthetic-draw randomness drives a large share of the "
        "B-vs-C gap; a shared fold-specific synthetic pool (deterministic, class-balanced, fixed seed) "
        "is warranted before the gap is interpreted."
    )
else:
    draw_sensitivity_decision = "KEEP current design; report residual uncertainty"
    draw_sensitivity_verdict = (
        f"Var_ratio = {draw_sensitivity_var_ratio:.3f} sits between 0.25 and 0.50 (headline sign "
        f"{'stable' if draw_sensitivity_headline_sign_stable else 'unstable'}). The current design is "
        f"kept, but synthetic-draw noise (SD_draw = {draw_sensitivity_sd_draw:.4f}) is a non-trivial "
        "share of delta variance and is reported as an explicit limitation."
    )

draw_sensitivity_summary = pd.DataFrame([{
    "Reps": DRAW_SENSITIVITY_REPS,
    "Folds": _n_folds_total,
    "SD_draw": draw_sensitivity_sd_draw,
    "SD_fold": draw_sensitivity_sd_fold,
    "Var_Ratio_Draw": draw_sensitivity_var_ratio,
    "Max_SE_Reduction_Pct": draw_sensitivity_max_se_reduction_pct,
    "Mean_Delta": draw_sensitivity_headline_mean_delta,
    "Mean_Delta_Min_Across_Reps": draw_sensitivity_headline_min_delta,
    "Mean_Delta_Max_Across_Reps": draw_sensitivity_headline_max_delta,
    "Headline_Sign_Stable": draw_sensitivity_headline_sign_stable,
    "Folds_Sign_Unanimous": draw_sensitivity_n_folds_unanimous,
    "Decision": draw_sensitivity_decision,
}])

print(
    "\nSynthetic-draw sensitivity (10 redraws x 5 seeds x 5 folds, development-only):\n"
    f"  SD_draw = {draw_sensitivity_sd_draw:.4f}, SD_fold = {draw_sensitivity_sd_fold:.4f}, "
    f"Var_ratio = {draw_sensitivity_var_ratio:.3f}\n"
    f"  Mean delta (C - B) = {draw_sensitivity_headline_mean_delta:+.4f} "
    f"(range across reps {draw_sensitivity_headline_min_delta:+.4f} .. {draw_sensitivity_headline_max_delta:+.4f}); "
    f"headline sign stable: {draw_sensitivity_headline_sign_stable}\n"
    f"  Folds with unanimous delta sign: {draw_sensitivity_n_folds_unanimous}/{_n_folds_total}\n"
    f"  Max shared-pool SE reduction: {draw_sensitivity_max_se_reduction_pct:.1f}% (upper bound)\n"
    f"  DECISION: {draw_sensitivity_decision}\n"
    f"  {draw_sensitivity_verdict}"
)

validation_report_appendix_sections.append(dataframe_to_report_section(
    "Synthetic-draw sensitivity",
    draw_sensitivity_summary,
    "Development-only test of whether random synthetic draws explain the Macro-F1 gap between B and C. "
    "Real rows and folds were fixed. Decision: "
    f"{draw_sensitivity_decision}.",
    section_id="draw-sensitivity-diagnostic",
))

validation_report_appendix_sections.append(
    dataframe_to_report_section(
        "Synthetic-draw sensitivity by fold",
        draw_sensitivity_wide,
        "C minus B Macro-F1 differences for each fold across repeated synthetic draws.",
        collapsible=True,
        section_id="draw-sensitivity-fold-level-results",
    )
)

Verification gate passed: synthetic_seed_offset=0 reproduces synthetic_experiment_summary exactly.

Synthetic-draw sensitivity (10 redraws x 5 seeds x 5 folds, development-only):
  SD_draw = 0.0213, SD_fold = 0.0281, Var_ratio = 0.364
  Mean delta (C - B) = +0.0084 (range across reps -0.0005 .. +0.0171); headline sign stable: False
  Folds with unanimous delta sign: 2/25
  Max shared-pool SE reduction: 20.2% (upper bound)
  DECISION: SWITCH to a shared fold-specific synthetic pool
  Var_ratio = 0.364 (sign-unstable); the mean-delta sign FLIPPED across the 10 replicates. Synthetic-draw randomness drives a large share of the B-vs-C gap; a shared fold-specific synthetic pool (deterministic, class-balanced, fixed seed) is warranted before the gap is interpreted.


## 26. Final Holdout and Coefficient Interpretation

Locked final Logistic Regression fit, holdout metrics, confusion matrix, ROC curve, predictions, coefficient table, and final single-file HTML report write.

In [31]:
locked_final_model = build_locked_final_pipeline(RANDOM_STATE)
with catch_convergence_warning() as locked_final_convergence_state:
    locked_final_model.fit(X_train_final.to_numpy(float), y_train_final)
lr_convergence_audit_rows.append(build_convergence_record(
    stage="Final deployed model fit (all development rows)",
    seed=RANDOM_STATE,
    outer_fold=None,
    scenario="Final deployed model (A, all development rows)",
    estimator=locked_final_model,
    convergence_triggered=locked_final_convergence_state["triggered"],
))
final_y_pred = locked_final_model.predict(X_test_final.to_numpy(float))
final_y_prob = continuous_model_score(locked_final_model, X_test_final.to_numpy(float))
final_metrics = pd.DataFrame([
    metric_summary(y_test_final, final_y_pred, final_y_prob, FINAL_MODEL_NAME, "final_test_20pct_groups")
])
final_metrics["Best_Params"] = str({"clf__C": FINAL_MODEL_C})
final_predictions = meta_test_final[[*CONTEXT_COLS, *LEAF_ID_COLS, "Split"]].copy()
final_predictions["Model"] = FINAL_MODEL_NAME
final_predictions["y_true"] = y_test_final
final_predictions["y_pred"] = final_y_pred
final_predictions["Tipburn_Score"] = final_y_prob
final_predictions["Best_Params"] = str({"clf__C": FINAL_MODEL_C})

# Build the final-holdout misclassification table for image review.
holdout_original_model_misclassification = final_predictions.loc[
    final_predictions["y_true"].ne(final_predictions["y_pred"])
].copy()

holdout_original_model_misclassification["True_Label"] = (
    holdout_original_model_misclassification["y_true"]
    .map(label_name_map)
)

holdout_original_model_misclassification["Predicted_Label"] = (
    holdout_original_model_misclassification["y_pred"]
    .map(label_name_map)
)

holdout_original_model_misclassification["Error_Type"] = np.select(
    [
        (
            holdout_original_model_misclassification["y_true"].eq(0)
            & holdout_original_model_misclassification["y_pred"].eq(1)
        ),
        (
            holdout_original_model_misclassification["y_true"].eq(1)
            & holdout_original_model_misclassification["y_pred"].eq(0)
        ),
    ],
    [
        "False Positive",
        "False Negative",
    ],
    default="Unexpected",
)

# Keep identifiers needed to trace each error to its source leaf and image.
holdout_original_model_misclassification = (
    holdout_original_model_misclassification
    .loc[:, [
        "Dataset_ID",
        "Sample_Group_ID",
        "Lettuce_Sample_ID",
        "Leaf_ID",
        "Label",
        "Week",
        "DAT",
        "True_Label",
        "Predicted_Label",
        "Error_Type",
        "Tipburn_Score",
    ]]
    .sort_values(
        [
            "Error_Type",
            "Dataset_ID",
            "Lettuce_Sample_ID",
            "Leaf_ID",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Verify the table row count equals FP + FN.
expected_holdout_errors = int(
    final_metrics.loc[0, "FP"]
    + final_metrics.loc[0, "FN"]
)

if (
    len(holdout_original_model_misclassification)
    != expected_holdout_errors
):
    raise AssertionError(
        "Holdout misclassification row count does not match FP + FN: "
        f"{len(holdout_original_model_misclassification)} "
        f"!= {expected_holdout_errors}."
    )

print(
    "Original-model final holdout errors: "
    f"{len(holdout_original_model_misclassification)}"
)

validation_report_appendix_sections.append(
    dataframe_to_report_section(
        "Final holdout misclassifications",
        holdout_original_model_misclassification,
        "Misclassified leaves from the final model's single evaluation on the fixed holdout.",
        collapsible=True,
        section_id="original-model-final-holdout-misclassifications",
    )
)

final_report_text = classification_report(
    y_test_final,
    final_y_pred,
    labels=[0, 1],
    target_names=["Healthy", "Tipburn"],
    zero_division=0,
)
final_report_df = pd.DataFrame({"Classification_Report": final_report_text.splitlines()})
validation_report_sections.append(dataframe_to_report_section(
    "Final holdout results", 
    final_metrics, 
    "Final five-feature Logistic Regression trained on all development data and evaluated once on the fixed plant-level holdout. The holdout was not used for tuning.",
    section_id="final-holdout-results"
    ))
fig, ax = plt.subplots(figsize=(5.8, 4.8))
cm = np.array([[final_metrics.loc[0, "TN"], final_metrics.loc[0, "FP"]], [final_metrics.loc[0, "FN"], final_metrics.loc[0, "TP"]]])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=["Healthy", "Tipburn"], yticklabels=["Healthy", "Tipburn"], ax=ax)
ax.set_title(f"Final holdout confusion matrix\nF1={final_metrics.loc[0, 'F1_macro']:.3f}, AUC={final_metrics.loc[0, 'ROC_AUC']:.3f}")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.tight_layout()
validation_report_sections.append(figure_to_base64_html(fig, "Final holdout confusion matrix", "Embedded confusion-matrix figure for the locked final model.", section_id="final-holdout-confusion-matrix"))

coefficient_values = locked_final_model.named_steps["clf"].coef_[0]
coefficient_table = pd.DataFrame({
    "Feature": FINAL_FEATURE_COLS,
    "Scaled_Logistic_Coefficient": coefficient_values,
})
coefficient_table["Absolute_Coefficient"] = coefficient_table["Scaled_Logistic_Coefficient"].abs()
coefficient_table["Direction"] = np.where(coefficient_table["Scaled_Logistic_Coefficient"] >= 0, "Increases Tipburn log-odds", "Decreases Tipburn log-odds")
coefficient_table = coefficient_table.sort_values("Absolute_Coefficient", ascending=False).reset_index(drop=True)
validation_report_sections.append(dataframe_to_report_section(
    "Five-feature Logistic Regression coefficient interpretation",
    coefficient_table,
    "",
    section_id="five-feature-logistic-regression-coefficient-interpretation",
))


# Compare frozen A/B/C models on the same untouched holdout.
FINAL_EXPECTED_DEVELOPMENT_LEAVES = 288
FINAL_EXPECTED_HOLDOUT_LEAVES = 72
FINAL_EXPECTED_HOLDOUT_PLANTS = 24

if len(X_train_final) != FINAL_EXPECTED_DEVELOPMENT_LEAVES:
    raise AssertionError(f"Expected 288 development leaves, got {len(X_train_final)}.")
if len(X_test_final) != FINAL_EXPECTED_HOLDOUT_LEAVES:
    raise AssertionError(f"Expected 72 holdout leaves, got {len(X_test_final)}.")
if pd.Series(groups_test_final).nunique() != FINAL_EXPECTED_HOLDOUT_PLANTS:
    raise AssertionError("Expected 24 genuine holdout plants.")
if set(meta_train_final["Sample_Group_ID"].astype(str)) != set(development_groups):
    raise AssertionError("Development plant membership changed before frozen holdout comparison.")
if set(meta_test_final["Sample_Group_ID"].astype(str)) != set(holdout_groups):
    raise AssertionError("Holdout plant membership changed before frozen holdout comparison.")
if set(development_groups).intersection(holdout_groups):
    raise AssertionError("Development and holdout plant sets overlap.")
final_holdout_order_cols = ["Dataset_ID", "Sample_Group_ID", "Lettuce_Sample_ID", "Leaf_ID", "Week", "DAT", "Label"]
expected_holdout_order = features_df.loc[test_mask, final_holdout_order_cols].reset_index(drop=True)
actual_holdout_order = meta_test_final.loc[:, final_holdout_order_cols].reset_index(drop=True)
if not expected_holdout_order.astype(str).equals(actual_holdout_order.astype(str)):
    raise AssertionError("Holdout row order changed before frozen holdout comparison.")
if hasattr(meta_test_final, "Is_Synthetic") and meta_test_final["Is_Synthetic"].any():
    raise AssertionError("Holdout contains synthetic rows.")
if not np.array_equal((final_y_prob >= FINAL_DECISION_THRESHOLD).astype(int), final_y_pred):
    raise AssertionError("Final A threshold-0.5 predictions do not match model predictions.")

leaf_train_final_for_holdout = leaf_df.loc[train_mask].reset_index(drop=True)
final_residual_pool = make_repeated_scan_residuals(raw_df, wavelength_cols).loc[
    lambda frame: frame["Sample_Group_ID"].astype(str).isin(development_groups)
].reset_index(drop=True)
if set(leaf_train_final_for_holdout["Sample_Group_ID"].astype(str)).intersection(holdout_groups):
    raise AssertionError("Holdout plants are present in final synthetic source spectra / acceptance-limit references.")
if set(final_residual_pool["Sample_Group_ID"].astype(str)).intersection(holdout_groups):
    raise AssertionError("Holdout plants are present in final synthetic residual sources.")

final_synthetic_generation_frames: list[pd.DataFrame] = []
final_synthetic_provenance_frames: list[pd.DataFrame] = []
final_synthetic_rejection_frames: list[pd.DataFrame] = []
final_holdout_prediction_frames: list[pd.DataFrame] = []
final_holdout_metric_rows: list[dict[str, Any]] = []
final_holdout_confusion_rows: list[dict[str, Any]] = []

FINAL_C_REAL_TARGET_PER_LABEL = {0: 72, 1: 72}
FINAL_C_SYNTHETIC_TARGET_PER_LABEL = {0: 72, 1: 72}
FINAL_C_EXPECTED_REAL_ROWS = sum(FINAL_C_REAL_TARGET_PER_LABEL.values())
FINAL_C_EXPECTED_SYNTHETIC_ROWS = sum(FINAL_C_SYNTHETIC_TARGET_PER_LABEL.values())
FINAL_C_EXPECTED_TOTAL_ROWS = FINAL_C_EXPECTED_REAL_ROWS + FINAL_C_EXPECTED_SYNTHETIC_ROWS

final_synthetic_design_targets = pd.DataFrame([
    {"Scenario": "A", "Real_Healthy": 144, "Real_Tipburn": 144, "Synthetic_Healthy": 0, "Synthetic_Tipburn": 0, "Total": 288},
    {"Scenario": "B", "Real_Healthy": 0, "Real_Tipburn": 0, "Synthetic_Healthy": 144, "Synthetic_Tipburn": 144, "Total": 288},
    {"Scenario": "C", "Real_Healthy": 72, "Real_Tipburn": 72, "Synthetic_Healthy": 72, "Synthetic_Tipburn": 72, "Total": 288},
])


# Reuse the group-preserving balanced selector for the fixed C holdout design.


def append_final_holdout_predictions(experiment_name: str,
                                     y_pred: np.ndarray,
                                     y_prob: np.ndarray) -> None:
    pred_table = meta_test_final[["Sample_Group_ID", "Lettuce_Sample_ID", "Leaf_ID", "Week", "DAT"]].copy()
    pred_table.insert(0, "Experiment", experiment_name)
    pred_table["True_Label"] = [label_name_map[int(value)] for value in y_test_final]
    pred_table["Predicted_Label"] = [label_name_map[int(value)] for value in y_pred]
    pred_table["Tipburn_Probability"] = y_prob
    pred_table["Is_Synthetic"] = False
    if len(pred_table) != FINAL_EXPECTED_HOLDOUT_LEAVES:
        raise AssertionError(f"{experiment_name} does not have exactly 72 holdout predictions.")
    final_holdout_prediction_frames.append(pred_table)


def evaluate_final_holdout_experiment(experiment_name: str,
                                      model,
                                      real_training_rows: int,
                                      synthetic_training_rows: int,
                                      generation_summary: pd.DataFrame | None = None,
                                      provenance: pd.DataFrame | None = None,
                                      rejections: pd.DataFrame | None = None) -> None:
    y_prob = continuous_model_score(model, X_test_final.to_numpy(float))
    y_pred = (y_prob >= FINAL_DECISION_THRESHOLD).astype(int)
    if not np.array_equal(y_pred, model.predict(X_test_final.to_numpy(float))):
        raise AssertionError(f"{experiment_name} threshold-0.5 predictions differ from model.predict().")
    metrics = binary_classification_metrics(
        y_test_final,
        y_pred,
        y_prob,
        "final_test_20pct_groups",
        FINAL_MODEL_NAME,
    )
    metrics.update({
        "Experiment": experiment_name,
        "Real_Training_Rows": int(real_training_rows),
        "Synthetic_Training_Rows": int(synthetic_training_rows),
        "Total_Training_Rows": int(real_training_rows + synthetic_training_rows),
        "Decision_Threshold": FINAL_DECISION_THRESHOLD,
    })
    final_holdout_metric_rows.append(metrics)
    append_final_holdout_predictions(experiment_name, y_pred, y_prob)
    final_holdout_confusion_rows.append({
        "Experiment": experiment_name,
        "TN": metrics["TN"],
        "FP": metrics["FP"],
        "FN": metrics["FN"],
        "TP": metrics["TP"],
    })
    if provenance is not None and not provenance.empty:
        source_groups = set(provenance["Source_Plant_A"].astype(str)).union(set(provenance["Source_Plant_B"].astype(str)))
        residual_groups = set(provenance["Residual_Source_Plant"].astype(str))
        if source_groups.intersection(holdout_groups) or residual_groups.intersection(holdout_groups):
            raise AssertionError(f"{experiment_name} synthetic provenance used holdout plants.")
    if generation_summary is not None and not generation_summary.empty:
        summary = generation_summary.copy()
        summary.insert(0, "Experiment", experiment_name)
        final_synthetic_generation_frames.append(summary)
    if provenance is not None and not provenance.empty:
        prov = provenance.copy()
        prov.insert(0, "Experiment", experiment_name)
        final_synthetic_provenance_frames.append(prov)
    if rejections is not None and not rejections.empty:
        rej = rejections.copy()
        rej.insert(0, "Experiment", experiment_name)
        final_synthetic_rejection_frames.append(rej)


# Evaluate the original-only model before synthetic holdout scenarios.
evaluate_final_holdout_experiment(
    experiment_name="Final A: Original only",
    model=locked_final_model,
    real_training_rows=len(X_train_final),
    synthetic_training_rows=0,
)


final_training_design_audit_frames: list[pd.DataFrame] = []


def build_final_synthetic_model(experiment_name: str,
                                requested_synthetic_count: int,
                                random_seed: int,
                                include_real_development: bool,
                                real_feature_subset: pd.DataFrame | None = None,
                                real_target_subset: np.ndarray | None = None,
                                real_meta_subset: pd.DataFrame | None = None,
                                expected_design: dict[str, int] | None = None) -> tuple[Any, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    allocation_table = allocate_synthetic_counts_by_label(
        meta_train_final if real_meta_subset is None else real_meta_subset,
        leaf_train_final_for_holdout,
        requested_synthetic_count,
    )
    generation_result = generate_synthetic_leaf_spectra(
        leaf_pool=leaf_train_final_for_holdout,
        residual_pool=final_residual_pool,
        wavelength_cols=wavelength_cols,
        allocation_table=allocation_table,
        rng=np.random.default_rng(random_seed),
        source_tag=experiment_name,
        random_seed=random_seed,
    )
    synthetic_spectra = generation_result["accepted_spectra"]
    synthetic_features = preprocess_synthetic_fold_spectra(synthetic_spectra, wavelength_cols, wavelengths)
    generation_summary = generation_result["generation_summary"]
    provenance = generation_result["provenance"]
    rejections = generation_result["rejections"]
    if int(generation_summary["Requested_Count"].sum()) != requested_synthetic_count:
        raise AssertionError(f"{experiment_name} requested count was not preserved.")
    if int(generation_summary["Accepted_Count"].sum()) != requested_synthetic_count:
        raise AssertionError(f"{experiment_name} accepted count did not reach the requested total.")
    if len(synthetic_features) != requested_synthetic_count:
        raise AssertionError(f"{experiment_name} synthetic feature count does not match request.")
    if set(provenance["Source_Plant_A"].astype(str)).intersection(holdout_groups):
        raise AssertionError(f"{experiment_name} Source Plant A includes holdout plants.")
    if set(provenance["Source_Plant_B"].astype(str)).intersection(holdout_groups):
        raise AssertionError(f"{experiment_name} Source Plant B includes holdout plants.")
    if set(provenance["Residual_Source_Plant"].astype(str)).intersection(holdout_groups):
        raise AssertionError(f"{experiment_name} residual sources include holdout plants.")

    X_parts: list[np.ndarray] = []
    y_parts: list[np.ndarray] = []
    if include_real_development:
        real_X = X_train_final if real_feature_subset is None else real_feature_subset
        real_y = y_train_final if real_target_subset is None else np.asarray(real_target_subset, dtype=int)
        real_meta = meta_train_final if real_meta_subset is None else real_meta_subset.reset_index(drop=True).copy()
        if len(real_X) != len(real_y) or len(real_meta) != len(real_y):
            raise AssertionError(f"{experiment_name} real feature, target, and metadata rows are not aligned.")
        if real_X.columns.tolist() != FINAL_FEATURE_COLS:
            raise AssertionError("Locked final feature order changed.")
        if set(real_meta["Sample_Group_ID"].astype(str)).intersection(holdout_groups):
            raise AssertionError(f"{experiment_name} real training subset contains holdout plants.")
        X_parts.append(real_X.loc[:, FINAL_FEATURE_COLS].to_numpy(float))
        y_parts.append(real_y)
    else:
        real_y = np.array([], dtype=int)
        real_meta = pd.DataFrame(columns=meta_train_final.columns)
    X_parts.append(synthetic_features.loc[:, FINAL_FEATURE_COLS].to_numpy(float))
    y_parts.append(synthetic_features["y"].to_numpy(int))
    X_fit = np.vstack(X_parts)
    y_fit = np.concatenate(y_parts)
    if X_train_final.columns.tolist() != FINAL_FEATURE_COLS:
        raise AssertionError("Locked final feature order changed.")
    provenance_fields = set(provenance.columns)
    if provenance_fields.intersection(FINAL_FEATURE_COLS) - set(FINAL_FEATURE_COLS):
        raise AssertionError("Unexpected provenance field entered FINAL_FEATURE_COLS.")

    real_count = int(len(real_y))
    synthetic_count = int(len(synthetic_features))
    real_label_counts = pd.Series(real_y).value_counts().to_dict()
    synthetic_label_counts = synthetic_features["y"].astype(int).value_counts().to_dict()
    real_healthy = int(real_label_counts.get(0, 0))
    real_tipburn = int(real_label_counts.get(1, 0))
    synthetic_healthy = int(synthetic_label_counts.get(0, 0))
    synthetic_tipburn = int(synthetic_label_counts.get(1, 0))
    holdout_rows_used_for_training = int(real_meta["Sample_Group_ID"].astype(str).isin(holdout_groups).sum()) if not real_meta.empty else 0
    holdout_rows_used_for_training += int(synthetic_features["Sample_Group_ID"].astype(str).isin(holdout_groups).sum())
    training_design_audit = pd.DataFrame([
        {
            "Experiment": experiment_name,
            "Source_Type": "Real",
            "Healthy_Rows": real_healthy,
            "Tipburn_Rows": real_tipburn,
            "Total_Rows": real_count,
            "Holdout_Rows_Used_For_Training": holdout_rows_used_for_training if synthetic_count == 0 else int(real_meta["Sample_Group_ID"].astype(str).isin(holdout_groups).sum()) if not real_meta.empty else 0,
        },
        {
            "Experiment": experiment_name,
            "Source_Type": "Synthetic",
            "Healthy_Rows": synthetic_healthy,
            "Tipburn_Rows": synthetic_tipburn,
            "Total_Rows": synthetic_count,
            "Holdout_Rows_Used_For_Training": int(synthetic_features["Sample_Group_ID"].astype(str).isin(holdout_groups).sum()),
        },
        {
            "Experiment": experiment_name,
            "Source_Type": "Combined",
            "Healthy_Rows": real_healthy + synthetic_healthy,
            "Tipburn_Rows": real_tipburn + synthetic_tipburn,
            "Total_Rows": int(len(X_fit)),
            "Holdout_Rows_Used_For_Training": holdout_rows_used_for_training,
        },
    ])
    final_training_design_audit_frames.append(training_design_audit)
    if expected_design is not None:
        X_final_c = X_fit
        assert len(X_final_c) == expected_design["total_rows"]
        assert real_count == expected_design["real_total"]
        assert synthetic_count == expected_design["synthetic_total"]
        assert real_healthy == expected_design["real_healthy"]
        assert real_tipburn == expected_design["real_tipburn"]
        assert synthetic_healthy == expected_design["synthetic_healthy"]
        assert synthetic_tipburn == expected_design["synthetic_tipburn"]
        assert holdout_rows_used_for_training == 0
    model = build_locked_final_pipeline(RANDOM_STATE)
    with catch_convergence_warning() as final_synthetic_convergence_state:
        model.fit(X_fit, y_fit)
    lr_convergence_audit_rows.append(build_convergence_record(
        stage="Final holdout scenario fit (single fixed-seed draw)",
        seed=random_seed,
        outer_fold=None,
        scenario=experiment_name,
        estimator=model,
        convergence_triggered=final_synthetic_convergence_state["triggered"],
    ))
    return model, generation_summary, provenance, rejections, synthetic_features


final_b_requested_synthetic = len(X_train_final)
final_b_model, final_b_generation, final_b_provenance, final_b_rejections, final_b_features = build_final_synthetic_model(
    experiment_name="Final B: Synthetic only",
    requested_synthetic_count=final_b_requested_synthetic,
    random_seed=FINAL_SYNTHETIC_ONLY_SEED,
    include_real_development=False,
)
evaluate_final_holdout_experiment(
    experiment_name="Final B: Synthetic only",
    model=final_b_model,
    real_training_rows=0,
    synthetic_training_rows=len(final_b_features),
    generation_summary=final_b_generation,
    provenance=final_b_provenance,
    rejections=final_b_rejections,
)

final_c_real_subset = select_balanced_real_subset(
    feature_frame=X_train_final,
    target_values=y_train_final,
    meta_frame=meta_train_final,
    target_per_label=FINAL_C_REAL_TARGET_PER_LABEL,
    random_seed=FINAL_COMBINED_SEED,
    group_col="Sample_Group_ID",
    forbidden_groups=holdout_groups,
)
final_c_real_subset_audit = final_c_real_subset["audit"]
final_c_real_subset_metadata = final_c_real_subset["meta"].copy()
final_c_requested_synthetic = FINAL_C_EXPECTED_SYNTHETIC_ROWS

if len(final_c_real_subset["X"]) != FINAL_C_EXPECTED_REAL_ROWS:
    raise AssertionError("Final C real subset must contain exactly 144 real development leaves.")
if int(np.sum(final_c_real_subset["y"] == 0)) != FINAL_C_REAL_TARGET_PER_LABEL[0]:
    raise AssertionError("Final C real subset must contain exactly 72 real Healthy leaves.")
if int(np.sum(final_c_real_subset["y"] == 1)) != FINAL_C_REAL_TARGET_PER_LABEL[1]:
    raise AssertionError("Final C real subset must contain exactly 72 real Tipburn leaves.")
if set(final_c_real_subset_metadata["Sample_Group_ID"].astype(str)).intersection(holdout_groups):
    raise AssertionError("Final C real subset contains holdout plants.")

final_c_model, final_c_generation, final_c_provenance, final_c_rejections, final_c_features = build_final_synthetic_model(
    experiment_name="Final C_Mixed_Fixed_50_50",
    requested_synthetic_count=final_c_requested_synthetic,
    random_seed=FINAL_COMBINED_SEED,
    include_real_development=True,
    real_feature_subset=final_c_real_subset["X"],
    real_target_subset=final_c_real_subset["y"],
    real_meta_subset=final_c_real_subset_metadata,
    expected_design={
        "total_rows": FINAL_C_EXPECTED_TOTAL_ROWS,
        "real_total": FINAL_C_EXPECTED_REAL_ROWS,
        "synthetic_total": FINAL_C_EXPECTED_SYNTHETIC_ROWS,
        "real_healthy": FINAL_C_REAL_TARGET_PER_LABEL[0],
        "real_tipburn": FINAL_C_REAL_TARGET_PER_LABEL[1],
        "synthetic_healthy": FINAL_C_SYNTHETIC_TARGET_PER_LABEL[0],
        "synthetic_tipburn": FINAL_C_SYNTHETIC_TARGET_PER_LABEL[1],
    },
)
evaluate_final_holdout_experiment(
    experiment_name="Final C_Mixed_Fixed_50_50",
    model=final_c_model,
    real_training_rows=len(final_c_real_subset["X"]),
    synthetic_training_rows=len(final_c_features),
    generation_summary=final_c_generation,
    provenance=final_c_provenance,
    rejections=final_c_rejections,
)

# Verify fixed C has 72 real and 72 synthetic rows per class and uses no holdout data.
_final_c_training_design_audit = final_training_design_audit_frames[-1]
_final_c_combined_row = _final_c_training_design_audit.loc[
    (_final_c_training_design_audit["Experiment"] == "Final C_Mixed_Fixed_50_50")
    & (_final_c_training_design_audit["Source_Type"] == "Combined")
].iloc[0]
_final_c_holdout_rows = final_holdout_prediction_frames[-1]
final_c_mixed_fixed_holdout_audit = pd.DataFrame([{
    "Experiment_Name": "Final C_Mixed_Fixed_50_50",
    "Scenario": "C_Mixed_Fixed_50_50",
    "Real_Healthy": FINAL_C_REAL_TARGET_PER_LABEL[0],
    "Real_Tipburn": FINAL_C_REAL_TARGET_PER_LABEL[1],
    "Synthetic_Healthy": FINAL_C_SYNTHETIC_TARGET_PER_LABEL[0],
    "Synthetic_Tipburn": FINAL_C_SYNTHETIC_TARGET_PER_LABEL[1],
    "Real_Total": FINAL_C_EXPECTED_REAL_ROWS,
    "Synthetic_Total": FINAL_C_EXPECTED_SYNTHETIC_ROWS,
    "Total_Training_Rows": FINAL_C_EXPECTED_TOTAL_ROWS,
    "Final_Test_Real_Rows": FINAL_EXPECTED_HOLDOUT_LEAVES,
    "Final_Test_Synthetic_Rows": int(_final_c_holdout_rows["Is_Synthetic"].sum()),
    "Final_Holdout_Rows_Used_In_Training": int(_final_c_combined_row["Holdout_Rows_Used_For_Training"]),
    "Final_Holdout_Rows_Used_In_Generation": int(
        set(final_c_provenance["Source_Plant_A"].astype(str)).union(
            set(final_c_provenance["Source_Plant_B"].astype(str))
        ).union(
            set(final_c_provenance["Residual_Source_Plant"].astype(str))
        ).intersection(holdout_groups).__len__()
        if not final_c_provenance.empty else 0
    ),
}])
if int(final_c_mixed_fixed_holdout_audit.loc[0, "Real_Healthy"]) != 72:
    raise AssertionError("Final C_Mixed_Fixed_50_50 Real_Healthy changed from 72.")
if int(final_c_mixed_fixed_holdout_audit.loc[0, "Real_Tipburn"]) != 72:
    raise AssertionError("Final C_Mixed_Fixed_50_50 Real_Tipburn changed from 72.")
if int(final_c_mixed_fixed_holdout_audit.loc[0, "Synthetic_Healthy"]) != 72:
    raise AssertionError("Final C_Mixed_Fixed_50_50 Synthetic_Healthy changed from 72.")
if int(final_c_mixed_fixed_holdout_audit.loc[0, "Synthetic_Tipburn"]) != 72:
    raise AssertionError("Final C_Mixed_Fixed_50_50 Synthetic_Tipburn changed from 72.")
if int(final_c_mixed_fixed_holdout_audit.loc[0, "Real_Total"]) != 144:
    raise AssertionError("Final C_Mixed_Fixed_50_50 Real_Total changed from 144.")
if int(final_c_mixed_fixed_holdout_audit.loc[0, "Synthetic_Total"]) != 144:
    raise AssertionError("Final C_Mixed_Fixed_50_50 Synthetic_Total changed from 144.")
if int(final_c_mixed_fixed_holdout_audit.loc[0, "Total_Training_Rows"]) != 288:
    raise AssertionError("Final C_Mixed_Fixed_50_50 Total_Training_Rows changed from 288.")
if int(final_c_mixed_fixed_holdout_audit.loc[0, "Final_Test_Real_Rows"]) != 72:
    raise AssertionError("Final C_Mixed_Fixed_50_50 Final_Test_Real_Rows changed from 72.")
if int(final_c_mixed_fixed_holdout_audit.loc[0, "Final_Test_Synthetic_Rows"]) != 0:
    raise AssertionError("Final C_Mixed_Fixed_50_50 holdout contains synthetic rows.")
if int(final_c_mixed_fixed_holdout_audit.loc[0, "Final_Holdout_Rows_Used_In_Training"]) != 0:
    raise AssertionError("Final C_Mixed_Fixed_50_50 used holdout rows for training.")
if int(final_c_mixed_fixed_holdout_audit.loc[0, "Final_Holdout_Rows_Used_In_Generation"]) != 0:
    raise AssertionError("Final C_Mixed_Fixed_50_50 used holdout rows for synthetic generation.")

final_synthetic_holdout_metrics = pd.DataFrame(final_holdout_metric_rows)
metric_cols = [
    "Experiment", "Real_Training_Rows", "Synthetic_Training_Rows", "Total_Training_Rows",
    "Macro_F1", "ROC_AUC", "Precision_Tipburn", "Recall_Tipburn", "Balanced_Accuracy",
    "TN", "FP", "FN", "TP", "Decision_Threshold",
]
final_synthetic_holdout_metrics = final_synthetic_holdout_metrics.loc[:, metric_cols]
final_combined_holdout_predictions = pd.concat(final_holdout_prediction_frames, ignore_index=True)
if not final_combined_holdout_predictions.groupby("Experiment").size().eq(FINAL_EXPECTED_HOLDOUT_LEAVES).all():
    raise AssertionError("Every final experiment must contain exactly 72 holdout predictions.")
if final_combined_holdout_predictions["Is_Synthetic"].any():
    raise AssertionError("Synthetic rows appeared in final holdout prediction table.")

final_synthetic_generation_summary = (
    pd.concat(final_synthetic_generation_frames, ignore_index=True)
    if final_synthetic_generation_frames else pd.DataFrame()
)
final_synthetic_generation_totals = (
    final_synthetic_generation_summary
    .groupby("Experiment", as_index=False)
    .agg(
        Requested_Count=("Requested_Count", "sum"),
        Accepted_Count=("Accepted_Count", "sum"),
        Rejected_Count=("Rejected_Count", "sum"),
        Attempt_Count=("Attempt_Count", "sum"),
    )
) if not final_synthetic_generation_summary.empty else pd.DataFrame()
final_synthetic_provenance = (
    pd.concat(final_synthetic_provenance_frames, ignore_index=True)
    if final_synthetic_provenance_frames else pd.DataFrame()
)
final_synthetic_rejections = (
    pd.concat(final_synthetic_rejection_frames, ignore_index=True)
    if final_synthetic_rejection_frames else pd.DataFrame()
)
final_training_design_audit = (
    pd.concat(final_training_design_audit_frames, ignore_index=True)
    if final_training_design_audit_frames else pd.DataFrame()
)

final_a_training_design_rows = pd.DataFrame([
    {"Experiment": "Final A: Original only", "Source_Type": "Real", "Healthy_Rows": 144, "Tipburn_Rows": 144, "Total_Rows": 288, "Holdout_Rows_Used_For_Training": 0},
    {"Experiment": "Final A: Original only", "Source_Type": "Synthetic", "Healthy_Rows": 0, "Tipburn_Rows": 0, "Total_Rows": 0, "Holdout_Rows_Used_For_Training": 0},
    {"Experiment": "Final A: Original only", "Source_Type": "Combined", "Healthy_Rows": 144, "Tipburn_Rows": 144, "Total_Rows": 288, "Holdout_Rows_Used_For_Training": 0},
])
training_composition_audit = pd.concat(
    [final_a_training_design_rows, final_training_design_audit], ignore_index=True
)


report_path = OUTPUT_DIR / "tipburn_model_validation_report.html"

Original-model final holdout errors: 11


## 27. Final Augmentation, Convergence Audit, and HTML Report

This section evaluates two supplementary augmentation-size variants, checks Logistic Regression convergence, and compiles the retained validation results into the standalone HTML report. The locked model, feature set, threshold, synthetic settings and fixed holdout remain unchanged.

In [32]:
# Evaluate C1/C2 separately as the full real development set plus added synthetic data.
# Reuse the audited synthetic-model builder and retain its leakage and threshold checks.
# Read the primary A/B/C results for comparison without modifying their stored outputs.

FINAL_C1_AUGMENTATION_SEED = 4203
FINAL_C2_AUGMENTATION_SEED = 4204

augmentation_final_holdout_metric_rows: list[dict[str, Any]] = []
augmentation_final_holdout_prediction_frames: list[pd.DataFrame] = []


def evaluate_augmentation_final_holdout_experiment(experiment_name: str,
                                             model,
                                             real_training_rows: int,
                                             synthetic_training_rows: int) -> None:
    y_prob = continuous_model_score(model, X_test_final.to_numpy(float))
    y_pred = (y_prob >= FINAL_DECISION_THRESHOLD).astype(int)
    if not np.array_equal(y_pred, model.predict(X_test_final.to_numpy(float))):
        raise AssertionError(f"{experiment_name} threshold-0.5 predictions differ from model.predict().")
    metrics = binary_classification_metrics(
        y_test_final,
        y_pred,
        y_prob,
        "final_test_20pct_groups_augmentation_c1_c2",
        FINAL_MODEL_NAME,
    )
    metrics.update({
        "Experiment": experiment_name,
        "Real_Training_Rows": int(real_training_rows),
        "Synthetic_Training_Rows": int(synthetic_training_rows),
        "Total_Training_Rows": int(real_training_rows + synthetic_training_rows),
        "Decision_Threshold": FINAL_DECISION_THRESHOLD,
    })
    augmentation_final_holdout_metric_rows.append(metrics)

    pred_table = meta_test_final[["Sample_Group_ID", "Lettuce_Sample_ID", "Leaf_ID", "Week", "DAT"]].copy()
    pred_table.insert(0, "Experiment", experiment_name)
    pred_table["True_Label"] = [label_name_map[int(value)] for value in y_test_final]
    pred_table["Predicted_Label"] = [label_name_map[int(value)] for value in y_pred]
    pred_table["Tipburn_Probability"] = y_prob
    if len(pred_table) != FINAL_EXPECTED_HOLDOUT_LEAVES:
        raise AssertionError(f"{experiment_name} does not have exactly 72 holdout predictions.")
    augmentation_final_holdout_prediction_frames.append(pred_table)


augmentation_c1_model, _, _, _, augmentation_c1_features = build_final_synthetic_model(
    experiment_name="C1: Augmented +50% synthetic",
    requested_synthetic_count=len(X_train_final) // 2,
    random_seed=FINAL_C1_AUGMENTATION_SEED,
    include_real_development=True,
)
evaluate_augmentation_final_holdout_experiment(
    experiment_name="C1: Augmented +50% synthetic (288 real + 144 synthetic)",
    model=augmentation_c1_model,
    real_training_rows=len(X_train_final),
    synthetic_training_rows=len(augmentation_c1_features),
)

augmentation_c2_model, _, _, _, augmentation_c2_features = build_final_synthetic_model(
    experiment_name="C2: Augmented +100% synthetic",
    requested_synthetic_count=len(X_train_final),
    random_seed=FINAL_C2_AUGMENTATION_SEED,
    include_real_development=True,
)
evaluate_augmentation_final_holdout_experiment(
    experiment_name="C2: Augmented +100% synthetic (288 real + 288 synthetic)",
    model=augmentation_c2_model,
    real_training_rows=len(X_train_final),
    synthetic_training_rows=len(augmentation_c2_features),
)

augmentation_final_holdout_metrics = pd.DataFrame(augmentation_final_holdout_metric_rows)

# Plot final-holdout confusion matrices for A/B/C/C1/C2.
_final_conf_rows = list(final_holdout_confusion_rows) + [
    {"Experiment": _r["Experiment"], "TN": int(_r["TN"]), "FP": int(_r["FP"]), "FN": int(_r["FN"]), "TP": int(_r["TP"])}
    for _r in augmentation_final_holdout_metrics.to_dict("records")
]
fig, axes = plt.subplots(1, len(_final_conf_rows), figsize=(4.4 * len(_final_conf_rows), 3.8))
axes = np.atleast_1d(axes)
for ax, row in zip(axes, _final_conf_rows):
    cm_values = np.array([[row["TN"], row["FP"]], [row["FN"], row["TP"]]])
    sns.heatmap(cm_values, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=["Healthy", "Tipburn"], yticklabels=["Healthy", "Tipburn"], ax=ax)
    ax.set_title(str(row["Experiment"]).replace(": ", ":\n"), fontsize=9)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
plt.tight_layout()
validation_report_sections.append(figure_to_base64_html(
    fig,
    "Final holdout confusion-matrix",
    "Confusion matrices for A, B, C, C1 and C2 using the same fixed holdout leaves.",
    section_id="frozen-final-holdout-confusion-matrix-comparison",
))
augmentation_final_holdout_predictions = pd.concat(augmentation_final_holdout_prediction_frames, ignore_index=True)

# Compare leaves misclassified by at least one model on the same fixed holdout.
final_holdout_experiment_order = [
    "Final A: Original only",
    "Final B: Synthetic only",
    "Final C_Mixed_Fixed_50_50",
    "C1: Augmented +50% synthetic (288 real + 144 synthetic)",
    "C2: Augmented +100% synthetic (288 real + 288 synthetic)",
]

all_final_holdout_predictions = pd.concat(
    [
        final_combined_holdout_predictions,
        augmentation_final_holdout_predictions,
    ],
    ignore_index=True,
)

holdout_key_cols = [
    "Sample_Group_ID",
    "Lettuce_Sample_ID",
    "Leaf_ID",
]

holdout_experiment_comparison = (
    all_final_holdout_predictions[
        holdout_key_cols + ["True_Label"]
    ]
    .drop_duplicates()
    .sort_values(holdout_key_cols)
    .reset_index(drop=True)
)

if len(holdout_experiment_comparison) != FINAL_EXPECTED_HOLDOUT_LEAVES:
    raise AssertionError(
        "Expected exactly "
        f"{FINAL_EXPECTED_HOLDOUT_LEAVES} unique holdout leaves, "
        f"got {len(holdout_experiment_comparison)}."
    )

for experiment_name in final_holdout_experiment_order:
    experiment_subset = (
        all_final_holdout_predictions
        .loc[
            all_final_holdout_predictions[
                "Experiment"
            ].eq(experiment_name),
            holdout_key_cols
            + [
                "Predicted_Label",
                "Tipburn_Probability",
            ],
        ]
        .rename(
            columns={
                "Predicted_Label": (
                    f"{experiment_name} -- Predicted"
                ),
                "Tipburn_Probability": (
                    f"{experiment_name} -- Score"
                ),
            }
        )
    )

    if len(experiment_subset) != FINAL_EXPECTED_HOLDOUT_LEAVES:
        raise AssertionError(
            f"{experiment_name} does not contain exactly "
            f"{FINAL_EXPECTED_HOLDOUT_LEAVES} holdout rows."
        )

    if experiment_subset.duplicated(
        holdout_key_cols
    ).any():
        raise AssertionError(
            f"{experiment_name} contains duplicate holdout leaves."
        )

    holdout_experiment_comparison = (
        holdout_experiment_comparison
        .merge(
            experiment_subset,
            on=holdout_key_cols,
            how="left",
            validate="one_to_one",
        )
    )

prediction_cols = [
    f"{experiment_name} -- Predicted"
    for experiment_name in final_holdout_experiment_order
]

if holdout_experiment_comparison[
    prediction_cols
].isna().any().any():
    raise AssertionError(
        "Some holdout leaves are missing experiment predictions."
    )

holdout_wrong_mask = pd.concat(
    [
        holdout_experiment_comparison[
            prediction_col
        ].ne(
            holdout_experiment_comparison["True_Label"]
        )
        for prediction_col in prediction_cols
    ],
    axis=1,
).any(axis=1)

holdout_original_vs_synthetic_misclassification = (
    holdout_experiment_comparison
    .loc[holdout_wrong_mask]
    .sort_values(holdout_key_cols)
    .reset_index(drop=True)
)

print(
    f"{len(holdout_original_vs_synthetic_misclassification)} "
    f"of {len(holdout_experiment_comparison)} final holdout "
    "leaves were misclassified by at least one experiment."
)

validation_report_appendix_sections.append(
    dataframe_to_report_section(
        "Cross-experiment holdout errors",
        holdout_original_vs_synthetic_misclassification,
        "Holdout leaves misclassified by at least one model. Predictions and scores are shown together.",
        collapsible=True,
        section_id=(
            "final-holdout-leaf-level-cross-experiment-comparison"
        ),
    )
)

print("Augmentation Final C variants (C1/C2): full real training set + added synthetic.")
augmentation_final_comparison_table = pd.concat(
    [final_synthetic_holdout_metrics, augmentation_final_holdout_metrics],
    ignore_index=True,
)
print(augmentation_final_comparison_table[[
    "Experiment", "Real_Training_Rows", "Synthetic_Training_Rows", "Total_Training_Rows",
    "Macro_F1", "ROC_AUC", "TN", "FP", "FN", "TP",
]].to_string())


# Append final augmentation and convergence sections before writing the HTML once.
augmentation_final_comparison_display = augmentation_final_comparison_table.loc[:, [
    "Experiment", "Real_Training_Rows", "Synthetic_Training_Rows", "Total_Training_Rows",
    "Macro_F1", "ROC_AUC", "Precision_Tipburn", "Recall_Tipburn", "Balanced_Accuracy",
    "TN", "FP", "FN", "TP", "Decision_Threshold",
]]
augmentation_final_comparison_display = augmentation_final_comparison_display.sort_values("Macro_F1", ascending=False).reset_index(drop=True)
validation_report_sections.append(dataframe_to_report_section(
    "Final holdout comparison",
    augmentation_final_comparison_display,
    "Performance of A, B, C, C1 and C2 on the same fixed holdout. C1 and C2 add synthetic rows to the full real training set.",
    section_id="frozen-final-synthetic-holdout-comparison",
))


# Summarise convergence after all report-relevant fits; the audit is observational only.
lr_convergence_audit = pd.DataFrame(lr_convergence_audit_rows)
if lr_convergence_audit.empty:
    raise AssertionError("Logistic Regression convergence audit collected zero rows.")
lr_convergence_audit = (
    lr_convergence_audit
    .loc[:, CONVERGENCE_AUDIT_COLUMNS]
    .sort_values(["Stage", "Seed", "Outer_Fold", "Scenario"], na_position="first")
    .reset_index(drop=True)
)
lr_convergence_summary = summarize_convergence_audit(lr_convergence_audit)

check_convergence_audit(lr_convergence_audit)
print(
    f"Logistic Regression convergence audit: {len(lr_convergence_audit)} fitted estimators audited, "
    f"{int(lr_convergence_summary.loc[0, 'Fits_Hitting_Max_Iter'])} hit max_iter, "
    f"{int(lr_convergence_summary.loc[0, 'Fits_With_Convergence_Warning'])} raised ConvergenceWarning."
)

validation_report_sections.append(dataframe_to_report_section(
    "Convergence summary",
    lr_convergence_summary,
    "Convergence results for all report-related Logistic Regression fits. This check did not change any model or prediction.",
    section_id="logistic-regression-convergence-audit-summary",
))
validation_report_appendix_sections.append(dataframe_to_report_section(
    "Convergence details",
    lr_convergence_audit,
    "One row per Logistic Regression fit, including the stage, seed, fold, scenario, C, solver settings, iterations and warnings.",
    collapsible=True,
    section_id="logistic-regression-convergence-audit-detail",
))

write_validation_html_report(
    output_path=report_path,
    report_sections=validation_report_sections,
    report_metadata=report_metadata,
    report_appendix_sections=validation_report_appendix_sections,
)
print(f"Complete validation report saved to: {report_path}")

13 of 72 final holdout leaves were misclassified by at least one experiment.
Augmentation Final C variants (C1/C2): full real training set + added synthetic.
                                                 Experiment  Real_Training_Rows  Synthetic_Training_Rows  Total_Training_Rows  Macro_F1   ROC_AUC  TN  FP  FN  TP
0                                    Final A: Original only                 288                        0                  288  0.846957  0.950617  32   4   7  29
1                                   Final B: Synthetic only                   0                      288                  288  0.888545  0.946759  34   2   6  30
2                                 Final C_Mixed_Fixed_50_50                 144                      144                  288  0.832817  0.952160  32   4   8  28
3   C1: Augmented +50% synthetic (288 real + 144 synthetic)                 288                      144                  432  0.832817  0.950617  32   4   8  28
4  C2: Augmented +100% synthetic

## 28. Export Locked Final Pipeline

Exports the fitted five-feature Logistic Regression pipeline and deployment metadata. The model was fitted on the development partition and is not refitted on the holdout set.

In [33]:
# Export the locked pipeline and metadata.
final_fitted_pipeline = locked_final_model

DEPLOYMENT_DIR = OUTPUT_DIR
DEPLOYMENT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_EXPORT_PATH = DEPLOYMENT_DIR / "final_tipburn_pipeline.joblib"
METADATA_EXPORT_PATH = DEPLOYMENT_DIR / "final_tipburn_model_metadata.json"

joblib.dump(final_fitted_pipeline, MODEL_EXPORT_PATH)

if not MODEL_EXPORT_PATH.exists():
    raise FileNotFoundError(f"Pipeline export failed: {MODEL_EXPORT_PATH}")

final_holdout_macro_f1 = float(final_metrics.loc[0, "F1_macro"])
final_holdout_roc_auc = float(final_metrics.loc[0, "ROC_AUC"])
final_tn = int(final_metrics.loc[0, "TN"])
final_fp = int(final_metrics.loc[0, "FP"])
final_fn = int(final_metrics.loc[0, "FN"])
final_tp = int(final_metrics.loc[0, "TP"])

model_metadata = {
    "model_name": FINAL_MODEL_NAME,
    "model_file": MODEL_EXPORT_PATH.name,
    "model_version": "1.0",
    "exported_at_utc": datetime.now(timezone.utc).isoformat(),
    "prediction_unit": "leaf",
    "grouping_unit": "plant",
    "group_column": "Sample_Group_ID",
    "feature_order": list(FINAL_FEATURE_COLS),
    "feature_count": len(FINAL_FEATURE_COLS),
    "class_mapping": {
        "Healthy": 0,
        "Tipburn": 1,
    },
    "positive_class": 1,
    "positive_class_name": "Tipburn",
    "decision_threshold": 0.5,
    "model_parameters": {
        "clf__C": FINAL_MODEL_C,
        "max_iter": 50,
        "random_state": RANDOM_STATE,
    },
    "training_scope": {
        "partition": "train_val",
        "description": (
            "Fitted using only the fixed 80% development partition. "
            "The final 20% holdout was not used for fitting or retuning."
        ),
        "development_leaf_count": int(len(locked_data["X_train"])),
        "development_plant_count": int(pd.Series(locked_data["groups_train"]).nunique()),
    },
    "holdout_scope": {
        "partition": "final_test",
        "holdout_leaf_count": int(len(locked_data["X_test"])),
        "holdout_plant_count": int(pd.Series(locked_data["groups_test"]).nunique()),
    },
    "preprocessing": {
        "scaler": "StandardScaler",
        "savgol_window": int(DEFAULT_WINDOW),
        "savgol_polyorder": int(POLYORDER),
        "normalization": "Row-wise min-max normalization after Savitzky-Golay smoothing",
        "ndre_ndvi_source": "Smoothed reflectance with wavelength interpolation",
    },
    "validated_holdout_results": {
        "macro_f1": final_holdout_macro_f1,
        "roc_auc": final_holdout_roc_auc,
        "true_negative": final_tn,
        "false_positive": final_fp,
        "false_negative": final_fn,
        "true_positive": final_tp,
    },
    "software_versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
}

with METADATA_EXPORT_PATH.open("w", encoding="utf-8") as file:
    json.dump(model_metadata, file, indent=2, ensure_ascii=False)

if not METADATA_EXPORT_PATH.exists():
    raise FileNotFoundError(f"Metadata export failed: {METADATA_EXPORT_PATH}")

print(f"Final pipeline saved to: {MODEL_EXPORT_PATH.resolve()}")
print(f"Final model metadata saved to: {METADATA_EXPORT_PATH.resolve()}")


# Export development-only reference distributions from the locked preprocessing output.
REFERENCE_EXPORT_PATH = DEPLOYMENT_DIR / "development_reference.npz"

if list(FINAL_FEATURE_COLS) != model_metadata["feature_order"]:
    raise AssertionError("Reference feature_order must match the locked model metadata feature_order.")

_dev_normalized = processed["normalized"][train_mask]
if _dev_normalized.shape[0] != len(features_train):
    raise AssertionError("Development normalized spectra are not aligned with features_train.")
_dev_y = features_train["y"].to_numpy(int)
_healthy_mask = _dev_y == 0
_tipburn_mask = _dev_y == 1
if int(_healthy_mask.sum() + _tipburn_mask.sum()) != int(model_metadata["training_scope"]["development_leaf_count"]):
    raise AssertionError("Reference class counts do not sum to the development leaf count.")

np.savez(
    REFERENCE_EXPORT_PATH,
    wavelengths=wavelengths.astype(float),
    healthy_curve_mean=_dev_normalized[_healthy_mask].mean(axis=0),
    healthy_curve_sd=_dev_normalized[_healthy_mask].std(axis=0),
    tipburn_curve_mean=_dev_normalized[_tipburn_mask].mean(axis=0),
    tipburn_curve_sd=_dev_normalized[_tipburn_mask].std(axis=0),
    healthy_features=features_train.loc[_healthy_mask, list(FINAL_FEATURE_COLS)].to_numpy(float),
    tipburn_features=features_train.loc[_tipburn_mask, list(FINAL_FEATURE_COLS)].to_numpy(float),
    feature_order=np.array(list(FINAL_FEATURE_COLS)),
    n_healthy=int(_healthy_mask.sum()),
    n_tipburn=int(_tipburn_mask.sum()),
)
if not REFERENCE_EXPORT_PATH.exists():
    raise FileNotFoundError(f"Development reference export failed: {REFERENCE_EXPORT_PATH}")
print(f"Development reference artefact saved to: {REFERENCE_EXPORT_PATH.resolve()} "
      f"({int(_healthy_mask.sum())} Healthy, {int(_tipburn_mask.sum())} Tipburn dev leaves)")

Final pipeline saved to: /Users/Dell/Documents/FYP/CP2 Submission/smart_leaf_analyzer/outputs/final_tipburn_pipeline.joblib
Final model metadata saved to: /Users/Dell/Documents/FYP/CP2 Submission/smart_leaf_analyzer/outputs/final_tipburn_model_metadata.json
Development reference artefact saved to: /Users/Dell/Documents/FYP/CP2 Submission/smart_leaf_analyzer/outputs/development_reference.npz (144 Healthy, 144 Tipburn dev leaves)


## 29. Reload Export and Verify Predictions

Reloads the exported pipeline and metadata, then confirms the reloaded model reproduces the original final-holdout probabilities, predictions, metrics, and confusion-matrix values exactly.

In [34]:
# Reload the exported artefacts and confirm identical holdout scores and predictions.
X_verification = locked_data["X_test"]
y_verification = locked_data["y_test"]

original_probabilities = final_fitted_pipeline.predict_proba(
    X_verification.to_numpy(float)
)[:, 1]
original_predictions = (
    original_probabilities >= model_metadata["decision_threshold"]
).astype(int)

reloaded_pipeline = joblib.load(MODEL_EXPORT_PATH)

with METADATA_EXPORT_PATH.open("r", encoding="utf-8") as file:
    reloaded_metadata = json.load(file)

assert reloaded_metadata["model_name"] == FINAL_MODEL_NAME
assert reloaded_metadata["feature_order"] == list(FINAL_FEATURE_COLS)
assert reloaded_metadata["feature_count"] == len(FINAL_FEATURE_COLS)
assert reloaded_metadata["model_parameters"]["clf__C"] == FINAL_MODEL_C
assert reloaded_metadata["decision_threshold"] == 0.5
assert reloaded_metadata["class_mapping"] == {"Healthy": 0, "Tipburn": 1}

if hasattr(reloaded_pipeline, "feature_names_in_"):
    assert list(reloaded_pipeline.feature_names_in_) == list(FINAL_FEATURE_COLS)

reloaded_probabilities = reloaded_pipeline.predict_proba(
    X_verification.to_numpy(float)
)[:, 1]
threshold = float(reloaded_metadata["decision_threshold"])
reloaded_predictions = (reloaded_probabilities >= threshold).astype(int)

np.testing.assert_allclose(
    reloaded_probabilities,
    original_probabilities,
    rtol=0.0,
    atol=1e-12,
    err_msg="Reloaded probabilities differ from the original fitted pipeline.",
)

if not np.array_equal(reloaded_predictions, original_predictions):
    changed_indices = np.flatnonzero(reloaded_predictions != original_predictions)
    raise AssertionError(
        "Reloaded predictions differ at rows: "
        f"{changed_indices.tolist()}"
    )

sk = require_sklearn()
f1_score = sk["f1_score"]
roc_auc_score = sk["roc_auc_score"]
confusion_matrix = sk["confusion_matrix"]

reloaded_macro_f1 = f1_score(
    y_verification,
    reloaded_predictions,
    average="macro",
)
reloaded_roc_auc = roc_auc_score(
    y_verification,
    reloaded_probabilities,
)
reloaded_tn, reloaded_fp, reloaded_fn, reloaded_tp = confusion_matrix(
    y_verification,
    reloaded_predictions,
    labels=[0, 1],
).ravel()

np.testing.assert_allclose(
    reloaded_macro_f1,
    final_holdout_macro_f1,
    rtol=0.0,
    atol=1e-12,
)
np.testing.assert_allclose(
    reloaded_roc_auc,
    final_holdout_roc_auc,
    rtol=0.0,
    atol=1e-12,
)
assert (
    int(reloaded_tn),
    int(reloaded_fp),
    int(reloaded_fn),
    int(reloaded_tp),
) == (final_tn, final_fp, final_fn, final_tp)

print("Reload verification passed: probabilities and predictions are identical.")
print("Reloaded holdout verification passed.")
print(f"Macro F1: {reloaded_macro_f1:.6f}")
print(f"ROC-AUC: {reloaded_roc_auc:.6f}")
print(
    "Confusion matrix: "
    f"TN={reloaded_tn}, "
    f"FP={reloaded_fp}, "
    f"FN={reloaded_fn}, "
    f"TP={reloaded_tp}"
)

Reload verification passed: probabilities and predictions are identical.
Reloaded holdout verification passed.
Macro F1: 0.846957
ROC-AUC: 0.950617
Confusion matrix: TN=32, FP=4, FN=7, TP=29
